# Datamart Month Acquiring — FinalDF + Excel Compare (декомпозиция)

Тетрадка разбита на независимые этапы, чтобы при ошибке перезапускать только проблемную секцию, а не весь pipeline.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def exact_match_rate_from_abs(abs_series, tol=0.0):
    if len(abs_series) == 0:
        return 0.0
    return round((abs_series.fillna(0) <= tol).mean() * 100, 2)

In [ ]:
# Параметры и подключение
report_month = '2026-04-01'
excel_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header = 0

run_invalidate_metadata = False
run_excel_compare = True
show_compare_top = True
save_final_csv = False
output_csv_path = './datamart_month_acquiring_final_2026_04.csv'

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

print(f'report_month={report_month}, month_start={month_start}, month_end={month_end}')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

invalidate_tables = [
    'ods_alpha.scd1_agreements', 'ods_alpha.scd1_companies', 'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext', 'cdiul.ext_id_org', 'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals', 'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_acq', 'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids', 'ods.scd1_z_r2_ip_merchants', 'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix', 'ods.scd1_z_cl_corp', 'ods.scd1_z_depart',
    'ods.scd1_z_branch', 'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    with imp:
        for t in invalidate_tables:
            imp.execute(f'invalidate metadata {t}')
    print('Invalidate metadata completed')
else:
    print('Invalidate metadata skipped')

## Секция 1. Формирование `final_df` (по шагам)

Ниже каждый этап вынесен в отдельную ячейку:
`01_sa_perimeter` -> `02_cdi_map` -> `03_cft_map` -> `04_operational_metrics` -> `05_transaction_metrics` -> `06_r2_legacy_attrs` -> `07_base_merge` -> `08_agr_fallback` -> `08b_tariff_fix_map` -> `09_actual_tariff_by_agr` -> `10_apply_tariff_fix_and_formulas`.

In [ ]:
# 01_sa_perimeter
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()
if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

print(f'SA perimeter rows: {len(sa_df):,}')
display(sa_df.head(3))

In [ ]:
# 02_cdi_map
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])
if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    allowed = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed), None)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'CDI rows: {len(cdi_map_df):,}')
display(cdi_map_df.head(3))

In [ ]:
# 03_cft_map
if 'cdi_map_df' not in globals():
    raise RuntimeError('Сначала выполните 02_cdi_map')

cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])
if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'CFT rows: {len(cft_map_df):,}')
display(cft_map_df.head(3))

In [ ]:
# 04_operational_metrics (hybrid_all optimized: 547-like core + fallback only for missing n_agr)
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

section04_start_ts = time.perf_counter()

if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    def _build_sql_cmp_hybrid(use_m_acq_filter=True):
        m_acq_filter_sql = "and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'" if use_m_acq_filter else ''

        return f"""
        with sa_agr as (
          select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.n_cmp_client as string) as n_cmp_client
          from ods_alpha.scd1_agreements a
          where cast(a.n_agr as string) in ({agr_in})
        ),

        -- 547-like perimeter by AGR_TERMS + merchant filters
        terms_547 as (
          select distinct
            sa.n_agr,
            sa.n_cmp_client,
            cast(t.c_nmrc as string) as c_nmrc
          from sa_agr sa
          join ods_alpha.scd1_agr_terms t
            on cast(t.n_agr as string) = sa.n_agr
          join ods_alpha.scd1_merchants m
            on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
          where t.c_nmrc is not null
            and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
            {m_acq_filter_sql}
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and coalesce(m.ods_deleted_flg, '0') <> '1'
        ),
        pos_period as (
          select distinct
            cast(p.c_nmrc as string) as c_nmrc,
            cast(p.c_pos_serial as string) as c_pos_serial,
            cast(p.c_nter as string) as c_nter
          from ods_alpha.scd1_pos_terminals p
          where p.c_pos_serial is not null
            and cast(p.d_ter_install as date) is not null
            and cast(p.d_ter_install as date) < date_add(cast('{month_end}' as date), 1)
            and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
            and coalesce(p.ods_deleted_flg, '0') <> '1'
        ),
        term_active_547 as (
          select
            t.n_agr,
            t.n_cmp_client,
            t.c_nmrc,
            p.c_pos_serial,
            p.c_nter
          from terms_547 t
          left join pos_period p
            on p.c_nmrc = t.c_nmrc
        ),
        retl_547 as (
          select n_agr, count(distinct c_nmrc) as retl_cnt_547
          from term_active_547
          group by n_agr
        ),
        term_547 as (
          select n_agr, count(distinct c_pos_serial) as term_cnt_547
          from term_active_547
          group by n_agr
        ),
        amort_547 as (
          select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_547
          from (
            select distinct n_agr, c_nter
            from term_active_547
            where c_nter is not null
          ) x
          left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
            on cast(am.c_nter as string) = x.c_nter
          group by x.n_agr
        ),

        -- fallback: compute old branch only where 547-like has no counts
        fallback_agrs as (
          select sa.n_agr, sa.n_cmp_client
          from sa_agr sa
          left join retl_547 r547 on r547.n_agr = sa.n_agr
          left join term_547 t547 on t547.n_agr = sa.n_agr
          where r547.n_agr is null and t547.n_agr is null
        ),
        old_nmrc as (
          select fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
          from fallback_agrs fa
          join ods_alpha.scd1_merchants mm
            on cast(mm.n_cmp as string) = fa.n_cmp_client
          where mm.c_nmrc is not null
            and coalesce(mm.ods_deleted_flg, '0') <> '1'
          group by fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string)
        ),
        old_term_active as (
          select
            o.n_agr,
            o.n_cmp_client,
            o.c_nmrc,
            cast(t.c_nter as string) as c_nter
          from old_nmrc o
          join ods_alpha.scd1_pos_terminals t
            on cast(t.c_nmrc as string) = o.c_nmrc
          where t.c_nter is not null
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and cast(t.d_ter_install as date) is not null
            and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
            and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
          group by o.n_agr, o.n_cmp_client, o.c_nmrc, cast(t.c_nter as string)
        ),
        old_retl as (
          select n_agr, count(distinct c_nmrc) as retl_cnt_old
          from old_term_active
          group by n_agr
        ),
        old_term as (
          select n_agr, count(distinct c_nter) as term_cnt_old
          from old_term_active
          group by n_agr
        ),
        old_amort as (
          select x.n_agr, sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization_old
          from (
            select distinct n_agr, c_nter
            from old_term_active
            where c_nter is not null
          ) x
          left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
            on cast(am.c_nter as string) = x.c_nter
          group by x.n_agr
        )

        select
          sa.n_agr,
          sa.n_cmp_client,
          coalesce(r547.retl_cnt_547, rold.retl_cnt_old) as retl_cnt,
          coalesce(t547.term_cnt_547, told.term_cnt_old) as term_cnt,
          coalesce(a547.amortization_547, aold.amortization_old, 0.0) as amortization,
          case
            when r547.n_agr is not null or t547.n_agr is not null then '547_like'
            when rold.n_agr is not null or told.n_agr is not null then 'fallback_old'
            else 'no_data'
          end as term_calc_source
        from sa_agr sa
        left join retl_547 r547 on r547.n_agr = sa.n_agr
        left join term_547 t547 on t547.n_agr = sa.n_agr
        left join amort_547 a547 on a547.n_agr = sa.n_agr
        left join old_retl rold on rold.n_agr = sa.n_agr
        left join old_term told on told.n_agr = sa.n_agr
        left join old_amort aold on aold.n_agr = sa.n_agr
        """

    used_m_acq_filter = True
    try:
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            cmp_df = imp.fetch(_build_sql_cmp_hybrid(use_m_acq_filter=True))
    except Exception as exc_hybrid:
        used_m_acq_filter = False
        print(f"04 hybrid: m.acq_class filter disabled due to: {type(exc_hybrid).__name__}")
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            cmp_df = imp.fetch(_build_sql_cmp_hybrid(use_m_acq_filter=False))

if cmp_df is None:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization', 'term_calc_source'])

if not cmp_df.empty:
    cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)
    cmp_df['retl_cnt'] = pd.to_numeric(cmp_df['retl_cnt'], errors='coerce')
    cmp_df['term_cnt'] = pd.to_numeric(cmp_df['term_cnt'], errors='coerce')
    cmp_df['amortization'] = pd.to_numeric(cmp_df['amortization'], errors='coerce')

section04_elapsed_sec = round(time.perf_counter() - section04_start_ts, 2)

print(f'Operational rows: {len(cmp_df):,}')
if 'used_m_acq_filter' in locals():
    print('04 hybrid note: used m.acq_class <> SV filter =', used_m_acq_filter)
print('section_04_elapsed_sec =', section04_elapsed_sec)
if len(cmp_df) and 'term_calc_source' in cmp_df.columns:
    print('term_calc_source split:')
    display(cmp_df.groupby('term_calc_source', as_index=False).agg(rows=('n_agr', 'size')))

display(cmp_df.head(3))

In [ ]:
# 05_transaction_metrics
if 'sa_df' not in globals():
    raise RuntimeError('Сначала выполните 01_sa_perimeter')

if sa_df.empty:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
else:
    n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_trx = f"""
    with sa_agr as (
      select distinct cast(n_agr as string) as n_agr, cast(n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements
      where cast(n_agr as string) in ({agr_in})
    ),
    trx_base_raw as (
      select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select cast(a.n_trx as string) as n_trx, cast(a.n_agr as string) as n_agr, coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb on tb.n_trx = ta.n_trx
      left join sa_agr sa on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    )
    select
      t.n_agr,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      a.n_cmp_client,
      a.active_term_cnt
    from trx_agg t
    left join sa_agr sa on sa.n_agr = t.n_agr
    left join active_term_agg a on a.n_cmp_client = sa.n_cmp_client
    """

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        trx_df = imp.fetch(sql_trx)

if trx_df is None:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
if not trx_df.empty:
    trx_df['n_agr'] = trx_df['n_agr'].astype(str)
    trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

print(f'Transaction rows: {len(trx_df):,}')
display(trx_df.head(3))

In [ ]:
# 06_r2_legacy_attrs
if 'cft_map_df' not in globals():
    raise RuntimeError('Сначала выполните 03_cft_map')

cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])
else:
    sql_r2 = f"""
    with r2 as (
      select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_depart as string) as c_depart,
        cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.c_cl_org as string) in ({cft_sql_list})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      r2.r2_id,
      r2.cft_id,
      cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
      cast(dep.c_name as string) as vsp_name,
      cast(dep.c_num as string) as vsp_code,
      case
        when br.c_shortlabel is null then null
        when upper(cast(br.c_shortlabel as string)) like '%РФ%'
          then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
        else cast(br.c_shortlabel as string)
      end as filial_rf,
      cast(tp.c_name as string) as tariff_name_legacy,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly_legacy
    from r2
    left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
    left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
    left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
    left join fix on fix.c_tariff_plan = r2.c_tariff_plan
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        r2_df = imp.fetch(sql_r2)

if r2_df is None:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy']:
        r2_df[c] = r2_df[c].astype(str)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

print(f'R2 legacy rows: {len(r2_df):,}')
display(r2_df.head(3))

In [ ]:
# 07_base_merge
required = ['sa_df', 'cdi_map_df', 'cft_map_df', 'cmp_df', 'trx_df', 'active_term_df', 'r2_df']
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f'Сначала выполните предыдущие шаги: отсутствуют {missing}')

base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name_legacy', 'commission_monthly_legacy']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

print(f'base_df rows after merge: {len(base_df):,}')
display(base_df.head(3))

In [ ]:
# 08_agr_fallback
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 07_base_merge')

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_before_mask = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_before_mask) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

filled_pct = round(100 * (base_df['agr_id'].notna().mean()), 2) if len(base_df) else 0.0
print(f'agr_id filled after fallback: {filled_pct}%')

In [ ]:
# 08b_tariff_fix_map (оптимизировано: только тарифы из текущего base_df)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров/подключения')
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 08_agr_fallback')

section08b_start_ts = time.perf_counter()

agr_id_scope = (
    base_df.get('agr_id', pd.Series(dtype=object))
    .map(normalize_agr_q1)
    .dropna()
    .astype(str)
    .str.strip()
)
agr_id_scope = sorted([x for x in agr_id_scope.unique().tolist() if x])

if agr_id_scope:
    agr_in = ', '.join([f"'{x}'" for x in agr_id_scope])

    sql_tariff_fix_map = f"""
    with plan_scope as (
      select distinct cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where m.c_tariff_plan is not null
        and cast(m.id as string) in ({agr_in})
    )
    select
      ps.c_tariff_plan,
      max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
    from plan_scope ps
    left join ods.scd1_z_r2_tariff_tune tt
      on cast(tt.c_tariff_plan as string) = ps.c_tariff_plan
    left join ods.scd1_z_r2_tariff_fix tf
      on tt.c_tariff = tf.id
    group by ps.c_tariff_plan
    """

    try:
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            tariff_fix_map_df = imp.fetch(sql_tariff_fix_map)
    except Exception as exc_08b:
        print(f"08b filtered map failed ({type(exc_08b).__name__}); fallback to full map")
        sql_tariff_fix_map_full = """
        select
          cast(tt.c_tariff_plan as string) as c_tariff_plan,
          max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
        from ods.scd1_z_r2_tariff_tune tt
        left join ods.scd1_z_r2_tariff_fix tf
          on tt.c_tariff = tf.id
        group by cast(tt.c_tariff_plan as string)
        """
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            tariff_fix_map_df = imp.fetch(sql_tariff_fix_map_full)
else:
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if tariff_fix_map_df is None:
    tariff_fix_map_df = pd.DataFrame(columns=['c_tariff_plan', 'commission_monthly_fix'])

if not tariff_fix_map_df.empty:
    tariff_fix_map_df['c_tariff_plan'] = tariff_fix_map_df['c_tariff_plan'].astype(str).str.strip()

section08b_elapsed_sec = round(time.perf_counter() - section08b_start_ts, 2)

print(f'08b agr_id scope: {len(agr_id_scope):,}')
print(f'tariff_fix_map rows: {len(tariff_fix_map_df):,}')
print('section_08b_elapsed_sec =', section08b_elapsed_sec)
display(tariff_fix_map_df.head(3))

In [ ]:
# 09_actual_tariff_by_agr
if 'base_df' not in globals():
    raise RuntimeError('Сначала выполните 08_agr_fallback')
if 'tariff_fix_map_df' not in globals():
    raise RuntimeError('Сначала выполните 08b_tariff_fix_map')

actual_tariff_df_prev = (
    actual_tariff_df.copy()
    if 'actual_tariff_df' in globals() and isinstance(actual_tariff_df, pd.DataFrame)
    else pd.DataFrame()
)

base_df['agr_id_key'] = base_df['agr_id'].map(normalize_agr_q1)
base_agr_keys_df = (
    base_df[['agr_id_key']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'agr_id_key': 'agr_id'})
)
base_agr_keys_cnt = len(base_agr_keys_df)

section09_start_ts = time.perf_counter()

if base_agr_keys_cnt:
    sql_actual_tariff_by_agr = f"""
    with agr_keys as (
      select distinct cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
        and exists (
          select 1
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) = cast(a.n_agr as string)
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
        )
    ),
    agr_actual as (
      select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr_actual,
        cast(a.c_agr_number as string) as contract_number_acq,
        cast(a.d_valid_from as date) as d_valid_from_actual,
        cast(a.d_valid_to as date) as d_valid_to_actual,
        row_number() over (
          partition by cast(a.abs_agr_id as string)
          order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
        ) as rn
      from ods_alpha.scd1_agreements a
      join agr_keys k
        on k.agr_id = cast(a.abs_agr_id as string)
      where cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    merchant_one as (
      select
        cast(m.id as string) as agr_id,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        row_number() over (
          partition by cast(m.id as string)
          order by cast(m.c_tariff_plan as string) desc
        ) as rn
      from ods.scd1_z_r2_ip_merchants m
      join agr_keys k
        on k.agr_id = cast(m.id as string)
    )
    select
      m.agr_id,
      aa.n_agr_actual,
      aa.contract_number_acq,
      aa.d_valid_from_actual,
      aa.d_valid_to_actual,
      m.c_tariff_plan,
      cast(tp.c_name as string) as tariff_name_actual
    from merchant_one m
    left join agr_actual aa
      on aa.agr_id = m.agr_id
     and aa.rn = 1
    left join ods.scd1_z_r2_tariff_plan tp
      on cast(tp.id as string) = m.c_tariff_plan
    where m.rn = 1
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        actual_tariff_df = imp.fetch(sql_actual_tariff_by_agr)
else:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if actual_tariff_df is None:
    actual_tariff_df = pd.DataFrame(columns=[
        'agr_id', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
        'c_tariff_plan', 'tariff_name_actual'
    ])

if not actual_tariff_df.empty:
    for c in ['agr_id', 'n_agr_actual', 'contract_number_acq', 'c_tariff_plan', 'tariff_name_actual']:
        actual_tariff_df[c] = actual_tariff_df[c].astype(str).str.strip()

    # Ограничиваем результат ключами текущего base_df (сохраняем ту же прикладную выборку)
    actual_tariff_df = actual_tariff_df.merge(base_agr_keys_df, on='agr_id', how='inner')

actual_tariff_df = actual_tariff_df.rename(columns={'agr_id': 'agr_id_key'})

if 'c_tariff_plan' not in actual_tariff_df.columns:
    actual_tariff_df['c_tariff_plan'] = None

if not actual_tariff_df.empty and not tariff_fix_map_df.empty:
    actual_tariff_df = actual_tariff_df.merge(tariff_fix_map_df, on='c_tariff_plan', how='left')
else:
    actual_tariff_df['commission_monthly_fix'] = None

actual_tariff_df['commission_monthly_actual'] = actual_tariff_df.get('commission_monthly_fix')
if 'commission_monthly_fix' in actual_tariff_df.columns:
    actual_tariff_df = actual_tariff_df.drop(columns=['commission_monthly_fix'])

section09_elapsed_sec = round(time.perf_counter() - section09_start_ts, 2)

# QC эквивалентности с предыдущим результатом (если был в памяти)
if isinstance(actual_tariff_df_prev, pd.DataFrame) and len(actual_tariff_df_prev):
    prev_cmp_df = actual_tariff_df_prev[['agr_id_key', 'tariff_name_actual']].copy() if 'agr_id_key' in actual_tariff_df_prev.columns and 'tariff_name_actual' in actual_tariff_df_prev.columns else pd.DataFrame(columns=['agr_id_key', 'tariff_name_actual'])
    new_cmp_df = actual_tariff_df[['agr_id_key', 'tariff_name_actual']].copy() if 'agr_id_key' in actual_tariff_df.columns and 'tariff_name_actual' in actual_tariff_df.columns else pd.DataFrame(columns=['agr_id_key', 'tariff_name_actual'])

    prev_cmp_df = prev_cmp_df.drop_duplicates(subset=['agr_id_key'])
    new_cmp_df = new_cmp_df.drop_duplicates(subset=['agr_id_key'])

    eq_cmp_df = prev_cmp_df.merge(new_cmp_df, on='agr_id_key', how='outer', suffixes=('_prev', '_new'), indicator='src')
    eq_cmp_df['_prev_norm'] = eq_cmp_df['tariff_name_actual_prev'].fillna('').astype(str).str.strip().str.lower()
    eq_cmp_df['_new_norm'] = eq_cmp_df['tariff_name_actual_new'].fillna('').astype(str).str.strip().str.lower()

    tariff_changed_cnt = int(((eq_cmp_df['src'] == 'both') & (eq_cmp_df['_prev_norm'] != eq_cmp_df['_new_norm'])).sum())
    only_prev_cnt = int((eq_cmp_df['src'] == 'left_only').sum())
    only_new_cnt = int((eq_cmp_df['src'] == 'right_only').sum())

    print('QC эквивалентности (prev vs new):')
    print(f'changed_tariff_name: {tariff_changed_cnt:,}')
    print(f'only_prev_keys: {only_prev_cnt:,}')
    print(f'only_new_keys: {only_new_cnt:,}')

print(f'base agr_id keys: {base_agr_keys_cnt:,}')
print(f'actual_tariff rows: {len(actual_tariff_df):,}')
print(f'section_09_elapsed_sec: {section09_elapsed_sec}')
display(actual_tariff_df.head(3))

In [ ]:
# 10_apply_tariff_fix_and_formulas -> final_df
if 'base_df' not in globals() or 'actual_tariff_df' not in globals():
    raise RuntimeError('Сначала выполните 09_actual_tariff_by_agr')

# Совместимость с предыдущими версиями 09 (где имена колонок могли отличаться)
if 'tariff_name_actual' not in actual_tariff_df.columns and 'tariff_name' in actual_tariff_df.columns:
    actual_tariff_df['tariff_name_actual'] = actual_tariff_df['tariff_name']
if 'commission_monthly_actual' not in actual_tariff_df.columns and 'commission_monthly' in actual_tariff_df.columns:
    actual_tariff_df['commission_monthly_actual'] = actual_tariff_df['commission_monthly']

required_tariff_cols = [
    'agr_id_key', 'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual',
    'd_valid_to_actual', 'c_tariff_plan', 'tariff_name_actual', 'commission_monthly_actual'
]
for col in required_tariff_cols:
    if col not in actual_tariff_df.columns:
        actual_tariff_df[col] = None

# Идемпотентность: при повторном запуске 10-й секции убираем старые колонки и суффиксы _x/_y
merge_payload_cols = [c for c in required_tariff_cols if c != 'agr_id_key']
drop_before_merge = []
for c in merge_payload_cols:
    for c_try in [c, f'{c}_x', f'{c}_y']:
        if c_try in base_df.columns:
            drop_before_merge.append(c_try)
if drop_before_merge:
    base_df = base_df.drop(columns=sorted(set(drop_before_merge)))

base_df = base_df.merge(
    actual_tariff_df[required_tariff_cols],
    on='agr_id_key',
    how='left'
)

base_df['tariff_name'] = base_df['tariff_name_actual'].where(
    base_df['tariff_name_actual'].notna() & (base_df['tariff_name_actual'].astype(str).str.strip() != ''),
    base_df['tariff_name_legacy']
)
base_df['tariff_source'] = base_df['tariff_name_actual'].apply(
    lambda x: 'agr_id_actual' if pd.notna(x) and str(x).strip() else 'legacy_cft'
)
base_df['commission_monthly'] = pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').where(
    pd.to_numeric(base_df['commission_monthly_actual'], errors='coerce').notna(),
    pd.to_numeric(base_df['commission_monthly_legacy'], errors='coerce')
)

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num
base_df['aur'] = retl_cnt_num * 1926
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - pd.to_numeric(base_df['aur'], errors='coerce').fillna(0) - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'n_agr_actual', 'contract_number_acq', 'd_valid_from_actual', 'd_valid_to_actual',
    'cdi_id', 'ssp_ocrm', 'cft_id', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code',
    'tariff_name', 'tariff_source',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

if final_df is None or len(final_df) == 0:
    raise RuntimeError('QC fail: final_df пустой')
if 'tariff_name' not in final_df.columns:
    raise RuntimeError('QC fail: отсутствует tariff_name')

tariff_actual_rows = int((final_df['tariff_source'] == 'agr_id_actual').sum()) if len(final_df) else 0
tariff_actual_pct = round(100 * tariff_actual_rows / len(final_df), 2) if len(final_df) else 0.0

print(f'final_df rows: {len(final_df):,}')
print(f'tariff_source=agr_id_actual: {tariff_actual_rows:,} ({tariff_actual_pct}%)')
display(final_df.head(5))

## Секция 2. Сравнение `final_df` и Excel (`cmp` + KPI)

Секция независима от шагов выше, кроме наличия `final_df`.

In [ ]:
# Compare + KPI (ключевые метрики из запроса)
if 'final_df' not in globals():
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas')
for key_col in ['inn', 'agr_id']:
    if key_col not in final_df.columns:
        raise RuntimeError(f'В final_df отсутствует ключевая колонка: {key_col}')

if not run_excel_compare:
    cmp = pd.DataFrame()
    kpi_compare_df = pd.DataFrame()
    totals_df = pd.DataFrame()
    print('Excel compare skipped (run_excel_compare=False)')
else:
    ex = pd.read_excel(excel_path, header=excel_header)

    # Маппинг колонок Excel под ключевые метрики
    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur'],
        'fin_result_col': ['Фин. Рез.', 'Фин.Рез.', 'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат', 'fin_result'],
    }

    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    print('Используемые колонки Excel:')
    display(pd.DataFrame([{'logical_name': k, 'excel_column': v} for k, v in resolved.items()]))

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved['trx_sum_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['int_component_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['chod_excel'] = to_num_series(ex[resolved['chod_col']])
    ex['aur_excel'] = to_num_series(ex[resolved['aur_col']])
    ex['fin_result_excel'] = to_num_series(ex[resolved['fin_result_col']])

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'commission_monthly_excel': 'max',
              'chod_excel': 'sum',
              'aur_excel': 'sum',
              'fin_result_excel': 'sum',
          })
    )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')
    lk['aur_lake'] = pd.to_numeric(lk['aur'], errors='coerce')
    lk['fin_result_lake'] = pd.to_numeric(lk['fin_result'], errors='coerce')

    lk_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'commission_monthly_lake': 'max',
              'chod_lake': 'max',
              'aur_lake': 'max',
              'fin_result_lake': 'max',
          })
    )

    cmp = lk_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='inner')

    metric_keys = [
        'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
        'commission_total', 'int_component', 'commission_monthly',
        'chod', 'aur', 'fin_result'
    ]

    for m in metric_keys:
        cmp[f'{m}_delta'] = cmp[f'{m}_lake'].fillna(0) - cmp[f'{m}_excel'].fillna(0)
        cmp[f'{m}_abs'] = cmp[f'{m}_delta'].abs()

    # KPI
    count_metrics = ['retl_cnt', 'term_cnt', 'trx_cnt']
    money_metrics = ['trx_sum', 'commission_total', 'int_component', 'commission_monthly', 'chod', 'aur', 'fin_result']

    kpi_rows = [{'metric': 'rows_compared', 'value': len(cmp)}]
    for m in count_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.0)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})
    for m in money_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0_01', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.01)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})

    kpi_compare_df = pd.DataFrame(kpi_rows)

    totals_row = {'report_month': report_month, 'rows_compared': len(cmp)}
    for m in metric_keys:
        totals_row[f'excel_{m}_total'] = cmp[f'{m}_excel'].fillna(0).sum()
        totals_row[f'lake_{m}_total'] = cmp[f'{m}_lake'].fillna(0).sum()
        totals_row[f'delta_{m}_total'] = totals_row[f'lake_{m}_total'] - totals_row[f'excel_{m}_total']
    totals_df = pd.DataFrame([totals_row])

    if cmp is None or len(cmp) == 0:
        raise RuntimeError('QC fail: cmp не сформирован или пустой')

    print(f'cmp rows: {len(cmp):,}')
    display(kpi_compare_df)
    display(totals_df)

    if show_compare_top:
        for m in metric_keys:
            cols = ['agr_id_key', 'inn_key', f'{m}_lake', f'{m}_excel', f'{m}_delta']
            print(f'TOP-10 {m} by abs delta')
            display(cmp.sort_values(f'{m}_abs', ascending=False)[cols].head(10))

if save_final_csv:
    out = Path(output_csv_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'final_df exported to: {out.resolve()}')

## Секция 3. Диагностика расхождений `retl_cnt` и `term_cnt`

Секция помогает понять источник расхождений между Excel и Озером:
- пустые/пропущенные значения в Lake;
- крупные дельты по `agr_id`;
- дубликаты договоров в Lake по ключу `inn+agr_id`;
- связь расхождений с периметром `merchant/terminal`.

In [ ]:
# 3.1 Базовая диагностика по cmp
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните compare-секцию (нужен DataFrame cmp).')
if 'base_df' not in globals() or base_df is None or len(base_df) == 0:
    raise RuntimeError('Сначала выполните секции до 10 (нужен DataFrame base_df).')

diag_top_n = 20

# Контроль NA/пропусков в Lake относительно Excel
retl_na_issue_df = cmp[cmp['retl_cnt_lake'].isna() & cmp['retl_cnt_excel'].notna()].copy()
term_na_issue_df = cmp[cmp['term_cnt_lake'].isna() & cmp['term_cnt_excel'].notna()].copy()

print(f"retl: Lake NA при наличии Excel = {len(retl_na_issue_df):,}")
print(f"term: Lake NA при наличии Excel = {len(term_na_issue_df):,}")

# TOP расхождений
retl_top_abs_df = cmp.sort_values('retl_cnt_abs', ascending=False).head(diag_top_n).copy()
term_top_abs_df = cmp.sort_values('term_cnt_abs', ascending=False).head(diag_top_n).copy()

print(f'\nTOP-{diag_top_n} retl_cnt by abs delta:')
display(retl_top_abs_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs']])

print(f'TOP-{diag_top_n} term_cnt by abs delta:')
display(term_top_abs_df[['agr_id_key', 'inn_key', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs']])

# Ключи для углубленной диагностики
focus_keys_df = pd.concat([
    retl_top_abs_df[['agr_id_key', 'inn_key']],
    term_top_abs_df[['agr_id_key', 'inn_key']]
], ignore_index=True).drop_duplicates().reset_index(drop=True)

# Диагностика дубликатов на стороне Lake (по base_df)
base_keys_df = base_df.copy()
base_keys_df['agr_id_key'] = base_keys_df['agr_id'].apply(normalize_agr_q1)
base_keys_df['inn_key'] = base_keys_df['inn'].apply(normalize_inn_q1)

dup_lake_keys_df = (
    base_keys_df.dropna(subset=['agr_id_key', 'inn_key'])
      .groupby(['agr_id_key', 'inn_key'], as_index=False)
      .agg(rows=('agr_id_key', 'size'), n_agr_nunique=('n_agr', 'nunique'))
      .query('rows > 1 or n_agr_nunique > 1')
      .sort_values(['rows', 'n_agr_nunique'], ascending=False)
)

print(f'\nКлючей inn+agr_id с дублями в base_df: {len(dup_lake_keys_df):,}')
if not dup_lake_keys_df.empty:
    display(dup_lake_keys_df.head(20))

focus_detail_df = (
    focus_keys_df
    .merge(cmp[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs',
                'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs']],
           on=['agr_id_key', 'inn_key'], how='left')
    .merge(base_keys_df[['agr_id_key', 'inn_key', 'n_agr', 'n_cmp_client', 'company_name']].drop_duplicates(),
           on=['agr_id_key', 'inn_key'], how='left')
)

print(f'\nFocus keys для SQL-диагностики: {len(focus_detail_df):,}')
display(focus_detail_df.sort_values(['retl_cnt_abs', 'term_cnt_abs'], ascending=False).head(50))

In [ ]:
# 3.2 SQL-диагностика периметра merchant/terminal по проблемным n_agr
if 'focus_detail_df' not in globals() or focus_detail_df is None or len(focus_detail_df) == 0:
    raise RuntimeError('Сначала выполните 3.1 (нужен DataFrame focus_detail_df).')

focus_n_agr_values = sorted([x for x in focus_detail_df['n_agr'].dropna().astype(str).unique().tolist() if x])

if not focus_n_agr_values:
    print('Для focus-ключей не найден n_agr. Пропускаем SQL-диагностику.')
    perimeter_diag_df = pd.DataFrame()
else:
    focus_n_agr_sql = ', '.join([f"'{x}'" for x in focus_n_agr_values])

    sql_perimeter_diag = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({focus_n_agr_sql})
    ),
    m_all as (
      select
        sa.n_agr,
        sa.n_cmp_client,
        cast(m.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants m
        on cast(m.n_cmp as string) = sa.n_cmp_client
      where m.c_nmrc is not null
        and coalesce(m.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(m.c_nmrc as string)
    ),
    term_all as (
      select
        ma.n_agr,
        ma.n_cmp_client,
        ma.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m_all ma
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = ma.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
      group by ma.n_agr, ma.n_cmp_client, ma.c_nmrc, cast(t.c_nter as string)
    ),
    term_active_month as (
      select
        ma.n_agr,
        ma.n_cmp_client,
        ma.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m_all ma
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = ma.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by ma.n_agr, ma.n_cmp_client, ma.c_nmrc, cast(t.c_nter as string)
    ),
    retl_all as (
      select n_agr, count(distinct c_nmrc) as retl_all_cnt
      from m_all
      group by n_agr
    ),
    retl_active as (
      select n_agr, count(distinct c_nmrc) as retl_active_cnt
      from term_active_month
      group by n_agr
    ),
    term_all_agg as (
      select n_agr, count(distinct c_nter) as term_all_cnt
      from term_all
      group by n_agr
    ),
    term_active_agg as (
      select n_agr, count(distinct c_nter) as term_active_cnt
      from term_active_month
      group by n_agr
    )
    select
      sa.agr_id,
      sa.n_agr,
      sa.n_cmp_client,
      ra.retl_all_cnt,
      rv.retl_active_cnt,
      ta.term_all_cnt,
      tv.term_active_cnt
    from sa_agr sa
    left join retl_all ra on ra.n_agr = sa.n_agr
    left join retl_active rv on rv.n_agr = sa.n_agr
    left join term_all_agg ta on ta.n_agr = sa.n_agr
    left join term_active_agg tv on tv.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        perimeter_diag_df = imp.fetch(sql_perimeter_diag)

    if perimeter_diag_df is None:
        perimeter_diag_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'retl_all_cnt', 'retl_active_cnt', 'term_all_cnt', 'term_active_cnt'])

print(f'perimeter_diag rows: {len(perimeter_diag_df):,}')

if len(perimeter_diag_df):
    perimeter_diag_df['agr_id_key'] = perimeter_diag_df['agr_id'].apply(normalize_agr_q1)
    perimeter_focus_cmp_df = focus_detail_df.merge(perimeter_diag_df, on=['agr_id_key', 'n_agr', 'n_cmp_client'], how='left')

    # Восстанавливаем дельты/abs при частичных перезапусках (чтобы 3.2 была self-healing)
    for m in ['retl_cnt', 'term_cnt']:
        lake_col = f'{m}_lake'
        excel_col = f'{m}_excel'
        delta_col = f'{m}_delta'
        abs_col = f'{m}_abs'

        if delta_col not in perimeter_focus_cmp_df.columns:
            perimeter_focus_cmp_df[delta_col] = (
                pd.to_numeric(perimeter_focus_cmp_df.get(lake_col), errors='coerce').fillna(0)
                - pd.to_numeric(perimeter_focus_cmp_df.get(excel_col), errors='coerce').fillna(0)
            )
        if abs_col not in perimeter_focus_cmp_df.columns:
            perimeter_focus_cmp_df[abs_col] = pd.to_numeric(perimeter_focus_cmp_df[delta_col], errors='coerce').abs()

    # Удобные диагностические флаги
    perimeter_focus_cmp_df['flag_lake_retl_missing'] = perimeter_focus_cmp_df['retl_cnt_lake'].isna() & perimeter_focus_cmp_df['retl_cnt_excel'].notna()
    perimeter_focus_cmp_df['flag_excel_gt_retl_active'] = perimeter_focus_cmp_df['retl_cnt_excel'].fillna(0) > perimeter_focus_cmp_df['retl_active_cnt'].fillna(0)
    perimeter_focus_cmp_df['flag_excel_gt_term_active'] = perimeter_focus_cmp_df['term_cnt_excel'].fillna(0) > perimeter_focus_cmp_df['term_active_cnt'].fillna(0)

    print('Диагностика причин по focus-ключам:')
    display_cols = [
        'agr_id_key', 'inn_key', 'n_agr', 'company_name',
        'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'retl_cnt_abs',
        'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta', 'term_cnt_abs',
        'retl_all_cnt', 'retl_active_cnt', 'term_all_cnt', 'term_active_cnt',
        'flag_lake_retl_missing', 'flag_excel_gt_retl_active', 'flag_excel_gt_term_active'
    ]
    display_cols = [c for c in display_cols if c in perimeter_focus_cmp_df.columns]

    display_df = perimeter_focus_cmp_df[display_cols].copy()
    sort_cols = [c for c in ['retl_cnt_abs', 'term_cnt_abs'] if c in display_df.columns]
    if sort_cols:
        display_df = display_df.sort_values(sort_cols, ascending=False)

    display(display_df.head(100))
else:
    print('Нет данных SQL-диагностики: проверьте focus_n_agr_values и права на таблицы.')

### 3.3 Проверка гипотезы: считать `retl_cnt`/`term_cnt` без фильтра активных терминалов

Сравниваем для focus-ключей два подхода:
- `active` (текущая логика из Lake),
- `all` (без фильтра по датам установки/закрытия терминалов).

Если `all` системно ближе к Excel, значит причина расхождений действительно в фильтре `term_active`.

In [ ]:
# 3.3 Сравнение quality: active vs all (без фильтра активных терминалов)
if 'perimeter_focus_cmp_df' not in globals() or perimeter_focus_cmp_df is None or len(perimeter_focus_cmp_df) == 0:
    raise RuntimeError('Сначала выполните 3.2 (нужен DataFrame perimeter_focus_cmp_df).')

hyp_df = perimeter_focus_cmp_df.copy()

num_cols = [
    'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt',
    'term_cnt_excel', 'term_active_cnt', 'term_all_cnt'
]
for c in num_cols:
    if c not in hyp_df.columns:
        hyp_df[c] = pd.NA
    hyp_df[c] = pd.to_numeric(hyp_df[c], errors='coerce')

# Абсолютные дельты к Excel
hyp_df['retl_abs_active'] = (hyp_df['retl_cnt_excel'] - hyp_df['retl_active_cnt']).abs()
hyp_df['retl_abs_all'] = (hyp_df['retl_cnt_excel'] - hyp_df['retl_all_cnt']).abs()
hyp_df['term_abs_active'] = (hyp_df['term_cnt_excel'] - hyp_df['term_active_cnt']).abs()
hyp_df['term_abs_all'] = (hyp_df['term_cnt_excel'] - hyp_df['term_all_cnt']).abs()

# Где вариант "без active-фильтра" лучше
hyp_df['retl_better_without_active'] = hyp_df['retl_abs_all'] < hyp_df['retl_abs_active']
hyp_df['term_better_without_active'] = hyp_df['term_abs_all'] < hyp_df['term_abs_active']

# Величина улучшения (положительное = all лучше)
hyp_df['retl_improvement_abs'] = hyp_df['retl_abs_active'] - hyp_df['retl_abs_all']
hyp_df['term_improvement_abs'] = hyp_df['term_abs_active'] - hyp_df['term_abs_all']

summary_rows = [
    {
        'metric': 'retl_cnt',
        'rows_total': int(len(hyp_df)),
        'better_without_active_cnt': int(hyp_df['retl_better_without_active'].fillna(False).sum()),
        'better_without_active_pct': round(100 * hyp_df['retl_better_without_active'].fillna(False).mean(), 2),
        'mae_active': round(float(hyp_df['retl_abs_active'].mean()), 4) if len(hyp_df) else 0.0,
        'mae_all': round(float(hyp_df['retl_abs_all'].mean()), 4) if len(hyp_df) else 0.0,
    },
    {
        'metric': 'term_cnt',
        'rows_total': int(len(hyp_df)),
        'better_without_active_cnt': int(hyp_df['term_better_without_active'].fillna(False).sum()),
        'better_without_active_pct': round(100 * hyp_df['term_better_without_active'].fillna(False).mean(), 2),
        'mae_active': round(float(hyp_df['term_abs_active'].mean()), 4) if len(hyp_df) else 0.0,
        'mae_all': round(float(hyp_df['term_abs_all'].mean()), 4) if len(hyp_df) else 0.0,
    }
]

hypothesis_summary_df = pd.DataFrame(summary_rows)
print('Гипотеза: вариант без фильтра active ближе к Excel')
display(hypothesis_summary_df)

print('\nTOP кейсов, где all заметно лучше для retl_cnt:')
display(
    hyp_df.sort_values('retl_improvement_abs', ascending=False)[[
        'agr_id_key', 'inn_key', 'company_name',
        'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt',
        'retl_abs_active', 'retl_abs_all', 'retl_improvement_abs'
    ]].head(30)
)

print('TOP кейсов, где all заметно лучше для term_cnt:')
display(
    hyp_df.sort_values('term_improvement_abs', ascending=False)[[
        'agr_id_key', 'inn_key', 'company_name',
        'term_cnt_excel', 'term_active_cnt', 'term_all_cnt',
        'term_abs_active', 'term_abs_all', 'term_improvement_abs'
    ]].head(30)
)

# Доп. срез: где текущая Lake-логика сильно ниже Excel и all закрывает разрыв
retl_recovered_df = hyp_df[
    (hyp_df['retl_cnt_excel'].fillna(0) > hyp_df['retl_active_cnt'].fillna(0))
    & (hyp_df['retl_all_cnt'].fillna(0) >= hyp_df['retl_cnt_excel'].fillna(0))
].copy()

print(f"Кейсов, где retl_active < Excel, но retl_all >= Excel: {len(retl_recovered_df):,}")
if not retl_recovered_df.empty:
    display(retl_recovered_df[['agr_id_key', 'inn_key', 'company_name', 'retl_cnt_excel', 'retl_active_cnt', 'retl_all_cnt']].head(30))

### 3.4 Проверка гипотезы: расхождения из-за нескольких `n_agr` внутри одного `agr_id`

Секция проверяет, насколько текущая агрегация Lake по ключу `inn+agr_id` через `max` может давать расхождения, когда в месяце у одного `agr_id` несколько версий договора (`n_agr`).

Сравниваются два варианта агрегации Lake для таких ключей:
- `max` (как сейчас),
- `sum` (альтернативная гипотеза).

Далее считается, какой вариант ближе к Excel по абсолютной дельте.

In [ ]:
# 3.4 multi n_agr hypothesis check: lake max vs lake sum
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')

# Нужны те же ключи и метрики, что в compare
metrics_check = [
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_total', 'int_component', 'commission_monthly',
    'chod', 'aur', 'fin_result'
]

lk_base = final_df.copy()
lk_base['inn_key'] = lk_base['inn'].apply(normalize_inn_q1)
lk_base['agr_id_key'] = lk_base['agr_id'].apply(normalize_agr_q1)
lk_base['n_agr_key'] = lk_base['n_agr'].astype(str)

for m in metrics_check:
    lk_base[f'{m}_lake'] = pd.to_numeric(lk_base[m], errors='coerce')

lk_base = lk_base.dropna(subset=['inn_key', 'agr_id_key'])

# Ключи, где в Lake больше 1 n_agr внутри месяца
multi_nagr_keys_df = (
    lk_base.groupby(['inn_key', 'agr_id_key'], as_index=False)
           .agg(n_agr_nunique=('n_agr_key', 'nunique'), rows=('n_agr_key', 'size'))
           .query('n_agr_nunique > 1')
           .sort_values(['n_agr_nunique', 'rows'], ascending=False)
)

print(f"Ключей inn+agr_id с несколькими n_agr: {len(multi_nagr_keys_df):,}")
if not multi_nagr_keys_df.empty:
    display(multi_nagr_keys_df.head(20))

if multi_nagr_keys_df.empty:
    print('Гипотеза по multi n_agr не подтверждается на текущем наборе: таких ключей нет.')
else:
    # Excel aggregation (как в compare-секции)
    ex_local = pd.read_excel(excel_path, header=excel_header)
    col_map_local = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur'],
        'fin_result_col': ['Фин. Рез.', 'Фин.Рез.', 'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат', 'fin_result'],
    }
    resolved_local = {k: pick_col_robust(ex_local.columns, v) for k, v in col_map_local.items()}
    missing_local = [k for k, v in resolved_local.items() if v is None]
    if missing_local:
        raise ValueError(f'Для 3.4 не найдены колонки Excel: {missing_local}')

    ex_local['inn_key'] = ex_local[resolved_local['inn_col']].apply(normalize_inn_q1)
    ex_local['agr_id_key'] = ex_local[resolved_local['agr_col']].apply(normalize_agr_q1)

    ex_local['retl_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['retl_col']], errors='coerce')
    ex_local['term_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['term_col']], errors='coerce')
    ex_local['trx_cnt_excel'] = pd.to_numeric(ex_local[resolved_local['trx_cnt_col']], errors='coerce')
    ex_local['trx_sum_excel'] = to_num_series(ex_local[resolved_local['trx_sum_col']])
    ex_local['commission_total_excel'] = to_num_series(ex_local[resolved_local['comm_total_col']])
    ex_local['int_component_excel'] = to_num_series(ex_local[resolved_local['int_component_col']])
    ex_local['commission_monthly_excel'] = to_num_series(ex_local[resolved_local['comm_monthly_col']])
    ex_local['chod_excel'] = to_num_series(ex_local[resolved_local['chod_col']])
    ex_local['aur_excel'] = to_num_series(ex_local[resolved_local['aur_col']])
    ex_local['fin_result_excel'] = to_num_series(ex_local[resolved_local['fin_result_col']])

    ex_local_agg = (
        ex_local.dropna(subset=['inn_key', 'agr_id_key'])
                .groupby(['inn_key', 'agr_id_key'], as_index=False)
                .agg({
                    'retl_cnt_excel': 'max',
                    'term_cnt_excel': 'max',
                    'trx_cnt_excel': 'max',
                    'trx_sum_excel': 'sum',
                    'commission_total_excel': 'sum',
                    'int_component_excel': 'sum',
                    'commission_monthly_excel': 'max',
                    'chod_excel': 'sum',
                    'aur_excel': 'sum',
                    'fin_result_excel': 'sum',
                })
    )

    # Оставляем только multi-keys
    lk_multi = lk_base.merge(multi_nagr_keys_df[['inn_key', 'agr_id_key']], on=['inn_key', 'agr_id_key'], how='inner')

    agg_map_max = {f'{m}_lake': 'max' for m in metrics_check}
    agg_map_sum = {f'{m}_lake': 'sum' for m in metrics_check}

    lk_multi_max = lk_multi.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(agg_map_max)
    lk_multi_sum = lk_multi.groupby(['inn_key', 'agr_id_key'], as_index=False).agg(agg_map_sum)

    cmp_multi_max = lk_multi_max.merge(ex_local_agg, on=['inn_key', 'agr_id_key'], how='inner')
    cmp_multi_sum = lk_multi_sum.merge(ex_local_agg, on=['inn_key', 'agr_id_key'], how='inner')

    for m in metrics_check:
        cmp_multi_max[f'{m}_abs'] = (cmp_multi_max[f'{m}_lake'] - cmp_multi_max[f'{m}_excel']).abs()
        cmp_multi_sum[f'{m}_abs'] = (cmp_multi_sum[f'{m}_lake'] - cmp_multi_sum[f'{m}_excel']).abs()

    summary_rows = []
    for m in metrics_check:
        mae_max = float(cmp_multi_max[f'{m}_abs'].mean()) if len(cmp_multi_max) else 0.0
        mae_sum = float(cmp_multi_sum[f'{m}_abs'].mean()) if len(cmp_multi_sum) else 0.0

        pair_df = cmp_multi_max[['inn_key', 'agr_id_key', f'{m}_abs']].merge(
            cmp_multi_sum[['inn_key', 'agr_id_key', f'{m}_abs']],
            on=['inn_key', 'agr_id_key'],
            suffixes=('_max', '_sum')
        )
        better_sum_cnt = int((pair_df[f'{m}_abs_sum'] < pair_df[f'{m}_abs_max']).sum())
        better_max_cnt = int((pair_df[f'{m}_abs_max'] < pair_df[f'{m}_abs_sum']).sum())
        equal_cnt = int((pair_df[f'{m}_abs_max'] == pair_df[f'{m}_abs_sum']).sum())

        summary_rows.append({
            'metric': m,
            'rows_compared_multi_keys': int(len(pair_df)),
            'mae_lake_max': round(mae_max, 4),
            'mae_lake_sum': round(mae_sum, 4),
            'better_sum_cnt': better_sum_cnt,
            'better_max_cnt': better_max_cnt,
            'equal_cnt': equal_cnt,
        })

    multi_nagr_hypothesis_df = pd.DataFrame(summary_rows)

    print('Итог гипотезы (multi n_agr: max vs sum):')
    display(multi_nagr_hypothesis_df)

    # Детализация на ключах для retl/term (как самых проблемных)
    details_metrics = ['retl_cnt', 'term_cnt']
    for dm in details_metrics:
        dm_pair = cmp_multi_max[['inn_key', 'agr_id_key', f'{dm}_lake', f'{dm}_excel', f'{dm}_abs']].merge(
            cmp_multi_sum[['inn_key', 'agr_id_key', f'{dm}_lake', f'{dm}_abs']],
            on=['inn_key', 'agr_id_key'],
            suffixes=('_max', '_sum')
        )
        dm_pair['improvement_sum_vs_max'] = dm_pair[f'{dm}_abs_max'] - dm_pair[f'{dm}_abs_sum']

        print(f"TOP ключей, где SUM лучше MAX для {dm}:")
        display(dm_pair.sort_values('improvement_sum_vs_max', ascending=False).head(30))

### 3.5 Кейс-диагностика `n_agr=30841`: почему `retl/term` не попадают в active

Секция детально проверяет воронку отбора терминалов по условиям active-логики:
- есть ли терминальный ID,
- не удалена ли запись,
- `d_ter_install <= month_end`,
- `d_ter_close >= month_start`.

Итог: показывает, на каком шаге происходит выпадение и почему `retl_cnt_lake`/`term_cnt_lake` становятся `NA` или сильно ниже Excel.

In [ ]:
# 3.5 Deep-dive по кейсу n_agr=30841
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_n_agr = '30841'
case_month_start = pd.to_datetime(month_start)
case_month_end = pd.to_datetime(month_end)

# 1) Контекст договора
sql_case_context = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  cast(c.c_cmp_name as string) as company_name,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.n_agr as string) = '{case_n_agr}'
order by cast(a.d_valid_from as date) desc
"""

# 2) Все терминалы по цепочке n_agr -> n_cmp_client -> c_nmrc -> terminals (без active-фильтра)
sql_case_terminals = f"""
with agr as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.abs_agr_id as string) as agr_id
  from ods_alpha.scd1_agreements a
  where cast(a.n_agr as string) = '{case_n_agr}'
),
m_all as (
  select
    agr.agr_id,
    agr.n_agr,
    agr.n_cmp_client,
    cast(m.c_nmrc as string) as c_nmrc,
    coalesce(m.ods_deleted_flg, '0') as m_deleted_flg
  from agr
  join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = agr.n_cmp_client
  where m.c_nmrc is not null
  group by agr.agr_id, agr.n_agr, agr.n_cmp_client, cast(m.c_nmrc as string), coalesce(m.ods_deleted_flg, '0')
)
select
  m.agr_id,
  m.n_agr,
  m.n_cmp_client,
  m.c_nmrc,
  m.m_deleted_flg,
  cast(t.c_nter as string) as c_nter,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
from m_all m
left join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = m.c_nmrc
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    case_context_df = imp.fetch(sql_case_context)
    case_terminals_df = imp.fetch(sql_case_terminals)

if case_context_df is None:
    case_context_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'company_name', 'inn'])
if case_terminals_df is None:
    case_terminals_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter', 'd_ter_install', 'd_ter_close', 't_deleted_flg'])

print(f'Кейс n_agr={case_n_agr}')
print(f'Контекст строк: {len(case_context_df):,}')
display(case_context_df)

if case_terminals_df.empty:
    print('По кейсу не найдено терминальных данных в цепочке merchants -> terminals.')
else:
    # Нормализация и флаги active-логики
    case_terminals_df['d_ter_install'] = pd.to_datetime(case_terminals_df['d_ter_install'], errors='coerce')
    case_terminals_df['d_ter_close'] = pd.to_datetime(case_terminals_df['d_ter_close'], errors='coerce')

    c_nter_series = case_terminals_df['c_nter'].astype(str).str.strip()
    case_terminals_df['is_c_nter_not_null'] = case_terminals_df['c_nter'].notna() & (~c_nter_series.isin(['', 'None', 'nan']))
    case_terminals_df['is_not_deleted'] = case_terminals_df['t_deleted_flg'].fillna('0').astype(str).ne('1')

    # Без fillna(Timestamp) — так стабильнее на текущей версии pandas
    case_terminals_df['is_install_ok'] = case_terminals_df['d_ter_install'].isna() | (case_terminals_df['d_ter_install'] <= case_month_end)
    case_terminals_df['is_close_ok'] = case_terminals_df['d_ter_close'].isna() | (case_terminals_df['d_ter_close'] >= case_month_start)

    case_terminals_df['is_active_term_logic'] = (
        case_terminals_df['is_c_nter_not_null']
        & case_terminals_df['is_not_deleted']
        & case_terminals_df['is_install_ok']
        & case_terminals_df['is_close_ok']
    )

    def _inactive_reason(row):
        if not row['is_c_nter_not_null']:
            return 'no_terminal_id'
        if not row['is_not_deleted']:
            return 'terminal_deleted'
        if not row['is_install_ok']:
            return 'install_after_month_end'
        if not row['is_close_ok']:
            return 'closed_before_month_start'
        return 'active_in_month'

    case_terminals_df['active_reason'] = case_terminals_df.apply(_inactive_reason, axis=1)

    # Сводка причин
    def _safe_term_nunique(series):
        s = series.astype(str).str.strip()
        s = s[~s.isin(['', 'None', 'nan', 'NaN', 'NaT'])]
        return int(s.nunique())

    reason_summary_df = (
        case_terminals_df.groupby('active_reason', as_index=False)
        .agg(
            rows=('c_nmrc', 'size'),
            retl_nunique=('c_nmrc', 'nunique'),
            term_nunique=('c_nter', _safe_term_nunique)
        )
        .sort_values('rows', ascending=False)
        .reset_index(drop=True)
    )

    # Воронка как в расчете active
    step_rows = [
        {
            'step': 'retl_all_cnt (без active-фильтра)',
            'value': int(case_terminals_df['c_nmrc'].nunique())
        },
        {
            'step': 'term_all_cnt (без active-фильтра)',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'], 'c_nter'])
        },
        {
            'step': 'term_not_deleted_cnt',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'] & case_terminals_df['is_not_deleted'], 'c_nter'])
        },
        {
            'step': 'term_install_ok_cnt',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_c_nter_not_null'] & case_terminals_df['is_not_deleted'] & case_terminals_df['is_install_ok'], 'c_nter'])
        },
        {
            'step': 'term_active_cnt (логика Lake)',
            'value': _safe_term_nunique(case_terminals_df.loc[case_terminals_df['is_active_term_logic'], 'c_nter'])
        },
        {
            'step': 'retl_active_cnt (логика Lake)',
            'value': int(case_terminals_df.loc[case_terminals_df['is_active_term_logic'], 'c_nmrc'].nunique())
        },
    ]
    case_funnel_df = pd.DataFrame(step_rows)

    print(f'Терминальных строк по кейсу: {len(case_terminals_df):,}')
    print('Сводка причин попадания/непопадания в active:')
    display(reason_summary_df)

    print('Воронка active-логики для кейса:')
    display(case_funnel_df)

    # Сравнение с cmp по agr_id кейса (если уже посчитано)
    if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
        case_agr_id = None
        if len(case_context_df):
            case_agr_id = normalize_agr_q1(case_context_df['agr_id'].iloc[0])
        if case_agr_id:
            case_cmp_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()
            print(f'Срез cmp для agr_id={case_agr_id}:')
            if not case_cmp_df.empty:
                display(case_cmp_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])
            else:
                print('В cmp нет строки по этому agr_id (проверьте ключи/пересечение).')

    print('TOP-100 терминалов/точек с флагами active-логики:')
    display(
        case_terminals_df[[
            'agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'c_nter',
            'd_ter_install', 'd_ter_close', 't_deleted_flg',
            'is_c_nter_not_null', 'is_not_deleted', 'is_install_ok', 'is_close_ok',
            'is_active_term_logic', 'active_reason'
        ]]
        .sort_values(['is_active_term_logic', 'active_reason', 'c_nmrc', 'c_nter'], ascending=[True, True, True, True])
        .head(100)
    )

### 3.6 Deep-dive по `agr_id=1106410898346`: полная воронка `retl_cnt/term_cnt`

Секция выполняет end-to-end разбор по конкретному `agr_id`:
- все версии договора (`n_agr`) и контекст клиента;
- цепочка `agr_id -> n_agr -> n_cmp_client -> c_nmrc -> c_nter`;
- проверка active-условий для терминалов;
- воронка количеств по шагам;
- сверка с `cmp`/Excel для этого `agr_id`.

In [ ]:
# 3.6 AgrID funnel debug: 1106410898346
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_agr_id = '1106410898346'
case_month_start = pd.to_datetime(month_start)
case_month_end = pd.to_datetime(month_end)

# 1) Контекст договоров по agr_id (все n_agr)
sql_agr_context = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  cast(c.c_cmp_name as string) as company_name,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
from ods_alpha.scd1_agreements a
left join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.abs_agr_id as string) = '{case_agr_id}'
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

# 2) Все точки/терминалы по цепочке (без active-фильтра)
sql_chain_all = f"""
with agr as (
  select distinct
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) = '{case_agr_id}'
),
m_all as (
  select
    agr.agr_id,
    agr.n_agr,
    agr.n_cmp_client,
    cast(m.c_nmrc as string) as c_nmrc,
    coalesce(m.ods_deleted_flg, '0') as m_deleted_flg
  from agr
  join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = agr.n_cmp_client
  where m.c_nmrc is not null
  group by agr.agr_id, agr.n_agr, agr.n_cmp_client, cast(m.c_nmrc as string), coalesce(m.ods_deleted_flg, '0')
)
select
  m.agr_id,
  m.n_agr,
  m.n_cmp_client,
  m.c_nmrc,
  m.m_deleted_flg,
  cast(t.c_nter as string) as c_nter,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
from m_all m
left join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = m.c_nmrc
"""

# 3) Текущие lake-метрики из cmp/final_df для этого agr_id
case_cmp_df = pd.DataFrame()
if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    case_cmp_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()

case_final_df = pd.DataFrame()
if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and len(final_df):
    ff = final_df.copy()
    ff['agr_id_key'] = ff['agr_id'].apply(normalize_agr_q1)
    case_final_df = ff[ff['agr_id_key'] == case_agr_id].copy()

with imp:
    imp.execute('set MEM_LIMIT=8g')
    agr_context_df = imp.fetch(sql_agr_context)
    chain_all_df = imp.fetch(sql_chain_all)

if agr_context_df is None:
    agr_context_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'agr_deleted_flg', 'company_name', 'inn'])
if chain_all_df is None:
    chain_all_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter', 'd_ter_install', 'd_ter_close', 't_deleted_flg'])

print(f'agr_id={case_agr_id}')
print(f'contracts rows: {len(agr_context_df):,}')
display(agr_context_df)

if chain_all_df.empty:
    print('Не найдены точки/терминалы по цепочке merchants -> terminals.')
else:
    chain_all_df['d_ter_install'] = pd.to_datetime(chain_all_df['d_ter_install'], errors='coerce')
    chain_all_df['d_ter_close'] = pd.to_datetime(chain_all_df['d_ter_close'], errors='coerce')

    c_nter_series = chain_all_df['c_nter'].astype(str).str.strip()
    chain_all_df['is_c_nter_not_null'] = chain_all_df['c_nter'].notna() & (~c_nter_series.isin(['', 'None', 'nan']))
    chain_all_df['is_not_deleted'] = chain_all_df['t_deleted_flg'].fillna('0').astype(str).ne('1')
    chain_all_df['is_install_ok'] = chain_all_df['d_ter_install'].isna() | (chain_all_df['d_ter_install'] <= case_month_end)
    chain_all_df['is_close_ok'] = chain_all_df['d_ter_close'].isna() | (chain_all_df['d_ter_close'] >= case_month_start)

    chain_all_df['is_active_term_logic'] = (
        chain_all_df['is_c_nter_not_null']
        & chain_all_df['is_not_deleted']
        & chain_all_df['is_install_ok']
        & chain_all_df['is_close_ok']
    )

    def _inactive_reason(row):
        if not row['is_c_nter_not_null']:
            return 'no_terminal_id'
        if not row['is_not_deleted']:
            return 'terminal_deleted'
        if not row['is_install_ok']:
            return 'install_after_month_end'
        if not row['is_close_ok']:
            return 'closed_before_month_start'
        return 'active_in_month'

    chain_all_df['active_reason'] = chain_all_df.apply(_inactive_reason, axis=1)

    def _safe_term_nunique(series):
        s = series.astype(str).str.strip()
        s = s[~s.isin(['', 'None', 'nan', 'NaN', 'NaT'])]
        return int(s.nunique())

    # 4) Воронка по каждому n_agr
    funnel_rows = []
    for n_agr_val, g in chain_all_df.groupby('n_agr'):
        funnel_rows.extend([
            {'n_agr': n_agr_val, 'step': 'retl_all_cnt', 'value': int(g['c_nmrc'].nunique())},
            {'n_agr': n_agr_val, 'step': 'term_all_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_not_deleted_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'] & g['is_not_deleted'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_install_ok_cnt', 'value': _safe_term_nunique(g.loc[g['is_c_nter_not_null'] & g['is_not_deleted'] & g['is_install_ok'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'term_active_cnt', 'value': _safe_term_nunique(g.loc[g['is_active_term_logic'], 'c_nter'])},
            {'n_agr': n_agr_val, 'step': 'retl_active_cnt', 'value': int(g.loc[g['is_active_term_logic'], 'c_nmrc'].nunique())},
        ])

    agr_funnel_df = pd.DataFrame(funnel_rows)

    # 5) Сводка причин выпадения по n_agr
    reason_summary_df = (
        chain_all_df.groupby(['n_agr', 'active_reason'], as_index=False)
        .agg(
            rows=('c_nmrc', 'size'),
            retl_nunique=('c_nmrc', 'nunique'),
            term_nunique=('c_nter', _safe_term_nunique)
        )
        .sort_values(['n_agr', 'rows'], ascending=[True, False])
        .reset_index(drop=True)
    )

    print(f'chain rows: {len(chain_all_df):,}; n_agr involved: {chain_all_df["n_agr"].nunique():,}')
    print('Воронка retl/term по каждому n_agr:')
    display(agr_funnel_df)

    print('Причины выпадения из active по каждому n_agr:')
    display(reason_summary_df)

    print('Сырые строки chain с флагами (TOP-200):')
    display(
        chain_all_df[[
            'agr_id', 'n_agr', 'n_cmp_client', 'c_nmrc', 'm_deleted_flg', 'c_nter',
            'd_ter_install', 'd_ter_close', 't_deleted_flg',
            'is_c_nter_not_null', 'is_not_deleted', 'is_install_ok', 'is_close_ok',
            'is_active_term_logic', 'active_reason'
        ]]
        .sort_values(['n_agr', 'is_active_term_logic', 'active_reason', 'c_nmrc', 'c_nter'], ascending=[True, True, True, True, True])
        .head(200)
    )

# 6) Сверка с cmp/final_df/Excel
print('\nСрез final_df по agr_id:')
if case_final_df.empty:
    print('В final_df нет строк по этому agr_id.')
else:
    display(case_final_df[['report_month', 'inn', 'agr_id', 'n_agr', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'company_name']])

print('Срез cmp по agr_id:')
if case_cmp_df.empty:
    print('В cmp нет строки по этому agr_id (проверьте пересечение ключей).')
else:
    display(case_cmp_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])

### 3.7 Разбор кейса `agr_id=1106410898346` через `n_agr=45011`

Секция проверяет путь формирования метрик именно через `n_agr=45011` и отвечает на вопрос:
- находится ли `agr_id=1106410898346` в SA-периметре как `abs_agr_id`;
- находится ли `n_agr=45011` в SA-периметре;
- не подставился ли `agr_id` через fallback-логику.

In [ ]:
# 3.7 Path debug: agr_id=1106410898346 via n_agr=45011
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров периода.')

case_agr_id = '1106410898346'
case_n_agr = '45011'

# 1) Проверка в уже посчитанном sa_df (если есть)
sa_df_check_df = pd.DataFrame()
if 'sa_df' in globals() and isinstance(sa_df, pd.DataFrame) and len(sa_df):
    tmp = sa_df.copy()
    tmp['agr_id_key'] = tmp['agr_id'].apply(normalize_agr_q1)
    tmp['n_agr_key'] = tmp['n_agr'].astype(str)
    sa_df_check_df = tmp[(tmp['agr_id_key'] == case_agr_id) | (tmp['n_agr_key'] == case_n_agr)].copy()

# 2) Проверка SA-периметра напрямую SQL (по тем же правилам, что в секции 01)
sql_sa_case_check = f"""
select distinct
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  upper(trim(cast(a.acq_class as string))) as acq_class
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
  and (
      cast(a.abs_agr_id as string) = '{case_agr_id}'
      or cast(a.n_agr as string) = '{case_n_agr}'
  )
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

# 3) Проверка agreements без SA/terms-фильтра, чтобы увидеть связь n_agr -> abs_agr_id
sql_agreements_raw_check = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  coalesce(a.ods_deleted_flg, '0') as agr_deleted_flg,
  upper(trim(cast(a.acq_class as string))) as acq_class
from ods_alpha.scd1_agreements a
where cast(a.abs_agr_id as string) = '{case_agr_id}'
   or cast(a.n_agr as string) = '{case_n_agr}'
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_sql_check_df = imp.fetch(sql_sa_case_check)
    agreements_raw_check_df = imp.fetch(sql_agreements_raw_check)

if sa_sql_check_df is None:
    sa_sql_check_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'inn', 'company_name', 'agr_deleted_flg', 'acq_class'])
if agreements_raw_check_df is None:
    agreements_raw_check_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'agr_deleted_flg', 'acq_class'])

# 4) Срез base_df/final_df для понимания пути подстановки
base_case_df = pd.DataFrame()
if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df):
    b = base_df.copy()
    b['agr_id_key'] = b['agr_id'].apply(normalize_agr_q1)
    b['n_agr_key'] = b['n_agr'].astype(str)
    keep_cols = [
        'agr_id', 'n_agr', 'agr_id_source', 'r2_id', 'n_cmp_client', 'cft_id',
        'company_name', 'inn', 'retl_cnt', 'term_cnt', 'active_term_cnt'
    ]
    keep_cols = [c for c in keep_cols if c in b.columns]
    base_case_df = b[(b['agr_id_key'] == case_agr_id) | (b['n_agr_key'] == case_n_agr)][keep_cols].copy()

final_case_df = pd.DataFrame()
if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and len(final_df):
    f = final_df.copy()
    f['agr_id_key'] = f['agr_id'].apply(normalize_agr_q1)
    f['n_agr_key'] = f['n_agr'].astype(str)
    final_case_df = f[(f['agr_id_key'] == case_agr_id) | (f['n_agr_key'] == case_n_agr)].copy()

# 5) Явный ответ на ключевой вопрос
sa_sql_check_df['agr_id_key'] = sa_sql_check_df['agr_id'].apply(normalize_agr_q1) if len(sa_sql_check_df) else pd.Series(dtype=object)
sa_sql_check_df['n_agr_key'] = sa_sql_check_df['n_agr'].astype(str) if len(sa_sql_check_df) else pd.Series(dtype=object)

is_agr_id_in_sa = bool((sa_sql_check_df['agr_id_key'] == case_agr_id).any()) if len(sa_sql_check_df) else False
is_n_agr_in_sa = bool((sa_sql_check_df['n_agr_key'] == case_n_agr).any()) if len(sa_sql_check_df) else False

print(f'Проверка кейса: agr_id={case_agr_id}, n_agr={case_n_agr}')
print(f'agr_id в SA-периметре (как abs_agr_id): {is_agr_id_in_sa}')
print(f'n_agr в SA-периметре: {is_n_agr_in_sa}')

if is_n_agr_in_sa and not is_agr_id_in_sa:
    print('Вывод: метрики идут через n_agr, а agr_id в итоге, вероятно, подставляется fallback-логикой (не как прямой abs_agr_id из SA).')

print('\nSA check via sa_df (если sa_df в памяти):')
if sa_df_check_df.empty:
    print('Нет строк в sa_df по этому agr_id/n_agr (или sa_df не в памяти).')
else:
    display(sa_df_check_df[['agr_id', 'n_agr', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'company_name', 'inn']])

print('\nSA check via SQL perimeter rules:')
display(sa_sql_check_df)

print('\nRaw agreements check (без SA/terms-фильтра):')
display(agreements_raw_check_df)

print('\nbase_df path check:')
if base_case_df.empty:
    print('В base_df нет строк по этому agr_id/n_agr.')
else:
    display(base_case_df)

print('\nfinal_df path check:')
if final_case_df.empty:
    print('В final_df нет строк по этому agr_id/n_agr.')
else:
    display(final_case_df[['report_month', 'inn', 'agr_id', 'n_agr', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'company_name']])

if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    cmp_case_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()
    print('\ncmp check by agr_id:')
    if cmp_case_df.empty:
        print('В cmp нет строк по этому agr_id.')
    else:
        display(cmp_case_df[['agr_id_key', 'inn_key', 'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta', 'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta']])

### 3.8 Проверка R2-слоя: где может лежать `retl=-307` для fallback `agr_id`

Идея проверки:
- берем `agr_id=1106410898346` (fallback из `r2_id`) и его `cft_id`;
- в `ods.scd1_z_r2_ip_merchants` собираем **все** `r2_id` этого `cft_id`, считаем `retl` (`c_nmrc`) и терминалы;
- сравниваем `retl` из R2 с `retl` из `cmp` (`Lake`/`Excel`) и смотрим, может ли "потерянный" объем сидеть на других `r2_id` того же клиента.

In [ ]:
# 3.8 R2-layer check: where retl=-307 may hide for fallback agr_id
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')

case_agr_id = '1106410898346'
case_n_agr = '45011'

# 1) Контекст из base_df/cmp
base_case_38_df = pd.DataFrame()
if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df):
    b = base_df.copy()
    b['agr_id_key'] = b['agr_id'].apply(normalize_agr_q1)
    b['n_agr_key'] = b['n_agr'].astype(str)
    keep_cols = [
        'agr_id', 'n_agr', 'agr_id_source', 'r2_id', 'cft_id',
        'n_cmp_client', 'company_name', 'inn', 'retl_cnt', 'term_cnt', 'active_term_cnt'
    ]
    keep_cols = [c for c in keep_cols if c in b.columns]
    base_case_38_df = b[(b['agr_id_key'] == case_agr_id) | (b['n_agr_key'] == case_n_agr)][keep_cols].copy()

cmp_case_38_df = pd.DataFrame()
if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    cmp_case_38_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()

# 2) Определяем cft_id (сначала из base_df, если нет — из R2 по id)
cft_candidates = []
if not base_case_38_df.empty and 'cft_id' in base_case_38_df.columns:
    cft_candidates = sorted({
        str(x).strip()
        for x in base_case_38_df['cft_id'].dropna().tolist()
        if str(x).strip() not in ['', 'None', 'nan']
    })

cft_id_for_check = cft_candidates[0] if cft_candidates else None
cft_from_r2_df = pd.DataFrame(columns=['cft_id'])

if cft_id_for_check is None:
    sql_cft_from_r2 = f"""
    select distinct cast(m.c_cl_org as string) as cft_id
    from ods.scd1_z_r2_ip_merchants m
    where cast(m.id as string) = '{case_agr_id}'
    """
    with imp:
        imp.execute('set MEM_LIMIT=4g')
        cft_from_r2_df = imp.fetch(sql_cft_from_r2)

    if cft_from_r2_df is None:
        cft_from_r2_df = pd.DataFrame(columns=['cft_id'])

    if len(cft_from_r2_df):
        vals = [str(x).strip() for x in cft_from_r2_df['cft_id'].dropna().tolist() if str(x).strip()]
        if vals:
            cft_id_for_check = vals[0]

print(f'Кейс 3.8: agr_id={case_agr_id}, n_agr={case_n_agr}')
print(f'cft_id для проверки: {cft_id_for_check}')

if not base_case_38_df.empty:
    print('\nКонтекст из base_df:')
    display(base_case_38_df)

if not cft_from_r2_df.empty:
    print('\nCFT из R2 по fallback agr_id:')
    display(cft_from_r2_df)

if cft_id_for_check is None:
    print('Не удалось определить cft_id. Выполните секции 01-10 и 3.7, затем повторите 3.8.')
else:
    cft_sql = cft_id_for_check.replace("'", "''")

    # 3) Проверка схемы R2-таблицы, чтобы не ссылаться на несуществующие поля
    with imp:
        imp.execute('set MEM_LIMIT=2g')
        r2_desc_df = imp.fetch('describe ods.scd1_z_r2_ip_merchants')

    if r2_desc_df is None:
        r2_desc_df = pd.DataFrame()

    if r2_desc_df.empty:
        print('DESCRIBE ods.scd1_z_r2_ip_merchants вернул пустой результат.')
    else:
        name_col = None
        for cand in ['name', 'col_name', 'column_name']:
            if cand in r2_desc_df.columns:
                name_col = cand
                break
        if name_col is None:
            name_col = r2_desc_df.columns[0]

        r2_cols = {str(x).strip().lower() for x in r2_desc_df[name_col].dropna().tolist()}
        required_base_cols = {'id', 'c_cl_org'}
        missing_base_cols = sorted(required_base_cols - r2_cols)
        r2_retl_col_candidates = ['c_nmrc', 'c_code_in_pr']
        r2_retl_col = next((col for col in r2_retl_col_candidates if col in r2_cols), None)

        if missing_base_cols or r2_retl_col is None:
            print(f'В ods.scd1_z_r2_ip_merchants не найдены базовые колонки: {missing_base_cols}')
            print(f'Кандидаты поля retail-кода (c_nmrc/c_code_in_pr), найдено: {r2_retl_col}')
            print('Проверьте схему таблицы ниже.')
            display(r2_desc_df.head(200))
        else:
            print(f'Для retail-кода в R2 используется колонка: {r2_retl_col}')
            # 4) Все merchant-строки по одному cft_id
            sql_r2_retl_rows = f"""
            select
              cast(m.id as string) as r2_id,
              cast(m.c_cl_org as string) as cft_id,
              cast(m.{r2_retl_col} as string) as c_nmrc,
              coalesce(m.ods_deleted_flg, '0') as m_deleted_flg,
              cast(m.c_tariff_plan as string) as c_tariff_plan
            from ods.scd1_z_r2_ip_merchants m
            where cast(m.c_cl_org as string) = '{cft_sql}'
              and m.{r2_retl_col} is not null
            """

            # 5) Агрегаты по каждому r2_id клиента
            sql_r2_counts_by_id = f"""
            with m as (
              select
                cast(id as string) as r2_id,
                cast({r2_retl_col} as string) as c_nmrc
              from ods.scd1_z_r2_ip_merchants
              where cast(c_cl_org as string) = '{cft_sql}'
                and {r2_retl_col} is not null
            ),
            mt as (
              select
                m.r2_id,
                m.c_nmrc,
                cast(t.c_nter as string) as c_nter,
                cast(t.d_ter_install as date) as d_ter_install,
                cast(t.d_ter_close as date) as d_ter_close,
                coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
              from m
              left join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = m.c_nmrc
            )
            select
              r2_id,
              count(distinct c_nmrc) as retl_all_cnt,
              count(distinct case when c_nter is not null then c_nter else null end) as term_all_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nmrc else null end
              ) as retl_active_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nter else null end
              ) as term_active_cnt
            from mt
            group by r2_id
            order by retl_all_cnt desc, r2_id
            """

            # 6) Агрегаты по клиенту целиком (по всем его r2_id)
            sql_r2_counts_total = f"""
            with m as (
              select
                cast(id as string) as r2_id,
                cast({r2_retl_col} as string) as c_nmrc
              from ods.scd1_z_r2_ip_merchants
              where cast(c_cl_org as string) = '{cft_sql}'
                and {r2_retl_col} is not null
            ),
            mt as (
              select
                m.r2_id,
                m.c_nmrc,
                cast(t.c_nter as string) as c_nter,
                cast(t.d_ter_install as date) as d_ter_install,
                cast(t.d_ter_close as date) as d_ter_close,
                coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
              from m
              left join ods_alpha.scd1_pos_terminals t
                on cast(t.c_nmrc as string) = m.c_nmrc
            )
            select
              count(distinct c_nmrc) as retl_all_cnt,
              count(distinct case when c_nter is not null then c_nter else null end) as term_all_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nmrc else null end
              ) as retl_active_cnt,
              count(distinct case
                when c_nter is not null
                 and t_deleted_flg <> '1'
                 and (d_ter_install is null or d_ter_install <= cast('{month_end}' as date))
                 and (d_ter_close is null or d_ter_close >= cast('{month_start}' as date))
                then c_nter else null end
              ) as term_active_cnt
            from mt
            """

            with imp:
                imp.execute('set MEM_LIMIT=8g')
                r2_retl_rows_df = imp.fetch(sql_r2_retl_rows)
                r2_counts_by_id_df = imp.fetch(sql_r2_counts_by_id)
                r2_counts_total_df = imp.fetch(sql_r2_counts_total)

            if r2_retl_rows_df is None:
                r2_retl_rows_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'c_nmrc', 'm_deleted_flg', 'c_tariff_plan'])
            if r2_counts_by_id_df is None:
                r2_counts_by_id_df = pd.DataFrame(columns=['r2_id', 'retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt'])
            if r2_counts_total_df is None:
                r2_counts_total_df = pd.DataFrame(columns=['retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt'])

            if not r2_retl_rows_df.empty:
                r2_retl_rows_df['r2_id'] = r2_retl_rows_df['r2_id'].astype(str)
                r2_retl_rows_df['c_nmrc'] = r2_retl_rows_df['c_nmrc'].astype(str).str.strip()

            for col in ['retl_all_cnt', 'term_all_cnt', 'retl_active_cnt', 'term_active_cnt']:
                if col in r2_counts_by_id_df.columns:
                    r2_counts_by_id_df[col] = pd.to_numeric(r2_counts_by_id_df[col], errors='coerce').fillna(0).astype(int)
                if col in r2_counts_total_df.columns:
                    r2_counts_total_df[col] = pd.to_numeric(r2_counts_total_df[col], errors='coerce').fillna(0).astype(int)

            def _max_num(df, col):
                if df is None or not isinstance(df, pd.DataFrame) or df.empty or col not in df.columns:
                    return None
                s = pd.to_numeric(df[col], errors='coerce').dropna()
                if s.empty:
                    return None
                return float(s.max())

            retl_excel = _max_num(cmp_case_38_df, 'retl_cnt_excel')
            retl_lake = _max_num(cmp_case_38_df, 'retl_cnt_lake')

            retl_focus_id = 0
            if not r2_counts_by_id_df.empty:
                m_focus = r2_counts_by_id_df['r2_id'].astype(str) == case_agr_id
                if m_focus.any():
                    retl_focus_id = int(r2_counts_by_id_df.loc[m_focus, 'retl_all_cnt'].max())

            retl_total_r2 = 0
            retl_active_total_r2 = 0
            if not r2_counts_total_df.empty:
                retl_total_r2 = int(r2_counts_total_df['retl_all_cnt'].iloc[0])
                retl_active_total_r2 = int(r2_counts_total_df['retl_active_cnt'].iloc[0])

            retl_other_ids = max(retl_total_r2 - retl_focus_id, 0)

            summary_rows = [
                {'metric': 'retl_cnt_excel (cmp)', 'value': retl_excel},
                {'metric': 'retl_cnt_lake (cmp)', 'value': retl_lake},
                {'metric': 'retl_cnt_r2_focus_id_all', 'value': retl_focus_id},
                {'metric': 'retl_cnt_r2_other_ids_all', 'value': retl_other_ids},
                {'metric': 'retl_cnt_r2_all_ids_all', 'value': retl_total_r2},
                {'metric': 'retl_cnt_r2_all_ids_active_logic', 'value': retl_active_total_r2},
            ]
            if retl_excel is not None and retl_lake is not None:
                summary_rows.append({'metric': 'excel_minus_lake', 'value': retl_excel - retl_lake})
            if retl_excel is not None:
                summary_rows.append({'metric': 'excel_minus_r2_all_ids', 'value': retl_excel - retl_total_r2})
                summary_rows.append({'metric': 'excel_minus_r2_active_logic', 'value': retl_excel - retl_active_total_r2})
            if retl_lake is not None:
                summary_rows.append({'metric': 'r2_all_ids_minus_lake', 'value': retl_total_r2 - retl_lake})

            r2_38_summary_df = pd.DataFrame(summary_rows)

            print('\nR2 coverage summary for same cft_id:')
            display(r2_38_summary_df)

            print('\nR2 counts by r2_id (same cft_id):')
            display(r2_counts_by_id_df)

            print('\nRows in R2 merchants for same cft_id (TOP-200):')
            display(r2_retl_rows_df.sort_values(['r2_id', 'c_nmrc']).head(200))

            if not cmp_case_38_df.empty:
                cmp_cols = [
                    'agr_id_key', 'inn_key',
                    'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta',
                    'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta'
                ]
                cmp_cols = [c for c in cmp_cols if c in cmp_case_38_df.columns]
                print('\ncmp slice for case agr_id:')
                display(cmp_case_38_df[cmp_cols])

### 3.9 Сравнение с Excel без `r2_fallback` + сколько `agr_id` есть только в Excel

Секция проверяет гипотезу, что значимая часть расхождений идет из fallback-кейсов:
- исключаем из Lake строки с `agr_id_source = 'r2_fallback'`;
- пересчитываем compare/KPI в формате Секции 2;
- показываем, сколько ключей и `agr_id` есть только в Excel (отсутствуют в Lake после исключения fallback).

In [ ]:
# 3.9 Compare without r2_fallback
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')
if 'base_df' not in globals() or base_df is None or len(base_df) == 0:
    raise RuntimeError('Сначала выполните секции до 10 (нужен base_df с agr_id_source).')
if 'cmp' not in globals() or cmp is None:
    raise RuntimeError('Сначала выполните Секцию 2 Compare + KPI (нужен cmp).')

# --- 1) Карта источника agr_id: inn+agr_id -> agr_id_source (из base_df)
base_src = base_df.copy()
base_src['inn_key'] = base_src['inn'].apply(normalize_inn_q1)
base_src['agr_id_key'] = base_src['agr_id'].apply(normalize_agr_q1)

if 'agr_id_source' not in base_src.columns:
    base_src['agr_id_source'] = None

base_src = base_src.dropna(subset=['inn_key', 'agr_id_key']).copy()

def _resolve_agr_source(series):
    vals = {
        str(x).strip()
        for x in series.dropna().tolist()
        if str(x).strip() not in ['', 'None', 'nan']
    }
    if 'r2_fallback' in vals:
        return 'r2_fallback'
    if 'sa' in vals:
        return 'sa'
    if len(vals):
        return sorted(vals)[0]
    return None

source_map_df = (
    base_src.groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        agr_id_source=('agr_id_source', _resolve_agr_source),
        base_rows=('agr_id_source', 'size'),
        n_agr_nunique=('n_agr', 'nunique')
    )
)

# --- 2) Подготовка Lake-данных + присоединение source-карты
lk = final_df.copy()
lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')
lk['aur_lake'] = pd.to_numeric(lk['aur'], errors='coerce')
lk['fin_result_lake'] = pd.to_numeric(lk['fin_result'], errors='coerce')

lk = lk.dropna(subset=['inn_key', 'agr_id_key']).copy()
lk = lk.merge(source_map_df[['inn_key', 'agr_id_key', 'agr_id_source']], on=['inn_key', 'agr_id_key'], how='left')
lk['agr_id_source'] = lk['agr_id_source'].fillna('unknown')

source_coverage_df = (
    lk.groupby('agr_id_source', as_index=False)
      .agg(
          rows=('agr_id_source', 'size'),
          key_cnt=('agr_id_key', 'size'),
          agr_id_nunique=('agr_id_key', 'nunique')
      )
      .sort_values('rows', ascending=False)
      .reset_index(drop=True)
)

print('Покрытие final_df по agr_id_source (до фильтра fallback):')
display(source_coverage_df)

metric_keys = [
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_total', 'int_component', 'commission_monthly',
    'chod', 'aur', 'fin_result'
]

# Lake aggregate: общий (до фильтра) и без fallback
lk_agg_all = (
    lk.groupby(['inn_key', 'agr_id_key'], as_index=False)
      .agg({
          'retl_cnt_lake': 'max',
          'term_cnt_lake': 'max',
          'trx_cnt_lake': 'max',
          'trx_sum_lake': 'max',
          'commission_total_lake': 'max',
          'int_component_lake': 'max',
          'commission_monthly_lake': 'max',
          'chod_lake': 'max',
          'aur_lake': 'max',
          'fin_result_lake': 'max',
      })
)

lk_no_fallback = lk[lk['agr_id_source'] != 'r2_fallback'].copy()

lk_agg_no_fallback = (
    lk_no_fallback.groupby(['inn_key', 'agr_id_key'], as_index=False)
      .agg({
          'retl_cnt_lake': 'max',
          'term_cnt_lake': 'max',
          'trx_cnt_lake': 'max',
          'trx_sum_lake': 'max',
          'commission_total_lake': 'max',
          'int_component_lake': 'max',
          'commission_monthly_lake': 'max',
          'chod_lake': 'max',
          'aur_lake': 'max',
          'fin_result_lake': 'max',
      })
)

# --- 3) Excel aggregate (используем ex_agg из Секции 2, если есть; иначе пересобираем)
if 'ex_agg' in globals() and isinstance(ex_agg, pd.DataFrame) and len(ex_agg):
    ex_agg_local = ex_agg.copy()
else:
    if 'excel_path' not in globals() or 'excel_header' not in globals():
        raise RuntimeError('Нет ex_agg и нет excel_path/excel_header для пересборки Excel-агрегата.')

    ex = pd.read_excel(excel_path, header=excel_header)
    col_map_local = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_total_col': ['Комиссия эквайринга'],
        'int_component_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'chod_col': ['ЧОД'],
        'aur_col': ['АУР', 'AUR', 'Aur'],
        'fin_result_col': ['Фин. Рез.', 'Фин.Рез.', 'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат', 'fin_result'],
    }

    resolved_local = {k: pick_col_robust(ex.columns, v) for k, v in col_map_local.items()}
    missing_local = [k for k, v in resolved_local.items() if v is None]
    if missing_local:
        raise ValueError(f'Не найдены колонки Excel при пересборке ex_agg: {missing_local}')

    ex['inn_key'] = ex[resolved_local['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved_local['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved_local['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved_local['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved_local['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved_local['trx_sum_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved_local['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved_local['int_component_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved_local['comm_monthly_col']])
    ex['chod_excel'] = to_num_series(ex[resolved_local['chod_col']])
    ex['aur_excel'] = to_num_series(ex[resolved_local['aur_col']])
    ex['fin_result_excel'] = to_num_series(ex[resolved_local['fin_result_col']])

    ex_agg_local = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'commission_monthly_excel': 'max',
              'chod_excel': 'sum',
              'aur_excel': 'sum',
              'fin_result_excel': 'sum',
          })
    )

# --- 4) Compare no-fallback
cmp_no_fallback = lk_agg_no_fallback.merge(ex_agg_local, on=['inn_key', 'agr_id_key'], how='inner')

for m in metric_keys:
    cmp_no_fallback[f'{m}_delta'] = cmp_no_fallback[f'{m}_lake'].fillna(0) - cmp_no_fallback[f'{m}_excel'].fillna(0)
    cmp_no_fallback[f'{m}_abs'] = cmp_no_fallback[f'{m}_delta'].abs()

count_metrics = ['retl_cnt', 'term_cnt', 'trx_cnt']
money_metrics = ['trx_sum', 'commission_total', 'int_component', 'commission_monthly', 'chod', 'aur', 'fin_result']

kpi_rows_no_fb = [{'metric': 'rows_compared', 'value': len(cmp_no_fallback)}]
for m in count_metrics:
    kpi_rows_no_fb.append({'metric': f'{m}_exact_match_rate_pct_tol_0', 'value': exact_match_rate_from_abs(cmp_no_fallback[f'{m}_abs'], tol=0.0)})
    kpi_rows_no_fb.append({'metric': f'{m}_mae', 'value': float(cmp_no_fallback[f'{m}_abs'].mean()) if len(cmp_no_fallback) else 0.0})
for m in money_metrics:
    kpi_rows_no_fb.append({'metric': f'{m}_exact_match_rate_pct_tol_0_01', 'value': exact_match_rate_from_abs(cmp_no_fallback[f'{m}_abs'], tol=0.01)})
    kpi_rows_no_fb.append({'metric': f'{m}_mae', 'value': float(cmp_no_fallback[f'{m}_abs'].mean()) if len(cmp_no_fallback) else 0.0})

kpi_compare_no_fallback_df = pd.DataFrame(kpi_rows_no_fb)

# --- 5) Сколько ключей/agr_id есть только в Excel (относительно Lake без fallback)
excel_keys_df = ex_agg_local[['inn_key', 'agr_id_key']].drop_duplicates()
lake_no_fallback_keys_df = lk_agg_no_fallback[['inn_key', 'agr_id_key']].drop_duplicates()

excel_only_no_fallback_df = (
    excel_keys_df.merge(lake_no_fallback_keys_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
                .query("_merge == 'left_only'")
                .drop(columns=['_merge'])
                .reset_index(drop=True)
)

excel_only_keys_cnt = int(len(excel_only_no_fallback_df))
excel_only_agr_id_cnt = int(excel_only_no_fallback_df['agr_id_key'].nunique()) if excel_only_keys_cnt else 0

# --- 6) До/после: изменение % совпадений
if 'kpi_compare_df' in globals() and isinstance(kpi_compare_df, pd.DataFrame) and len(kpi_compare_df):
    kpi_before_df = kpi_compare_df.copy()
else:
    # fallback: считаем до/после напрямую из cmp
    kpi_before_rows = [{'metric': 'rows_compared', 'value': len(cmp)}]
    for m in count_metrics:
        kpi_before_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.0)})
    for m in money_metrics:
        kpi_before_rows.append({'metric': f'{m}_exact_match_rate_pct_tol_0_01', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.01)})
    kpi_before_df = pd.DataFrame(kpi_before_rows)

def _kpi_val(df, metric_name):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return None
    sel = df.loc[df['metric'] == metric_name, 'value']
    if sel.empty:
        return None
    try:
        return float(sel.iloc[0])
    except Exception:
        return None

rows_compared_before = int(_kpi_val(kpi_before_df, 'rows_compared') or 0)
rows_compared_no_fallback = int(_kpi_val(kpi_compare_no_fallback_df, 'rows_compared') or 0)

cmp_source_before_df = cmp[['inn_key', 'agr_id_key']].merge(
    source_map_df[['inn_key', 'agr_id_key', 'agr_id_source']],
    on=['inn_key', 'agr_id_key'],
    how='left'
)
cmp_source_before_df['agr_id_source'] = cmp_source_before_df['agr_id_source'].fillna('unknown')
rows_removed_as_fallback = int((cmp_source_before_df['agr_id_source'] == 'r2_fallback').sum())

summary_no_fallback_df = pd.DataFrame([
    {'metric': 'rows_compared_before', 'value': rows_compared_before},
    {'metric': 'rows_compared_no_fallback', 'value': rows_compared_no_fallback},
    {'metric': 'rows_removed_as_fallback_in_cmp', 'value': rows_removed_as_fallback},
    {'metric': 'excel_only_keys_cnt_no_fallback', 'value': excel_only_keys_cnt},
    {'metric': 'excel_only_agr_id_cnt_no_fallback', 'value': excel_only_agr_id_cnt},
])

match_rate_metric_names = (
    [f'{m}_exact_match_rate_pct_tol_0' for m in count_metrics] +
    [f'{m}_exact_match_rate_pct_tol_0_01' for m in money_metrics]
)

match_rate_delta_rows = []
for metric_name in match_rate_metric_names:
    before_val = _kpi_val(kpi_before_df, metric_name)
    after_val = _kpi_val(kpi_compare_no_fallback_df, metric_name)
    match_rate_delta_rows.append({
        'metric': metric_name,
        'before_pct': before_val,
        'no_fallback_pct': after_val,
        'delta_pct_points': (after_val - before_val) if (before_val is not None and after_val is not None) else None,
    })

match_rate_delta_df = pd.DataFrame(match_rate_delta_rows)

print('Сводка влияния исключения fallback:')
display(summary_no_fallback_df)

print('\nKPI no_fallback (как в Секции 2):')
display(kpi_compare_no_fallback_df)

print('\nИзменение процента совпадений: до vs no_fallback')
display(match_rate_delta_df)

print('\nTOP-50 ключей, которые есть только в Excel (после исключения fallback):')
display(excel_only_no_fallback_df.head(50))

if cmp_no_fallback is None or len(cmp_no_fallback) == 0:
    print('Внимание: после исключения fallback compare-пересечение пустое.')
else:
    print(f'cmp_no_fallback rows: {len(cmp_no_fallback):,}')

### 3.10 Кейс `agr_id=493131251933`: пошаговая воронка `retl_cnt/term_cnt` + все исходные записи

Секция выгружает все записи из таблиц, участвующих в формировании `retl_cnt/term_cnt`, и показывает расчет по этапам:
- `ods_alpha.scd1_agreements`
- `ods_alpha.scd1_agr_terms`
- `ods_alpha.scd1_companies`
- `ods_alpha.scd1_merchants`
- `ods_alpha.scd1_pos_terminals`

Далее отдельно строится воронка в логике `04_operational_metrics` и сверка с `final_df`/`cmp`.

In [ ]:
# 3.10 Deep-dive: agr_id=493131251933 (all source records + retl/term funnel)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')

case_agr_id = '493131251933'
case_month_start = pd.to_datetime(month_start)
case_month_end = pd.to_datetime(month_end)


def _clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in(values):
    vals = _clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


# 1) agreements: все строки по agr_id
sql_agreements_case = f"""
select *
from ods_alpha.scd1_agreements a
where cast(a.abs_agr_id as string) = '{case_agr_id}'
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    agreements_case_df = imp.fetch(sql_agreements_case)

if agreements_case_df is None:
    agreements_case_df = pd.DataFrame()

n_agr_values = _clean_keys(agreements_case_df['n_agr'].tolist()) if len(agreements_case_df) and 'n_agr' in agreements_case_df.columns else []
n_cmp_values = _clean_keys(agreements_case_df['n_cmp_client'].tolist()) if len(agreements_case_df) and 'n_cmp_client' in agreements_case_df.columns else []

n_agr_in = _sql_in(n_agr_values)
n_cmp_in = _sql_in(n_cmp_values)

# 2) agr_terms: все строки по найденным n_agr
if n_agr_values:
    sql_terms_case = f"""
    select *
    from ods_alpha.scd1_agr_terms t
    where cast(t.n_agr as string) in ({n_agr_in})
    order by cast(t.n_agr as string), cast(t.d_valid_from as date)
    """
else:
    sql_terms_case = "select * from ods_alpha.scd1_agr_terms where 1=0"

# 3) companies: все строки по n_cmp_client
if n_cmp_values:
    sql_companies_case = f"""
    select *
    from ods_alpha.scd1_companies c
    where cast(c.n_cmp as string) in ({n_cmp_in})
    order by cast(c.n_cmp as string)
    """
else:
    sql_companies_case = "select * from ods_alpha.scd1_companies where 1=0"

# 4) merchants: все строки по n_cmp_client
if n_cmp_values:
    sql_merchants_case = f"""
    select *
    from ods_alpha.scd1_merchants m
    where cast(m.n_cmp as string) in ({n_cmp_in})
    order by cast(m.n_cmp as string), cast(m.c_nmrc as string)
    """
else:
    sql_merchants_case = "select * from ods_alpha.scd1_merchants where 1=0"

with imp:
    imp.execute('set MEM_LIMIT=12g')
    agr_terms_case_df = imp.fetch(sql_terms_case)
    companies_case_df = imp.fetch(sql_companies_case)
    merchants_case_df = imp.fetch(sql_merchants_case)

if agr_terms_case_df is None:
    agr_terms_case_df = pd.DataFrame()
if companies_case_df is None:
    companies_case_df = pd.DataFrame()
if merchants_case_df is None:
    merchants_case_df = pd.DataFrame()

# 5) pos_terminals: все строки по c_nmrc из merchants
nmrc_values = []
if len(merchants_case_df) and 'c_nmrc' in merchants_case_df.columns:
    nmrc_values = _clean_keys(merchants_case_df['c_nmrc'].tolist())

nmrc_in = _sql_in(nmrc_values)
if nmrc_values:
    sql_terminals_case = f"""
    select *
    from ods_alpha.scd1_pos_terminals t
    where cast(t.c_nmrc as string) in ({nmrc_in})
    order by cast(t.c_nmrc as string), cast(t.c_nter as string), cast(t.d_ter_install as date)
    """
else:
    sql_terminals_case = "select * from ods_alpha.scd1_pos_terminals where 1=0"

with imp:
    imp.execute('set MEM_LIMIT=12g')
    terminals_case_df = imp.fetch(sql_terminals_case)

if terminals_case_df is None:
    terminals_case_df = pd.DataFrame()

# 6) SA-периметр по правилам секции 01 (для контроля)
sql_sa_case = f"""
select distinct
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.abs_agr_id as string) = '{case_agr_id}'
  and upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
order by cast(a.d_valid_from as date), cast(a.n_agr as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_case_df = imp.fetch(sql_sa_case)

if sa_case_df is None:
    sa_case_df = pd.DataFrame()

# 7) Пошаговая воронка в логике 04_operational_metrics
if n_agr_values:
    sql_funnel_case = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_pre as (
      select
        m.n_agr,
        m.n_cmp_client,
        m.c_nmrc,
        cast(t.c_nter as string) as c_nter,
        cast(t.d_ter_install as date) as d_ter_install,
        cast(t.d_ter_close as date) as d_ter_close,
        coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    ),
    term_active as (
      select *
      from term_pre
      where coalesce(d_ter_install, cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(d_ter_close, cast('2999-12-31' as date)) >= cast('{month_start}' as date)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    stage_counts as (
      select n_agr, 'm_distinct_nmrc' as stage, count(distinct c_nmrc) as value
      from m
      group by n_agr
      union all
      select n_agr, 'term_pre_distinct_nter' as stage, count(distinct c_nter) as value
      from term_pre
      group by n_agr
      union all
      select n_agr, 'term_active_distinct_nter' as stage, count(distinct c_nter) as value
      from term_active
      group by n_agr
      union all
      select n_agr, 'retl_cnt_final' as stage, count(distinct c_nmrc) as value
      from term_active
      group by n_agr
      union all
      select n_agr, 'term_cnt_final' as stage, count(distinct c_nter) as value
      from term_active
      group by n_agr
    )
    select cast(null as string) as row_type,
           cast(null as string) as n_agr,
           cast(null as string) as n_cmp_client,
           cast(null as string) as c_nmrc,
           cast(null as string) as c_nter,
           cast(null as date) as d_ter_install,
           cast(null as date) as d_ter_close,
           cast(null as string) as t_deleted_flg,
           cast(null as string) as stage,
           cast(null as bigint) as stage_value,
           cast(null as bigint) as retl_cnt,
           cast(null as bigint) as term_cnt
    where 1=0
    """

    sql_term_pre_case = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    )
    select
      m.n_agr,
      m.n_cmp_client,
      m.c_nmrc,
      cast(t.c_nter as string) as c_nter,
      cast(t.d_ter_install as date) as d_ter_install,
      cast(t.d_ter_close as date) as d_ter_close,
      coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
    from m
    join ods_alpha.scd1_pos_terminals t
      on cast(t.c_nmrc as string) = m.c_nmrc
    where t.c_nter is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
    order by m.n_agr, m.c_nmrc, cast(t.c_nter as string)
    """

    sql_term_active_case = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    )
    select
      m.n_agr,
      m.n_cmp_client,
      m.c_nmrc,
      cast(t.c_nter as string) as c_nter,
      cast(t.d_ter_install as date) as d_ter_install,
      cast(t.d_ter_close as date) as d_ter_close,
      coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
    from m
    join ods_alpha.scd1_pos_terminals t
      on cast(t.c_nmrc as string) = m.c_nmrc
    where t.c_nter is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
      and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
      and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
    order by m.n_agr, m.c_nmrc, cast(t.c_nter as string)
    """

    sql_funnel_counts_case = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_pre as (
      select m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string) as c_nter,
             cast(t.d_ter_install as date) as d_ter_install, cast(t.d_ter_close as date) as d_ter_close,
             coalesce(t.ods_deleted_flg, '0') as t_deleted_flg
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    ),
    term_active as (
      select *
      from term_pre
      where coalesce(d_ter_install, cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(d_ter_close, cast('2999-12-31' as date)) >= cast('{month_start}' as date)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    m_counts as (
      select n_agr, count(distinct c_nmrc) as m_distinct_nmrc
      from m
      group by n_agr
    ),
    term_pre_counts as (
      select n_agr, count(distinct c_nter) as term_pre_distinct_nter
      from term_pre
      group by n_agr
    ),
    term_active_counts as (
      select n_agr, count(distinct c_nter) as term_active_distinct_nter
      from term_active
      group by n_agr
    )
    select
      sa.n_agr,
      sa.n_cmp_client,
      r.retl_cnt,
      t.term_cnt,
      mc.m_distinct_nmrc,
      pc.term_pre_distinct_nter,
      ac.term_active_distinct_nter
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    left join m_counts mc on mc.n_agr = sa.n_agr
    left join term_pre_counts pc on pc.n_agr = sa.n_agr
    left join term_active_counts ac on ac.n_agr = sa.n_agr
    order by sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        term_pre_case_df = imp.fetch(sql_term_pre_case)
        term_active_case_df = imp.fetch(sql_term_active_case)
        funnel_counts_case_df = imp.fetch(sql_funnel_counts_case)

    if term_pre_case_df is None:
        term_pre_case_df = pd.DataFrame()
    if term_active_case_df is None:
        term_active_case_df = pd.DataFrame()
    if funnel_counts_case_df is None:
        funnel_counts_case_df = pd.DataFrame()
else:
    term_pre_case_df = pd.DataFrame()
    term_active_case_df = pd.DataFrame()
    funnel_counts_case_df = pd.DataFrame()

# 8) Сверка с final_df и cmp
case_final_df = pd.DataFrame()
if 'final_df' in globals() and isinstance(final_df, pd.DataFrame) and len(final_df):
    ff = final_df.copy()
    ff['agr_id_key'] = ff['agr_id'].apply(normalize_agr_q1)
    case_final_df = ff[ff['agr_id_key'] == case_agr_id].copy()

case_cmp_df = pd.DataFrame()
if 'cmp' in globals() and isinstance(cmp, pd.DataFrame) and len(cmp):
    case_cmp_df = cmp[cmp['agr_id_key'] == case_agr_id].copy()

# 9) Вывод
print(f'Кейс: agr_id={case_agr_id}')
print(f'Период среза: {month_start} .. {month_end}')
print(f'Найденные n_agr: {n_agr_values}')
print(f'Найденные n_cmp_client: {n_cmp_values}')

print('\n1) agreements (все записи):')
print(f'rows: {len(agreements_case_df):,}')
display(agreements_case_df)

print('\n2) agr_terms (все записи по n_agr):')
print(f'rows: {len(agr_terms_case_df):,}')
display(agr_terms_case_df)

print('\n3) companies (все записи по n_cmp_client):')
print(f'rows: {len(companies_case_df):,}')
display(companies_case_df)

print('\n4) merchants (все записи по n_cmp_client):')
print(f'rows: {len(merchants_case_df):,}')
display(merchants_case_df)

print('\n5) pos_terminals (все записи по c_nmrc из merchants):')
print(f'rows: {len(terminals_case_df):,}')
display(terminals_case_df)

print('\n6) SA perimeter check (по правилам секции 01):')
print(f'rows: {len(sa_case_df):,}')
display(sa_case_df)

print('\n7) Воронка retl/term в логике 04_operational_metrics:')
print('7.1 term_pre (после c_nter not null + terminal not deleted):')
print(f'rows: {len(term_pre_case_df):,}')
display(term_pre_case_df)

print('7.2 term_active (добавлен фильтр активности по датам месяца):')
print(f'rows: {len(term_active_case_df):,}')
display(term_active_case_df)

print('7.3 Итоговые счетчики по n_agr:')
display(funnel_counts_case_df)

if len(term_pre_case_df):
    # Причины исключения между term_pre и term_active
    tp = term_pre_case_df.copy()
    tp['d_ter_install'] = pd.to_datetime(tp['d_ter_install'], errors='coerce')
    tp['d_ter_close'] = pd.to_datetime(tp['d_ter_close'], errors='coerce')
    tp['is_install_ok'] = tp['d_ter_install'].isna() | (tp['d_ter_install'] <= case_month_end)
    tp['is_close_ok'] = tp['d_ter_close'].isna() | (tp['d_ter_close'] >= case_month_start)
    tp['is_active_month'] = tp['is_install_ok'] & tp['is_close_ok']

    def _drop_reason(r):
        if r['is_active_month']:
            return 'active_in_month'
        if not r['is_install_ok']:
            return 'install_after_month_end'
        if not r['is_close_ok']:
            return 'closed_before_month_start'
        return 'other'

    tp['drop_reason'] = tp.apply(_drop_reason, axis=1)
    term_reason_case_df = (
        tp.groupby(['n_agr', 'drop_reason'], as_index=False)
          .agg(rows=('c_nter', 'size'), c_nmrc_nunique=('c_nmrc', 'nunique'), c_nter_nunique=('c_nter', 'nunique'))
          .sort_values(['n_agr', 'rows'], ascending=[True, False])
    )

    print('7.4 Разбор причин выпадения между term_pre и term_active:')
    display(term_reason_case_df)

print('\n8) Сверка с final_df по agr_id:')
if case_final_df.empty:
    print('По agr_id строк в final_df не найдено.')
else:
    show_cols = ['report_month', 'inn', 'agr_id', 'n_agr', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'company_name', 'agr_id_source']
    show_cols = [c for c in show_cols if c in case_final_df.columns]
    display(case_final_df[show_cols])

print('\n9) Сверка с cmp (Lake vs Excel) по agr_id:')
if case_cmp_df.empty:
    print('По agr_id строк в cmp не найдено.')
else:
    show_cmp_cols = [
        'agr_id_key', 'inn_key',
        'retl_cnt_lake', 'retl_cnt_excel', 'retl_cnt_delta',
        'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta',
        'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta',
        'trx_sum_lake', 'trx_sum_excel', 'trx_sum_delta',
        'commission_total_lake', 'commission_total_excel', 'commission_total_delta',
        'int_component_lake', 'int_component_excel', 'int_component_delta',
    ]
    show_cmp_cols = [c for c in show_cmp_cols if c in case_cmp_df.columns]
    display(case_cmp_df[show_cmp_cols])

    requested_metric_labels = [
        ('trx_cnt', 'Количество транзакций'),
        ('trx_sum', 'Сумма транзакций'),
        ('commission_total', 'Комиссия эквайринга'),
        ('int_component', 'Комиссия int'),
    ]
    metric_diff_rows = []
    for metric_name, metric_label in requested_metric_labels:
        lake_col = f'{metric_name}_lake'
        excel_col = f'{metric_name}_excel'
        if lake_col in case_cmp_df.columns and excel_col in case_cmp_df.columns:
            lake_val = pd.to_numeric(case_cmp_df[lake_col], errors='coerce').fillna(0).sum()
            excel_val = pd.to_numeric(case_cmp_df[excel_col], errors='coerce').fillna(0).sum()
            delta_val = lake_val - excel_val
            metric_diff_rows.append({
                'metric': metric_label,
                'lake_value': lake_val,
                'excel_value': excel_val,
                'delta_lake_minus_excel': delta_val,
                'abs_delta': abs(delta_val),
            })

    if metric_diff_rows:
        case_cmp_tx_comm_diff_df = pd.DataFrame(metric_diff_rows)
        print('\n9.1 Расхождения по транзакциям и комиссиям (Lake vs Excel):')
        display(case_cmp_tx_comm_diff_df)
    else:
        print('\n9.1 Не найдены нужные колонки в cmp для расчета расхождений по trx/commission.')

print('\n10) Терминалы с транзакциями в периоде (периметр Section 05):')
if not n_agr_values:
    print('Для agr_id не найден n_agr, проверить транзакционные терминалы невозможно.')
    case_trx_detail_df = pd.DataFrame()
    terminals_with_trx_df = pd.DataFrame()
    trx_vs_active_term_df = pd.DataFrame()
else:
    sql_case_trx_terminals = f"""
    with trx_base_raw as (
      select
        cast(t.n_trx as string) as n_trx,
        cast(t.c_nter as string) as c_nter,
        cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({n_agr_in})
    ),
    ta as (
      select n_trx, n_agr, max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select cast(i.n_trx as string) as n_trx, sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    )
    select
      ta.n_agr,
      ta.n_trx,
      tb.c_nter,
      tb.n_amt_src,
      ta.n_amt_tax,
      coalesce(i.n_amt_fee, 0.0) as n_amt_fee
    from ta
    join trx_base tb on tb.n_trx = ta.n_trx
    left join trx_int_agg i on i.n_trx = ta.n_trx
    order by ta.n_agr, tb.c_nter, ta.n_trx
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        case_trx_detail_df = imp.fetch(sql_case_trx_terminals)

    if case_trx_detail_df is None:
        case_trx_detail_df = pd.DataFrame(columns=['n_agr', 'n_trx', 'c_nter', 'n_amt_src', 'n_amt_tax', 'n_amt_fee'])

    if not case_trx_detail_df.empty:
        for col in ['n_amt_src', 'n_amt_tax', 'n_amt_fee']:
            case_trx_detail_df[col] = pd.to_numeric(case_trx_detail_df[col], errors='coerce')

        terminals_with_trx_df = (
            case_trx_detail_df.groupby('c_nter', as_index=False)
            .agg(
                trx_cnt=('n_trx', 'nunique'),
                trx_sum=('n_amt_src', 'sum'),
                commission_from_ops=('n_amt_tax', 'sum'),
                int_component=('n_amt_fee', 'sum'),
                n_agr_nunique=('n_agr', 'nunique')
            )
            .sort_values(['trx_cnt', 'trx_sum'], ascending=[False, False])
            .reset_index(drop=True)
        )
    else:
        terminals_with_trx_df = pd.DataFrame(columns=['c_nter', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_agr_nunique'])

    distinct_terminals_with_trx = int(terminals_with_trx_df['c_nter'].nunique()) if len(terminals_with_trx_df) else 0
    total_trx_cnt = int(case_trx_detail_df['n_trx'].nunique()) if len(case_trx_detail_df) else 0

    print(f'distinct_terminals_with_trx: {distinct_terminals_with_trx}')
    print(f'total_trx_cnt: {total_trx_cnt}')

    print('\n10.1 Агрегат по терминалам с транзакциями:')
    display(terminals_with_trx_df)

    print('10.2 Детальные строки транзакций по терминалам:')
    display(case_trx_detail_df)

    active_terms = set()
    if 'term_active_case_df' in globals() and isinstance(term_active_case_df, pd.DataFrame) and len(term_active_case_df) and 'c_nter' in term_active_case_df.columns:
        active_terms = {
            str(x).strip() for x in term_active_case_df['c_nter'].dropna().tolist()
            if str(x).strip() not in ['', 'None', 'nan', 'NaN']
        }

    trx_terms = {
        str(x).strip() for x in terminals_with_trx_df.get('c_nter', pd.Series(dtype=object)).dropna().tolist()
        if str(x).strip() not in ['', 'None', 'nan', 'NaN']
    }

    all_terms = sorted(active_terms | trx_terms)
    if all_terms:
        rows = []
        for t in all_terms:
            in_active = t in active_terms
            in_trx = t in trx_terms
            if in_active and in_trx:
                relation = 'in_both'
            elif in_active:
                relation = 'only_active'
            else:
                relation = 'only_trx'
            rows.append({
                'c_nter': t,
                'in_term_active_case': in_active,
                'in_trx_case': in_trx,
                'relation': relation,
            })

        trx_vs_active_term_df = pd.DataFrame(rows)
        print('10.3 Сверка терминалов: active-периметр vs trx-периметр:')
        display(trx_vs_active_term_df)

        cross_summary_df = (
            trx_vs_active_term_df.groupby('relation', as_index=False)
            .agg(terminals_cnt=('c_nter', 'nunique'))
            .sort_values('terminals_cnt', ascending=False)
            .reset_index(drop=True)
        )
        print('10.4 Итого по типам попадания терминалов:')
        display(cross_summary_df)
    else:
        trx_vs_active_term_df = pd.DataFrame(columns=['c_nter', 'in_term_active_case', 'in_trx_case', 'relation'])
        print('10.3 Нет данных для сверки active/trx терминалов.')

### 3.11 Пилот: амортизация для 3 терминалов (КХД + модель + цена)

Пилотная секция (без изменения `final_df`):
- подключение к КХД через `KHD.env`;
- получение модели терминала из `bmrt.bm_det_device`;
- загрузка стоимости модели из Excel;
- расчет амортизации `price/48` и суммы за отчетный месяц для 3 автоматически выбранных терминалов с полными данными.

In [ ]:
# 3.11 Pilot amortization for 3 terminals (CDWH + model + price)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')

from rail_connectors.connection import connect as rail_connect

term_model_excel_path = '/home/jovyan/documents/Equaring/Data/term_model.xlsx'
pilot_limit_candidates = 1200
pilot_take_n = 3


def _clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in(values):
    vals = _clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def _norm_model(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\s+', ' ', str(v).strip().lower())
    return s or None


# 1) CDWH connection (must be before bmrt.bm_det_device query)
khd_user = 'DS_LII'
khd_password = 'dl3S$wolx5dz'

cdwh_connection = rail_connect(
    to='CDWH',
    user_params={
        'user_name': khd_user,
        'password': khd_password,
    }
)
cdwh_connection._init_connection()
print('CDWH connection initialized')

# 2) Candidate terminals in current month perimeter
sql_pilot_terminals = f"""
select
  cast(t.c_nter as string) as c_nter,
  cast(t.c_pos_serial as string) as c_pos_serial,
  cast(t.d_ter_delivery as date) as d_ter_delivery,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close
from ods_alpha.scd1_pos_terminals t
where t.c_nter is not null
  and coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_pos_serial is not null
  and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
  and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
limit {pilot_limit_candidates}
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    pilot_terminals_df = imp.fetch(sql_pilot_terminals)

if pilot_terminals_df is None:
    pilot_terminals_df = pd.DataFrame(columns=['c_nter', 'c_pos_serial', 'd_ter_delivery', 'd_ter_install', 'd_ter_close'])

for col in ['c_nter', 'c_pos_serial']:
    if col in pilot_terminals_df.columns:
        pilot_terminals_df[col] = pilot_terminals_df[col].astype(str).str.strip()

# 3) first_d_ter_deliver by serial (strictly by d_ter_delivery)
serial_values = _clean_keys(pilot_terminals_df['c_pos_serial'].tolist()) if len(pilot_terminals_df) else []
if serial_values:
    serial_in = _sql_in(serial_values)
    sql_first_deliver = f"""
    select
      cast(t.c_pos_serial as string) as c_pos_serial,
      min(cast(t.d_ter_delivery as date)) as first_d_ter_deliver
    from ods_alpha.scd1_pos_terminals t
    where cast(t.c_pos_serial as string) in ({serial_in})
      and t.d_ter_delivery is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
    group by cast(t.c_pos_serial as string)
    """
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        serial_first_deliver_df = imp.fetch(sql_first_deliver)
else:
    serial_first_deliver_df = pd.DataFrame(columns=['c_pos_serial', 'first_d_ter_deliver'])

if serial_first_deliver_df is None:
    serial_first_deliver_df = pd.DataFrame(columns=['c_pos_serial', 'first_d_ter_deliver'])
if len(serial_first_deliver_df):
    serial_first_deliver_df['c_pos_serial'] = serial_first_deliver_df['c_pos_serial'].astype(str).str.strip()

# 4) Model from CDWH BMRT.BM_DET_DEVICE (join by c_nter: ID_DEVICE -> CODE_DEVICE)
nter_values = _clean_keys(pilot_terminals_df['c_nter'].tolist()) if len(pilot_terminals_df) else []

cdwh_id_batches = 0
cdwh_code_batches = 0
cdwh_code_fallback_batches = 0


def _iter_chunks(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def _split_nter_for_id_and_code(values):
    id_numeric = []
    code_values = []
    for v in values:
        s = str(v).strip()
        if not s:
            continue
        code_values.append(s)
        if re.fullmatch(r'\d+', s):
            id_numeric.append(str(int(s)))
    return sorted(set(id_numeric)), sorted(set(code_values))


if nter_values:
    id_values_numeric, code_values = _split_nter_for_id_and_code(nter_values)
    bm_chunks = []

    # Oracle IN-list safeguard (<1000)
    id_chunk_size = 950
    code_chunk_size = 900

    with cdwh_connection:
        if id_values_numeric:
            for id_chunk in _iter_chunks(id_values_numeric, id_chunk_size):
                id_in = ', '.join(id_chunk)

                sql_bm_det_device_by_id = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.ID_DEVICE in ({id_in})
                """

                part_df = cdwh_connection.fetch(sql_bm_det_device_by_id)
                cdwh_id_batches += 1
                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

        if code_values:
            for code_chunk in _iter_chunks(code_values, code_chunk_size):
                code_in = _sql_in(code_chunk)

                sql_bm_det_device_by_code_fast = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.CODE_DEVICE in ({code_in})
                """

                try:
                    part_df = cdwh_connection.fetch(sql_bm_det_device_by_code_fast)
                    cdwh_code_batches += 1
                except Exception as exc:
                    sql_bm_det_device_by_code_fallback = f"""
                    select distinct
                      trim(to_char(d.ID_DEVICE)) as device,
                      trim(to_char(d.CODE_DEVICE)) as code_device,
                      trim(to_char(d.MODEL_DEVICE)) as model_device
                    from BMRT.BM_DET_DEVICE d
                    where trim(to_char(d.CODE_DEVICE)) in ({code_in})
                    """
                    part_df = cdwh_connection.fetch(sql_bm_det_device_by_code_fallback)
                    cdwh_code_fallback_batches += 1
                    print(f'CODE_DEVICE fallback chunk due to: {type(exc).__name__}')

                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

    if bm_chunks:
        bm_det_device_df = pd.concat(bm_chunks, ignore_index=True)
    else:
        bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])
else:
    bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])

if bm_det_device_df is None:
    bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])
if len(bm_det_device_df):
    bm_det_device_df.columns = [str(c).strip().lower() for c in bm_det_device_df.columns]
    for col in ['device', 'code_device', 'model_device']:
        if col in bm_det_device_df.columns:
            bm_det_device_df[col] = bm_det_device_df[col].astype(str).str.strip()
    bm_det_device_df = bm_det_device_df.drop_duplicates(subset=['device', 'code_device', 'model_device'])

print('bm_det_device columns fixed: ID_DEVICE, CODE_DEVICE, MODEL_DEVICE')
print(f'CDWH batch stats -> by_id: {cdwh_id_batches}, by_code_fast: {cdwh_code_batches}, by_code_fallback: {cdwh_code_fallback_batches}')

# 5) Price map from Excel
price_src_df = pd.read_excel(term_model_excel_path)
required_price_cols = ['model_name', 'price']
missing_price_cols = [c for c in required_price_cols if c not in price_src_df.columns]
if missing_price_cols:
    raise RuntimeError(f'В term_model.xlsx отсутствуют колонки: {missing_price_cols}')

price_map_df = price_src_df[['model_name', 'price']].copy()
price_map_df['model_key'] = price_map_df['model_name'].apply(_norm_model)
price_map_df['price'] = pd.to_numeric(price_map_df['price'], errors='coerce')
price_map_df = (
    price_map_df.dropna(subset=['model_key', 'price'])
    .groupby('model_key', as_index=False)
    .agg(price=('price', 'max'))
)

# 6) Build pipeline terminal -> model -> price
pilot_pipe_df = pilot_terminals_df.copy()
if len(pilot_pipe_df):
    pilot_pipe_df = pilot_pipe_df.merge(serial_first_deliver_df, on='c_pos_serial', how='left')

    device_map = {}
    code_map = {}
    if len(bm_det_device_df):
        device_map = (
            bm_det_device_df.loc[
                bm_det_device_df['device'].notna() & (bm_det_device_df['device'].astype(str).str.strip() != ''),
                ['device', 'model_device']
            ]
            .drop_duplicates(subset=['device'], keep='first')
            .set_index('device')['model_device']
            .to_dict()
        )
        code_map = (
            bm_det_device_df.loc[
                bm_det_device_df['code_device'].notna() & (bm_det_device_df['code_device'].astype(str).str.strip() != ''),
                ['code_device', 'model_device']
            ]
            .drop_duplicates(subset=['code_device'], keep='first')
            .set_index('code_device')['model_device']
            .to_dict()
        )

    pilot_pipe_df['model_device'] = pilot_pipe_df['c_nter'].map(device_map)
    pilot_pipe_df['join_key_used'] = pilot_pipe_df['model_device'].apply(
        lambda x: 'device' if pd.notna(x) and str(x).strip() != '' else None
    )

    need_code_mask = pilot_pipe_df['model_device'].isna() | (pilot_pipe_df['model_device'].astype(str).str.strip() == '')
    pilot_pipe_df.loc[need_code_mask, 'model_device'] = pilot_pipe_df.loc[need_code_mask, 'c_nter'].map(code_map)
    pilot_pipe_df.loc[
        need_code_mask
        & pilot_pipe_df['model_device'].notna()
        & (pilot_pipe_df['model_device'].astype(str).str.strip() != ''),
        'join_key_used'
    ] = 'code_device'

    pilot_pipe_df['model_key'] = pilot_pipe_df['model_device'].apply(_norm_model)
    pilot_pipe_df = pilot_pipe_df.merge(price_map_df, on='model_key', how='left')

    pilot_pipe_df['first_d_ter_deliver'] = pd.to_datetime(pilot_pipe_df['first_d_ter_deliver'], errors='coerce')

    pilot_pipe_df['missing_serial'] = pilot_pipe_df['c_pos_serial'].isna() | (pilot_pipe_df['c_pos_serial'].astype(str).str.strip() == '')
    pilot_pipe_df['missing_deliver'] = pilot_pipe_df['first_d_ter_deliver'].isna()
    pilot_pipe_df['missing_model'] = pilot_pipe_df['model_device'].isna() | (pilot_pipe_df['model_device'].astype(str).str.strip() == '')
    pilot_pipe_df['missing_price'] = pd.to_numeric(pilot_pipe_df['price'], errors='coerce').isna()

    def _qc_status(row):
        if row['missing_serial']:
            return 'missing_serial'
        if row['missing_deliver']:
            return 'missing_deliver'
        if row['missing_model']:
            return 'missing_model'
        if row['missing_price']:
            return 'missing_price'
        return 'complete'

    pilot_pipe_df['qc_status'] = pilot_pipe_df.apply(_qc_status, axis=1)

    report_month_start_ts = pd.to_datetime(month_start)
    report_month_first_day_ts = pd.Timestamp(year=report_month_start_ts.year, month=report_month_start_ts.month, day=1)

    pilot_complete_df = pilot_pipe_df[pilot_pipe_df['qc_status'] == 'complete'].copy()
    if len(pilot_complete_df):
        first_month_first_day = pilot_complete_df['first_d_ter_deliver'].dt.to_period('M').dt.to_timestamp()
        months_from_start = (
            (report_month_first_day_ts.year - first_month_first_day.dt.year) * 12
            + (report_month_first_day_ts.month - first_month_first_day.dt.month)
        )

        pilot_complete_df['months_from_start'] = months_from_start
        pilot_complete_df['is_in_48m_window'] = (
            pilot_complete_df['months_from_start'].notna()
            & (pilot_complete_df['months_from_start'] >= 0)
            & (pilot_complete_df['months_from_start'] < 48)
        )
        pilot_complete_df['amort_monthly'] = pd.to_numeric(pilot_complete_df['price'], errors='coerce') / 48.0
        pilot_complete_df['amort_for_report_month'] = pilot_complete_df['amort_monthly'].where(pilot_complete_df['is_in_48m_window'], 0.0)
        pilot_complete_df['report_month'] = report_month_first_day_ts.strftime('%Y-%m')

        pilot_amort_3term_df = (
            pilot_complete_df.sort_values(['first_d_ter_deliver', 'c_nter'], ascending=[True, True])
            .head(pilot_take_n)
            .copy()
        )
    else:
        pilot_complete_df = pd.DataFrame()
        pilot_amort_3term_df = pd.DataFrame()

    qc_missing_df = (
        pilot_pipe_df.groupby('qc_status', as_index=False)
        .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
        .sort_values(['rows', 'terminals_nunique'], ascending=False)
        .reset_index(drop=True)
    )

    join_cov_df = pilot_pipe_df.copy()
    join_cov_df['join_key_used'] = join_cov_df['join_key_used'].fillna('no_match')
    join_coverage_df = (
        join_cov_df.groupby('join_key_used', as_index=False)
        .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
        .sort_values(['rows', 'terminals_nunique'], ascending=False)
        .reset_index(drop=True)
    )
else:
    pilot_pipe_df = pd.DataFrame()
    pilot_complete_df = pd.DataFrame()
    pilot_amort_3term_df = pd.DataFrame()
    qc_missing_df = pd.DataFrame(columns=['qc_status', 'rows', 'terminals_nunique'])
    join_coverage_df = pd.DataFrame(columns=['join_key_used', 'rows', 'terminals_nunique'])

print(f'Pilot candidates from pos_terminals: {len(pilot_terminals_df):,}')
print(f'CDWH device rows fetched: {len(bm_det_device_df):,}')
print(f'Price dictionary rows: {len(price_map_df):,}')
print(f'Complete candidates: {len(pilot_complete_df):,}')
print(f'Selected terminals for pilot: {len(pilot_amort_3term_df):,} (target={pilot_take_n})')

print('\nQC: missing reasons')
display(qc_missing_df)

print('QC: join coverage (device vs code_device)')
display(join_coverage_df)

if pilot_amort_3term_df.empty:
    print('Не удалось выбрать 3 терминала с полными данными (deliver+model+price).')
else:
    pilot_out_cols = [
        'c_nter', 'c_pos_serial', 'model_device', 'price',
        'first_d_ter_deliver', 'amort_monthly',
        'report_month', 'is_in_48m_window', 'amort_for_report_month', 'join_key_used'
    ]
    pilot_out_cols = [c for c in pilot_out_cols if c in pilot_amort_3term_df.columns]
    print('\nPilot amortization result (3 terminals):')
    display(pilot_amort_3term_df[pilot_out_cols])

print('\nSample of full pilot pipeline (TOP-30):')
display(pilot_pipe_df.head(30))

### 3.12 A/B: `retl_cnt` / `term_cnt` через `AGR_TERMS` vs текущая логика

Секция сравнивает:
- **A (old):** текущий расчет через `n_cmp_client -> merchants -> pos_terminals`;
- **B (AGR_TERMS):** `n_agr -> agr_terms.c_nmrc -> pos_terminals`.

Выводит:
- A/B-таблицу по `n_agr`;
- KPI совпадения с Excel (до/после);
- QC по отсутствующим связкам в `AGR_TERMS`;
- отдельное подтверждение по кейсу `agr_id=493131251933`.

In [ ]:
# 3.12 A/B check: current retl/term vs AGR_TERMS retl/term
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')
if 'sa_df' not in globals() or sa_df is None or len(sa_df) == 0:
    raise RuntimeError('Сначала выполните 01_sa_perimeter (нужен sa_df).')
if 'cmp_df' not in globals() or cmp_df is None or len(cmp_df) == 0:
    raise RuntimeError('Сначала выполните 04_operational_metrics (нужен cmp_df).')
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните Секцию 2 compare (нужен cmp).')


def _clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in(values):
    vals = _clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


n_agr_values_ab = _clean_keys(sa_df.get('n_agr', pd.Series(dtype=object)).tolist())

if not n_agr_values_ab:
    print('В sa_df нет n_agr для A/B проверки.')
    ab_counts_df = pd.DataFrame()
    ab_compare_df = pd.DataFrame()
    cmp_ab_retl_term_df = pd.DataFrame()
    kpi_retl_term_ab_df = pd.DataFrame()
    qc_no_agrterms_nmrc_df = pd.DataFrame()
    qc_fanout_reduction_df = pd.DataFrame()
    case493_terms_ab_df = pd.DataFrame()
    case493_terminals_ab_df = pd.DataFrame()
    case493_proof_df = pd.DataFrame()
else:
    n_agr_in_ab = _sql_in(n_agr_values_ab)

    sql_ab_counts = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in_ab})
    ),
    agr_terms_link as (
      select
        sa.n_agr,
        sa.agr_id,
        sa.n_cmp_client,
        cast(t.c_nmrc as string) as c_nmrc
      from sa_agr sa
      left join ods_alpha.scd1_agr_terms t
        on cast(t.n_agr as string) = sa.n_agr
       and t.c_nmrc is not null
       and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
       and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
       and upper(trim(cast(t.cf_ter_type as string))) = 'P'
       and coalesce(t.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.agr_id, sa.n_cmp_client, cast(t.c_nmrc as string)
    ),
    agr_terms_nonnull as (
      select *
      from agr_terms_link
      where c_nmrc is not null
    ),
    term_active_ab as (
      select
        l.n_agr,
        l.agr_id,
        l.n_cmp_client,
        l.c_nmrc,
        cast(p.c_nter as string) as c_nter
      from agr_terms_nonnull l
      join ods_alpha.scd1_pos_terminals p
        on cast(p.c_nmrc as string) = l.c_nmrc
      where p.c_nter is not null
        and coalesce(p.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(p.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by l.n_agr, l.agr_id, l.n_cmp_client, l.c_nmrc, cast(p.c_nter as string)
    ),
    terms_cnt as (
      select n_agr, count(distinct c_nmrc) as agr_terms_nmrc_cnt
      from agr_terms_nonnull
      group by n_agr
    ),
    ab_cnt as (
      select
        n_agr,
        count(distinct c_nmrc) as retl_cnt_ab,
        count(distinct c_nter) as term_cnt_ab
      from term_active_ab
      group by n_agr
    )
    select
      sa.n_agr,
      sa.agr_id,
      sa.n_cmp_client,
      tc.agr_terms_nmrc_cnt,
      ab.retl_cnt_ab,
      ab.term_cnt_ab
    from sa_agr sa
    left join terms_cnt tc on tc.n_agr = sa.n_agr
    left join ab_cnt ab on ab.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        ab_counts_df = imp.fetch(sql_ab_counts)

    if ab_counts_df is None:
        ab_counts_df = pd.DataFrame(columns=['n_agr', 'agr_id', 'n_cmp_client', 'agr_terms_nmrc_cnt', 'retl_cnt_ab', 'term_cnt_ab'])

    if len(ab_counts_df):
        for c in ['n_agr', 'agr_id', 'n_cmp_client']:
            if c in ab_counts_df.columns:
                ab_counts_df[c] = ab_counts_df[c].astype(str)
        for c in ['agr_terms_nmrc_cnt', 'retl_cnt_ab', 'term_cnt_ab']:
            if c in ab_counts_df.columns:
                ab_counts_df[c] = pd.to_numeric(ab_counts_df[c], errors='coerce')

    # A/B table by n_agr
    old_cmp_small_df = cmp_df[['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt']].copy()
    old_cmp_small_df['n_agr'] = old_cmp_small_df['n_agr'].astype(str)
    old_cmp_small_df['n_cmp_client'] = old_cmp_small_df['n_cmp_client'].astype(str)
    old_cmp_small_df['retl_cnt_old'] = pd.to_numeric(old_cmp_small_df['retl_cnt'], errors='coerce')
    old_cmp_small_df['term_cnt_old'] = pd.to_numeric(old_cmp_small_df['term_cnt'], errors='coerce')

    ab_compare_df = old_cmp_small_df.merge(
        ab_counts_df[['n_agr', 'agr_id', 'agr_terms_nmrc_cnt', 'retl_cnt_ab', 'term_cnt_ab']],
        on='n_agr',
        how='left'
    )

    ab_compare_df['retl_delta_ab_minus_old'] = pd.to_numeric(ab_compare_df['retl_cnt_ab'], errors='coerce').fillna(0) - pd.to_numeric(ab_compare_df['retl_cnt_old'], errors='coerce').fillna(0)
    ab_compare_df['term_delta_ab_minus_old'] = pd.to_numeric(ab_compare_df['term_cnt_ab'], errors='coerce').fillna(0) - pd.to_numeric(ab_compare_df['term_cnt_old'], errors='coerce').fillna(0)
    ab_compare_df['retl_reduction_old_minus_ab'] = pd.to_numeric(ab_compare_df['retl_cnt_old'], errors='coerce').fillna(0) - pd.to_numeric(ab_compare_df['retl_cnt_ab'], errors='coerce').fillna(0)
    ab_compare_df['term_reduction_old_minus_ab'] = pd.to_numeric(ab_compare_df['term_cnt_old'], errors='coerce').fillna(0) - pd.to_numeric(ab_compare_df['term_cnt_ab'], errors='coerce').fillna(0)

    print('A/B by n_agr (old vs AGR_TERMS):')
    display_cols_ab = [
        'agr_id', 'n_agr', 'n_cmp_client',
        'agr_terms_nmrc_cnt',
        'retl_cnt_old', 'retl_cnt_ab', 'retl_delta_ab_minus_old', 'retl_reduction_old_minus_ab',
        'term_cnt_old', 'term_cnt_ab', 'term_delta_ab_minus_old', 'term_reduction_old_minus_ab'
    ]
    display_cols_ab = [c for c in display_cols_ab if c in ab_compare_df.columns]
    display(ab_compare_df[display_cols_ab].sort_values(['retl_reduction_old_minus_ab', 'term_reduction_old_minus_ab'], ascending=False).head(200))

    # KPI before vs after against Excel for retl/term
    lk_ab = final_df.copy()
    lk_ab['inn_key'] = lk_ab['inn'].apply(normalize_inn_q1)
    lk_ab['agr_id_key'] = lk_ab['agr_id'].apply(normalize_agr_q1)
    lk_ab['n_agr'] = lk_ab['n_agr'].astype(str)

    lk_ab = lk_ab.merge(ab_counts_df[['n_agr', 'retl_cnt_ab', 'term_cnt_ab']], on='n_agr', how='left')
    lk_ab['retl_cnt_lake_ab'] = pd.to_numeric(lk_ab['retl_cnt_ab'], errors='coerce')
    lk_ab['term_cnt_lake_ab'] = pd.to_numeric(lk_ab['term_cnt_ab'], errors='coerce')

    lk_ab_agg = (
        lk_ab.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt_lake_ab': 'max',
            'term_cnt_lake_ab': 'max',
        })
    )

    if 'ex_agg' in globals() and isinstance(ex_agg, pd.DataFrame) and len(ex_agg):
        ex_retl_term_df = ex_agg[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']].copy()
    else:
        ex_retl_term_df = (
            cmp[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
            .drop_duplicates(subset=['inn_key', 'agr_id_key'])
            .copy()
        )

    cmp_ab_retl_term_df = lk_ab_agg.merge(ex_retl_term_df, on=['inn_key', 'agr_id_key'], how='inner')

    for m in ['retl_cnt', 'term_cnt']:
        cmp_ab_retl_term_df[f'{m}_delta_ab'] = pd.to_numeric(cmp_ab_retl_term_df[f'{m}_lake_ab'], errors='coerce').fillna(0) - pd.to_numeric(cmp_ab_retl_term_df[f'{m}_excel'], errors='coerce').fillna(0)
        cmp_ab_retl_term_df[f'{m}_abs_ab'] = pd.to_numeric(cmp_ab_retl_term_df[f'{m}_delta_ab'], errors='coerce').abs()

    # before (existing cmp)
    cmp_before_local = cmp.copy()
    if 'retl_cnt_abs' not in cmp_before_local.columns:
        cmp_before_local['retl_cnt_abs'] = (pd.to_numeric(cmp_before_local['retl_cnt_lake'], errors='coerce').fillna(0) - pd.to_numeric(cmp_before_local['retl_cnt_excel'], errors='coerce').fillna(0)).abs()
    if 'term_cnt_abs' not in cmp_before_local.columns:
        cmp_before_local['term_cnt_abs'] = (pd.to_numeric(cmp_before_local['term_cnt_lake'], errors='coerce').fillna(0) - pd.to_numeric(cmp_before_local['term_cnt_excel'], errors='coerce').fillna(0)).abs()

    kpi_retl_term_ab_df = pd.DataFrame([
        {
            'metric': 'rows_compared',
            'before_old_logic': len(cmp_before_local),
            'after_agrterms_logic': len(cmp_ab_retl_term_df),
            'delta': len(cmp_ab_retl_term_df) - len(cmp_before_local),
        },
        {
            'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_local['retl_cnt_abs'], tol=0.0),
            'after_agrterms_logic': exact_match_rate_from_abs(cmp_ab_retl_term_df['retl_cnt_abs_ab'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_ab_retl_term_df['retl_cnt_abs_ab'], tol=0.0) - exact_match_rate_from_abs(cmp_before_local['retl_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'term_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_local['term_cnt_abs'], tol=0.0),
            'after_agrterms_logic': exact_match_rate_from_abs(cmp_ab_retl_term_df['term_cnt_abs_ab'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_ab_retl_term_df['term_cnt_abs_ab'], tol=0.0) - exact_match_rate_from_abs(cmp_before_local['term_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'retl_cnt_mae',
            'before_old_logic': float(pd.to_numeric(cmp_before_local['retl_cnt_abs'], errors='coerce').mean()) if len(cmp_before_local) else 0.0,
            'after_agrterms_logic': float(pd.to_numeric(cmp_ab_retl_term_df['retl_cnt_abs_ab'], errors='coerce').mean()) if len(cmp_ab_retl_term_df) else 0.0,
            'delta': (float(pd.to_numeric(cmp_ab_retl_term_df['retl_cnt_abs_ab'], errors='coerce').mean()) if len(cmp_ab_retl_term_df) else 0.0) - (float(pd.to_numeric(cmp_before_local['retl_cnt_abs'], errors='coerce').mean()) if len(cmp_before_local) else 0.0),
        },
        {
            'metric': 'term_cnt_mae',
            'before_old_logic': float(pd.to_numeric(cmp_before_local['term_cnt_abs'], errors='coerce').mean()) if len(cmp_before_local) else 0.0,
            'after_agrterms_logic': float(pd.to_numeric(cmp_ab_retl_term_df['term_cnt_abs_ab'], errors='coerce').mean()) if len(cmp_ab_retl_term_df) else 0.0,
            'delta': (float(pd.to_numeric(cmp_ab_retl_term_df['term_cnt_abs_ab'], errors='coerce').mean()) if len(cmp_ab_retl_term_df) else 0.0) - (float(pd.to_numeric(cmp_before_local['term_cnt_abs'], errors='coerce').mean()) if len(cmp_before_local) else 0.0),
        },
    ])

    print('\nKPI: old vs AGR_TERMS retl/term')
    display(kpi_retl_term_ab_df)

    # Дополнительно: процент расхождения Lake vs Excel (старый vs новый подход)
    def _series_num(df, col):
        if col not in df.columns:
            return pd.Series(dtype='float64')
        return pd.to_numeric(df[col], errors='coerce')

    def _pct_div(lake_total, excel_total):
        if pd.isna(excel_total) or float(excel_total) == 0.0:
            return None
        return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0

    divergence_pct_rows = []
    for metric_name in ['retl_cnt', 'term_cnt']:
        old_lake_total = _series_num(cmp_before_local, f'{metric_name}_lake').fillna(0).sum()
        old_excel_total = _series_num(cmp_before_local, f'{metric_name}_excel').fillna(0).sum()

        new_lake_total = _series_num(cmp_ab_retl_term_df, f'{metric_name}_lake_ab').fillna(0).sum()
        new_excel_total = _series_num(cmp_ab_retl_term_df, f'{metric_name}_excel').fillna(0).sum()

        old_pct = _pct_div(old_lake_total, old_excel_total)
        new_pct = _pct_div(new_lake_total, new_excel_total)

        divergence_pct_rows.append({
            'metric': metric_name,
            'old_lake_total': old_lake_total,
            'old_excel_total': old_excel_total,
            'old_divergence_pct': old_pct,
            'old_abs_divergence_pct': abs(old_pct) if old_pct is not None else None,
            'new_lake_total': new_lake_total,
            'new_excel_total': new_excel_total,
            'new_divergence_pct': new_pct,
            'new_abs_divergence_pct': abs(new_pct) if new_pct is not None else None,
            'abs_divergence_improvement_pct_points': (
                abs(old_pct) - abs(new_pct)
                if old_pct is not None and new_pct is not None
                else None
            ),
        })

    divergence_pct_df = pd.DataFrame(divergence_pct_rows)
    print('\nПроцент расхождения Lake vs Excel (old vs AGR_TERMS):')
    display(divergence_pct_df)

    # QC 1: no AGR_TERMS links
    qc_no_agrterms_nmrc_df = ab_compare_df[
        pd.to_numeric(ab_compare_df['agr_terms_nmrc_cnt'], errors='coerce').fillna(0) <= 0
    ].copy()

    print('\nQC: договоры без c_nmrc в AGR_TERMS (на период):')
    display(qc_no_agrterms_nmrc_df[['agr_id', 'n_agr', 'n_cmp_client', 'retl_cnt_old', 'term_cnt_old']].head(200))

    # QC 2: fan-out reduction
    qc_fanout_reduction_df = ab_compare_df[
        (pd.to_numeric(ab_compare_df['retl_reduction_old_minus_ab'], errors='coerce').fillna(0) > 0)
        | (pd.to_numeric(ab_compare_df['term_reduction_old_minus_ab'], errors='coerce').fillna(0) > 0)
    ].copy()

    qc_fanout_reduction_df = qc_fanout_reduction_df.sort_values(
        ['retl_reduction_old_minus_ab', 'term_reduction_old_minus_ab'],
        ascending=False
    )

    print('QC: договоры, где AGR_TERMS уменьшает fan-out:')
    display(qc_fanout_reduction_df[['agr_id', 'n_agr', 'n_cmp_client', 'retl_cnt_old', 'retl_cnt_ab', 'retl_reduction_old_minus_ab', 'term_cnt_old', 'term_cnt_ab', 'term_reduction_old_minus_ab']].head(200))

    # Case 493131251933 proof
    case_agr_id = '493131251933'
    case_n_agr_values = _clean_keys(ab_counts_df.loc[ab_counts_df['agr_id'].astype(str) == case_agr_id, 'n_agr'].tolist()) if len(ab_counts_df) else []
    if not case_n_agr_values:
        case_n_agr_values = _clean_keys(sa_df.loc[sa_df['agr_id'].apply(normalize_agr_q1) == case_agr_id, 'n_agr'].tolist())

    if case_n_agr_values:
        case_n_agr_in = _sql_in(case_n_agr_values)

        sql_case493_terms_ab = f"""
        select distinct
          cast(t.n_agr as string) as n_agr,
          cast(t.c_nmrc as string) as c_nmrc
        from ods_alpha.scd1_agr_terms t
        where cast(t.n_agr as string) in ({case_n_agr_in})
          and t.c_nmrc is not null
          and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
          and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
          and upper(trim(cast(t.cf_ter_type as string))) = 'P'
          and coalesce(t.ods_deleted_flg, '0') <> '1'
        order by cast(t.n_agr as string), cast(t.c_nmrc as string)
        """

        sql_case493_terminals_ab = f"""
        with terms as (
          select distinct
            cast(t.n_agr as string) as n_agr,
            cast(t.c_nmrc as string) as c_nmrc
          from ods_alpha.scd1_agr_terms t
          where cast(t.n_agr as string) in ({case_n_agr_in})
            and t.c_nmrc is not null
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
            and upper(trim(cast(t.cf_ter_type as string))) = 'P'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
        )
        select distinct
          terms.n_agr,
          terms.c_nmrc,
          cast(p.c_nter as string) as c_nter,
          cast(p.d_ter_install as date) as d_ter_install,
          cast(p.d_ter_close as date) as d_ter_close
        from terms
        join ods_alpha.scd1_pos_terminals p
          on cast(p.c_nmrc as string) = terms.c_nmrc
        where p.c_nter is not null
          and coalesce(p.ods_deleted_flg, '0') <> '1'
          and coalesce(cast(p.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
          and coalesce(cast(p.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
        order by terms.n_agr, terms.c_nmrc, cast(p.c_nter as string)
        """

        with imp:
            imp.execute('set MEM_LIMIT=8g')
            case493_terms_ab_df = imp.fetch(sql_case493_terms_ab)
            case493_terminals_ab_df = imp.fetch(sql_case493_terminals_ab)
    else:
        case493_terms_ab_df = pd.DataFrame(columns=['n_agr', 'c_nmrc'])
        case493_terminals_ab_df = pd.DataFrame(columns=['n_agr', 'c_nmrc', 'c_nter', 'd_ter_install', 'd_ter_close'])

    if case493_terms_ab_df is None:
        case493_terms_ab_df = pd.DataFrame(columns=['n_agr', 'c_nmrc'])
    if case493_terminals_ab_df is None:
        case493_terminals_ab_df = pd.DataFrame(columns=['n_agr', 'c_nmrc', 'c_nter', 'd_ter_install', 'd_ter_close'])

    case493_proof_df = pd.DataFrame([
        {
            'agr_id': '493131251933',
            'n_agr_list': ','.join(case_n_agr_values) if case_n_agr_values else None,
            'agr_terms_nmrc_cnt': int(case493_terms_ab_df['c_nmrc'].nunique()) if len(case493_terms_ab_df) else 0,
            'term_active_cnt_ab': int(case493_terminals_ab_df['c_nter'].nunique()) if len(case493_terminals_ab_df) else 0,
        }
    ])

    print('\nCase proof (agr_id=493131251933):')
    display(case493_proof_df)

    print('Case 493 AGR_TERMS c_nmrc list:')
    display(case493_terms_ab_df)

    print('Case 493 terminals by AGR_TERMS c_nmrc (active in month):')
    display(case493_terminals_ab_df)

    if 'terminals_with_trx_df' in globals() and isinstance(terminals_with_trx_df, pd.DataFrame) and len(terminals_with_trx_df):
        ab_case_terms = {
            str(x).strip() for x in case493_terminals_ab_df.get('c_nter', pd.Series(dtype=object)).dropna().tolist()
            if str(x).strip() not in ['', 'None', 'nan', 'NaN']
        }
        trx_case_terms = {
            str(x).strip() for x in terminals_with_trx_df.get('c_nter', pd.Series(dtype=object)).dropna().tolist()
            if str(x).strip() not in ['', 'None', 'nan', 'NaN']
        }
        all_case_terms = sorted(ab_case_terms | trx_case_terms)

        if all_case_terms:
            case_cross_rows = []
            for t in all_case_terms:
                in_ab = t in ab_case_terms
                in_trx = t in trx_case_terms
                if in_ab and in_trx:
                    rel = 'in_both'
                elif in_ab:
                    rel = 'only_ab_active'
                else:
                    rel = 'only_trx'
                case_cross_rows.append({
                    'c_nter': t,
                    'in_ab_active_terms': in_ab,
                    'in_trx_terms': in_trx,
                    'relation': rel,
                })
            case493_cross_ab_trx_df = pd.DataFrame(case_cross_rows)
            print('Case 493 cross-check: AGR_TERMS active terminals vs trx terminals:')
            display(case493_cross_ab_trx_df)


### 3.13 A/B: multi-agr по ИНН (current vs expanded SA perimeter) и влияние на расхождения с Excel

Цель секции:
- проверить гипотезу, что часть расхождений с Excel возникает из-за потери `agr_id` для ИНН с несколькими договорами;
- сравнить **текущую логику** и **expanded perimeter** (без `exists` по `scd1_agr_terms`) по KPI `retl_cnt`/`term_cnt` против Excel;
- отдельно посмотреть кейс `ИНН=7106527696`.

In [ ]:
# 3.13 A/B: current vs expanded SA perimeter (multi-agr by INN)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните Секцию 2 compare (нужен cmp).')
if 'sa_df' not in globals() or sa_df is None or len(sa_df) == 0:
    raise RuntimeError('Сначала выполните 01_sa_perimeter (нужен sa_df).')


def _clean_keys_local(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in_local(values):
    vals = _clean_keys_local(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def _mae(series):
    s = pd.to_numeric(series, errors='coerce')
    return float(s.mean()) if len(s) else 0.0


# A) Expanded SA perimeter: убираем exists(sc1_agr_terms...) hard-filter
sql_sa_perimeter_expanded = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
"""

with imp:
    imp.execute('set MEM_LIMIT=12g')
    sa_expanded_df = imp.fetch(sql_sa_perimeter_expanded)

if sa_expanded_df is None:
    sa_expanded_df = pd.DataFrame(columns=['n_agr', 'agr_id', 'n_cmp_client', 'contract_number', 'd_valid_from', 'd_valid_to', 'inn', 'company_name'])

if len(sa_expanded_df):
    sa_expanded_df['inn'] = sa_expanded_df['inn'].map(normalize_inn)
    sa_expanded_df['n_agr'] = sa_expanded_df['n_agr'].astype(str)
    sa_expanded_df['agr_id'] = sa_expanded_df['agr_id'].astype(str)

# B) retl/term counts for expanded perimeter (same counting logic as 04_operational_metrics)
n_agrs_expanded = _clean_keys_local(sa_expanded_df.get('n_agr', pd.Series(dtype=object)).tolist()) if len(sa_expanded_df) else []

if n_agrs_expanded:
    n_agr_in_expanded = _sql_in_local(n_agrs_expanded)
    sql_cmp_expanded = f"""
    with sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr, cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in_expanded})
    ),
    m as (
      select sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_active as (
      select m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string) as c_nter
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    )
    select sa.n_agr, sa.n_cmp_client, r.retl_cnt, t.term_cnt
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        cmp_expanded_df = imp.fetch(sql_cmp_expanded)
else:
    cmp_expanded_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt'])

if cmp_expanded_df is None:
    cmp_expanded_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt'])

if len(cmp_expanded_df):
    cmp_expanded_df['n_agr'] = cmp_expanded_df['n_agr'].astype(str)
    cmp_expanded_df['retl_cnt'] = pd.to_numeric(cmp_expanded_df['retl_cnt'], errors='coerce')
    cmp_expanded_df['term_cnt'] = pd.to_numeric(cmp_expanded_df['term_cnt'], errors='coerce')

# C) OLD lake keys from current final_df
old_lk_df = final_df.copy()
old_lk_df['inn_key'] = old_lk_df['inn'].apply(normalize_inn_q1)
old_lk_df['agr_id_key'] = old_lk_df['agr_id'].apply(normalize_agr_q1)
old_lk_df['retl_cnt_lake_old'] = pd.to_numeric(old_lk_df['retl_cnt'], errors='coerce')
old_lk_df['term_cnt_lake_old'] = pd.to_numeric(old_lk_df['term_cnt'], errors='coerce')

old_agg_df = (
    old_lk_df.dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg({
        'retl_cnt_lake_old': 'max',
        'term_cnt_lake_old': 'max',
    })
)

# D) NEW lake keys from expanded SA perimeter + expanded retl/term
expanded_metrics_df = sa_expanded_df.merge(
    cmp_expanded_df[['n_agr', 'retl_cnt', 'term_cnt']],
    on='n_agr',
    how='left'
)
expanded_metrics_df['inn_key'] = expanded_metrics_df['inn'].apply(normalize_inn_q1)
expanded_metrics_df['agr_id_key'] = expanded_metrics_df['agr_id'].apply(normalize_agr_q1)
expanded_metrics_df['retl_cnt_lake_new'] = pd.to_numeric(expanded_metrics_df['retl_cnt'], errors='coerce')
expanded_metrics_df['term_cnt_lake_new'] = pd.to_numeric(expanded_metrics_df['term_cnt'], errors='coerce')

new_agg_df = (
    expanded_metrics_df.dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg({
        'retl_cnt_lake_new': 'max',
        'term_cnt_lake_new': 'max',
    })
)

# E) Excel keys (retl/term) from section 2
if 'ex_agg' in globals() and isinstance(ex_agg, pd.DataFrame) and len(ex_agg):
    ex_retl_term_df = ex_agg[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']].copy()
else:
    ex_retl_term_df = (
        cmp[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
        .drop_duplicates(subset=['inn_key', 'agr_id_key'])
        .copy()
    )

ex_retl_term_df['retl_cnt_excel'] = pd.to_numeric(ex_retl_term_df['retl_cnt_excel'], errors='coerce')
ex_retl_term_df['term_cnt_excel'] = pd.to_numeric(ex_retl_term_df['term_cnt_excel'], errors='coerce')

cmp_old_ab_df = old_agg_df.merge(ex_retl_term_df, on=['inn_key', 'agr_id_key'], how='inner')
cmp_new_ab_df = new_agg_df.merge(ex_retl_term_df, on=['inn_key', 'agr_id_key'], how='inner')

for m in ['retl_cnt', 'term_cnt']:
    cmp_old_ab_df[f'{m}_delta_old'] = pd.to_numeric(cmp_old_ab_df[f'{m}_lake_old'], errors='coerce').fillna(0) - pd.to_numeric(cmp_old_ab_df[f'{m}_excel'], errors='coerce').fillna(0)
    cmp_old_ab_df[f'{m}_abs_old'] = pd.to_numeric(cmp_old_ab_df[f'{m}_delta_old'], errors='coerce').abs()

    cmp_new_ab_df[f'{m}_delta_new'] = pd.to_numeric(cmp_new_ab_df[f'{m}_lake_new'], errors='coerce').fillna(0) - pd.to_numeric(cmp_new_ab_df[f'{m}_excel'], errors='coerce').fillna(0)
    cmp_new_ab_df[f'{m}_abs_new'] = pd.to_numeric(cmp_new_ab_df[f'{m}_delta_new'], errors='coerce').abs()

# F) Coverage and KPI
excel_key_df = ex_retl_term_df[['inn_key', 'agr_id_key']].drop_duplicates()
old_key_df = old_agg_df[['inn_key', 'agr_id_key']].drop_duplicates()
new_key_df = new_agg_df[['inn_key', 'agr_id_key']].drop_duplicates()

old_cov_cnt = len(excel_key_df.merge(old_key_df, on=['inn_key', 'agr_id_key'], how='inner'))
new_cov_cnt = len(excel_key_df.merge(new_key_df, on=['inn_key', 'agr_id_key'], how='inner'))
excel_key_cnt = len(excel_key_df)

added_keys_vs_old_df = (
    new_key_df.merge(old_key_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

kpi_multi_agr_ab_df = pd.DataFrame([
    {
        'metric': 'excel_key_count',
        'old_logic': excel_key_cnt,
        'new_expanded_perimeter': excel_key_cnt,
        'delta_new_minus_old': 0,
    },
    {
        'metric': 'covered_excel_keys',
        'old_logic': old_cov_cnt,
        'new_expanded_perimeter': new_cov_cnt,
        'delta_new_minus_old': new_cov_cnt - old_cov_cnt,
    },
    {
        'metric': 'rows_compared_retl_term',
        'old_logic': len(cmp_old_ab_df),
        'new_expanded_perimeter': len(cmp_new_ab_df),
        'delta_new_minus_old': len(cmp_new_ab_df) - len(cmp_old_ab_df),
    },
    {
        'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
        'old_logic': exact_match_rate_from_abs(cmp_old_ab_df['retl_cnt_abs_old'], tol=0.0),
        'new_expanded_perimeter': exact_match_rate_from_abs(cmp_new_ab_df['retl_cnt_abs_new'], tol=0.0),
        'delta_new_minus_old': exact_match_rate_from_abs(cmp_new_ab_df['retl_cnt_abs_new'], tol=0.0) - exact_match_rate_from_abs(cmp_old_ab_df['retl_cnt_abs_old'], tol=0.0),
    },
    {
        'metric': 'term_cnt_exact_match_rate_pct_tol_0',
        'old_logic': exact_match_rate_from_abs(cmp_old_ab_df['term_cnt_abs_old'], tol=0.0),
        'new_expanded_perimeter': exact_match_rate_from_abs(cmp_new_ab_df['term_cnt_abs_new'], tol=0.0),
        'delta_new_minus_old': exact_match_rate_from_abs(cmp_new_ab_df['term_cnt_abs_new'], tol=0.0) - exact_match_rate_from_abs(cmp_old_ab_df['term_cnt_abs_old'], tol=0.0),
    },
    {
        'metric': 'retl_cnt_mae',
        'old_logic': _mae(cmp_old_ab_df['retl_cnt_abs_old']),
        'new_expanded_perimeter': _mae(cmp_new_ab_df['retl_cnt_abs_new']),
        'delta_new_minus_old': _mae(cmp_new_ab_df['retl_cnt_abs_new']) - _mae(cmp_old_ab_df['retl_cnt_abs_old']),
    },
    {
        'metric': 'term_cnt_mae',
        'old_logic': _mae(cmp_old_ab_df['term_cnt_abs_old']),
        'new_expanded_perimeter': _mae(cmp_new_ab_df['term_cnt_abs_new']),
        'delta_new_minus_old': _mae(cmp_new_ab_df['term_cnt_abs_new']) - _mae(cmp_old_ab_df['term_cnt_abs_old']),
    },
])

# G) INN-level multi-agr coverage diagnostics
inn_agr_current_df = (
    sa_df.assign(inn_key=sa_df['inn'].apply(normalize_inn_q1))
    .dropna(subset=['inn_key'])
    .groupby('inn_key', as_index=False)
    .agg(current_n_agr=('n_agr', lambda s: pd.Series(s.astype(str)).nunique()))
)

inn_agr_expanded_df = (
    sa_expanded_df.assign(inn_key=sa_expanded_df['inn'].apply(normalize_inn_q1))
    .dropna(subset=['inn_key'])
    .groupby('inn_key', as_index=False)
    .agg(expanded_n_agr=('n_agr', lambda s: pd.Series(s.astype(str)).nunique()))
)

inn_agr_excel_df = (
    ex_retl_term_df.groupby('inn_key', as_index=False)
    .agg(excel_n_agr=('agr_id_key', 'nunique'))
)

inn_agr_coverage_df = (
    inn_agr_excel_df
    .merge(inn_agr_current_df, on='inn_key', how='left')
    .merge(inn_agr_expanded_df, on='inn_key', how='left')
)
inn_agr_coverage_df[['current_n_agr', 'expanded_n_agr']] = inn_agr_coverage_df[['current_n_agr', 'expanded_n_agr']].fillna(0)
inn_agr_coverage_df['missing_vs_excel_old'] = inn_agr_coverage_df['excel_n_agr'] - inn_agr_coverage_df['current_n_agr']
inn_agr_coverage_df['missing_vs_excel_new'] = inn_agr_coverage_df['excel_n_agr'] - inn_agr_coverage_df['expanded_n_agr']
inn_agr_coverage_df['improvement_agr_coverage'] = inn_agr_coverage_df['missing_vs_excel_old'] - inn_agr_coverage_df['missing_vs_excel_new']

inn_agr_coverage_df = inn_agr_coverage_df.sort_values(
    ['missing_vs_excel_old', 'improvement_agr_coverage', 'excel_n_agr'],
    ascending=[False, False, False]
)

# H) Focus INN = 7106527696
focus_inn = '7106527696'
focus_excel_df = ex_retl_term_df[ex_retl_term_df['inn_key'] == focus_inn].copy()
focus_old_df = old_agg_df[old_agg_df['inn_key'] == focus_inn].copy()
focus_new_df = new_agg_df[new_agg_df['inn_key'] == focus_inn].copy()

focus_compare_df = (
    focus_excel_df
    .merge(focus_old_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_old', 'term_cnt_lake_old']], on=['inn_key', 'agr_id_key'], how='left')
    .merge(focus_new_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_new', 'term_cnt_lake_new']], on=['inn_key', 'agr_id_key'], how='left')
)

if len(focus_compare_df):
    focus_compare_df['retl_delta_old'] = pd.to_numeric(focus_compare_df['retl_cnt_lake_old'], errors='coerce').fillna(0) - pd.to_numeric(focus_compare_df['retl_cnt_excel'], errors='coerce').fillna(0)
    focus_compare_df['retl_delta_new'] = pd.to_numeric(focus_compare_df['retl_cnt_lake_new'], errors='coerce').fillna(0) - pd.to_numeric(focus_compare_df['retl_cnt_excel'], errors='coerce').fillna(0)
    focus_compare_df['term_delta_old'] = pd.to_numeric(focus_compare_df['term_cnt_lake_old'], errors='coerce').fillna(0) - pd.to_numeric(focus_compare_df['term_cnt_excel'], errors='coerce').fillna(0)
    focus_compare_df['term_delta_new'] = pd.to_numeric(focus_compare_df['term_cnt_lake_new'], errors='coerce').fillna(0) - pd.to_numeric(focus_compare_df['term_cnt_excel'], errors='coerce').fillna(0)

print('A/B KPI (current vs expanded perimeter):')
display(kpi_multi_agr_ab_df)

print('Added lake keys in NEW vs OLD (inn_key + agr_id_key):')
print(len(added_keys_vs_old_df))
display(added_keys_vs_old_df.head(200))

print('INN-level multi-agr coverage (Excel vs old vs new):')
display(inn_agr_coverage_df.head(200))

print(f'Focus INN {focus_inn}: Excel vs old/new lake keys and deltas')
display(focus_compare_df)

print('Current SA perimeter rows:', len(sa_df), '| Expanded SA perimeter rows:', len(sa_expanded_df))
print('Current unique (inn,agr_id):', sa_df[['inn', 'agr_id']].dropna().drop_duplicates().shape[0])
print('Expanded unique (inn,agr_id):', sa_expanded_df[['inn', 'agr_id']].dropna().drop_duplicates().shape[0])

### 3.14 A/B: `AGR_TERMS-first` (fallback в `n_cmp_client` только если нет `c_nmrc`) — влияние на KPI и кейс `ИНН=7106527696`

Идея проверки:
- для каждого `n_agr` сначала берем периметр точек из `ods_alpha.scd1_agr_terms`;
- fallback к `ods_alpha.scd1_merchants` по `n_cmp_client` используем **только** если у этого `n_agr` нет `c_nmrc` в `AGR_TERMS` за период;
- сравниваем со старой логикой по KPI `retl_cnt`/`term_cnt` и отдельно проверяем фокус-кейс `7106527696`.

In [ ]:
# 3.14 A/B check: AGR_TERMS-first + fallback-by-n_cmp only when no AGR_TERMS c_nmrc
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')
if 'sa_df' not in globals() or sa_df is None or len(sa_df) == 0:
    raise RuntimeError('Сначала выполните 01_sa_perimeter (нужен sa_df).')
if 'cmp_df' not in globals() or cmp_df is None or len(cmp_df) == 0:
    raise RuntimeError('Сначала выполните 04_operational_metrics (нужен cmp_df).')
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните Секцию 2 compare (нужен cmp).')


def _clean_keys_314(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in_314(values):
    vals = _clean_keys_314(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


n_agr_values_314 = _clean_keys_314(sa_df.get('n_agr', pd.Series(dtype=object)).tolist())

if not n_agr_values_314:
    print('В sa_df нет n_agr для 3.14 проверки.')
    ab314_counts_df = pd.DataFrame()
    cmp_ab314_retl_term_df = pd.DataFrame()
    kpi_retl_term_ab314_df = pd.DataFrame()
    divergence_pct_ab314_df = pd.DataFrame()
    fallback_summary_314_df = pd.DataFrame()
    focus_7106527696_ab314_df = pd.DataFrame()
else:
    n_agr_in_314 = _sql_in_314(n_agr_values_314)

    # A) Build AGR_TERMS-first perimeter with fallback only for n_agr without AGR_TERMS c_nmrc
    sql_ab314_counts = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({n_agr_in_314})
    ),
    agr_terms_nmrc as (
      select
        sa.n_agr,
        sa.agr_id,
        sa.n_cmp_client,
        cast(t.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_agr_terms t
        on cast(t.n_agr as string) = sa.n_agr
      where t.c_nmrc is not null
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.agr_id, sa.n_cmp_client, cast(t.c_nmrc as string)
    ),
    agr_terms_presence as (
      select
        n_agr,
        count(distinct c_nmrc) as agr_terms_nmrc_cnt
      from agr_terms_nmrc
      group by n_agr
    ),
    fallback_nmrc as (
      select
        sa.n_agr,
        sa.agr_id,
        sa.n_cmp_client,
        cast(m.c_nmrc as string) as c_nmrc
      from sa_agr sa
      left join agr_terms_presence p
        on p.n_agr = sa.n_agr
      join ods_alpha.scd1_merchants m
        on cast(m.n_cmp as string) = sa.n_cmp_client
      where coalesce(p.agr_terms_nmrc_cnt, 0) = 0
        and m.c_nmrc is not null
        and coalesce(m.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.agr_id, sa.n_cmp_client, cast(m.c_nmrc as string)
    ),
    perimeter_nmrc as (
      select n_agr, agr_id, n_cmp_client, c_nmrc, 'agr_terms' as source_nmrc
      from agr_terms_nmrc
      union all
      select n_agr, agr_id, n_cmp_client, c_nmrc, 'fallback_n_cmp' as source_nmrc
      from fallback_nmrc
    ),
    perimeter_nmrc_dedup as (
      select n_agr, agr_id, n_cmp_client, c_nmrc, max(source_nmrc) as source_nmrc
      from perimeter_nmrc
      group by n_agr, agr_id, n_cmp_client, c_nmrc
    ),
    term_active as (
      select
        p.n_agr,
        p.agr_id,
        p.n_cmp_client,
        p.c_nmrc,
        p.source_nmrc,
        cast(t.c_nter as string) as c_nter
      from perimeter_nmrc_dedup p
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = p.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by p.n_agr, p.agr_id, p.n_cmp_client, p.c_nmrc, p.source_nmrc, cast(t.c_nter as string)
    ),
    cnt as (
      select
        n_agr,
        count(distinct c_nmrc) as retl_cnt_ab314,
        count(distinct c_nter) as term_cnt_ab314
      from term_active
      group by n_agr
    ),
    src as (
      select
        n_agr,
        max(case when source_nmrc = 'agr_terms' then 1 else 0 end) as has_agr_terms_source,
        max(case when source_nmrc = 'fallback_n_cmp' then 1 else 0 end) as has_fallback_source
      from perimeter_nmrc_dedup
      group by n_agr
    )
    select
      sa.n_agr,
      sa.agr_id,
      sa.n_cmp_client,
      coalesce(p.agr_terms_nmrc_cnt, 0) as agr_terms_nmrc_cnt,
      coalesce(c.retl_cnt_ab314, 0) as retl_cnt_ab314,
      coalesce(c.term_cnt_ab314, 0) as term_cnt_ab314,
      coalesce(s.has_agr_terms_source, 0) as has_agr_terms_source,
      coalesce(s.has_fallback_source, 0) as has_fallback_source
    from sa_agr sa
    left join agr_terms_presence p on p.n_agr = sa.n_agr
    left join cnt c on c.n_agr = sa.n_agr
    left join src s on s.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        ab314_counts_df = imp.fetch(sql_ab314_counts)

    if ab314_counts_df is None:
        ab314_counts_df = pd.DataFrame(columns=[
            'n_agr', 'agr_id', 'n_cmp_client', 'agr_terms_nmrc_cnt',
            'retl_cnt_ab314', 'term_cnt_ab314', 'has_agr_terms_source', 'has_fallback_source'
        ])

    if len(ab314_counts_df):
        for c in ['n_agr', 'agr_id', 'n_cmp_client']:
            if c in ab314_counts_df.columns:
                ab314_counts_df[c] = ab314_counts_df[c].astype(str)
        for c in ['agr_terms_nmrc_cnt', 'retl_cnt_ab314', 'term_cnt_ab314', 'has_agr_terms_source', 'has_fallback_source']:
            if c in ab314_counts_df.columns:
                ab314_counts_df[c] = pd.to_numeric(ab314_counts_df[c], errors='coerce')

    # B) Build lake_ab314 by inn_key+agr_id_key and compare to Excel
    lk_ab314 = final_df.copy()
    lk_ab314['inn_key'] = lk_ab314['inn'].apply(normalize_inn_q1)
    lk_ab314['agr_id_key'] = lk_ab314['agr_id'].apply(normalize_agr_q1)
    lk_ab314['n_agr'] = lk_ab314['n_agr'].astype(str)

    lk_ab314 = lk_ab314.merge(
        ab314_counts_df[['n_agr', 'retl_cnt_ab314', 'term_cnt_ab314']],
        on='n_agr',
        how='left'
    )

    lk_ab314['retl_cnt_lake_ab314'] = pd.to_numeric(lk_ab314['retl_cnt_ab314'], errors='coerce')
    lk_ab314['term_cnt_lake_ab314'] = pd.to_numeric(lk_ab314['term_cnt_ab314'], errors='coerce')

    lk_ab314_agg = (
        lk_ab314.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt_lake_ab314': 'max',
            'term_cnt_lake_ab314': 'max',
        })
    )

    if 'ex_agg' in globals() and isinstance(ex_agg, pd.DataFrame) and len(ex_agg):
        ex_retl_term_df_314 = ex_agg[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']].copy()
    else:
        ex_retl_term_df_314 = (
            cmp[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
            .drop_duplicates(subset=['inn_key', 'agr_id_key'])
            .copy()
        )

    cmp_ab314_retl_term_df = lk_ab314_agg.merge(
        ex_retl_term_df_314,
        on=['inn_key', 'agr_id_key'],
        how='inner'
    )

    for m in ['retl_cnt', 'term_cnt']:
        cmp_ab314_retl_term_df[f'{m}_delta_ab314'] = pd.to_numeric(cmp_ab314_retl_term_df[f'{m}_lake_ab314'], errors='coerce').fillna(0) - pd.to_numeric(cmp_ab314_retl_term_df[f'{m}_excel'], errors='coerce').fillna(0)
        cmp_ab314_retl_term_df[f'{m}_abs_ab314'] = pd.to_numeric(cmp_ab314_retl_term_df[f'{m}_delta_ab314'], errors='coerce').abs()

    # before (existing cmp)
    cmp_before_314 = cmp.copy()
    if 'retl_cnt_abs' not in cmp_before_314.columns:
        cmp_before_314['retl_cnt_abs'] = (
            pd.to_numeric(cmp_before_314['retl_cnt_lake'], errors='coerce').fillna(0)
            - pd.to_numeric(cmp_before_314['retl_cnt_excel'], errors='coerce').fillna(0)
        ).abs()
    if 'term_cnt_abs' not in cmp_before_314.columns:
        cmp_before_314['term_cnt_abs'] = (
            pd.to_numeric(cmp_before_314['term_cnt_lake'], errors='coerce').fillna(0)
            - pd.to_numeric(cmp_before_314['term_cnt_excel'], errors='coerce').fillna(0)
        ).abs()

    # coverage counters
    excel_keys_314 = ex_retl_term_df_314[['inn_key', 'agr_id_key']].drop_duplicates()
    old_keys_314 = cmp_before_314[['inn_key', 'agr_id_key']].drop_duplicates()
    new_keys_314 = cmp_ab314_retl_term_df[['inn_key', 'agr_id_key']].drop_duplicates()

    covered_old_314 = len(excel_keys_314.merge(old_keys_314, on=['inn_key', 'agr_id_key'], how='inner'))
    covered_new_314 = len(excel_keys_314.merge(new_keys_314, on=['inn_key', 'agr_id_key'], how='inner'))

    kpi_retl_term_ab314_df = pd.DataFrame([
        {
            'metric': 'excel_key_count',
            'before_old_logic': len(excel_keys_314),
            'after_ab314_logic': len(excel_keys_314),
            'delta': 0,
        },
        {
            'metric': 'covered_excel_keys',
            'before_old_logic': covered_old_314,
            'after_ab314_logic': covered_new_314,
            'delta': covered_new_314 - covered_old_314,
        },
        {
            'metric': 'rows_compared',
            'before_old_logic': len(cmp_before_314),
            'after_ab314_logic': len(cmp_ab314_retl_term_df),
            'delta': len(cmp_ab314_retl_term_df) - len(cmp_before_314),
        },
        {
            'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_314['retl_cnt_abs'], tol=0.0),
            'after_ab314_logic': exact_match_rate_from_abs(cmp_ab314_retl_term_df['retl_cnt_abs_ab314'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_ab314_retl_term_df['retl_cnt_abs_ab314'], tol=0.0) - exact_match_rate_from_abs(cmp_before_314['retl_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'term_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_314['term_cnt_abs'], tol=0.0),
            'after_ab314_logic': exact_match_rate_from_abs(cmp_ab314_retl_term_df['term_cnt_abs_ab314'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_ab314_retl_term_df['term_cnt_abs_ab314'], tol=0.0) - exact_match_rate_from_abs(cmp_before_314['term_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'retl_cnt_mae',
            'before_old_logic': float(pd.to_numeric(cmp_before_314['retl_cnt_abs'], errors='coerce').mean()) if len(cmp_before_314) else 0.0,
            'after_ab314_logic': float(pd.to_numeric(cmp_ab314_retl_term_df['retl_cnt_abs_ab314'], errors='coerce').mean()) if len(cmp_ab314_retl_term_df) else 0.0,
            'delta': (float(pd.to_numeric(cmp_ab314_retl_term_df['retl_cnt_abs_ab314'], errors='coerce').mean()) if len(cmp_ab314_retl_term_df) else 0.0) - (float(pd.to_numeric(cmp_before_314['retl_cnt_abs'], errors='coerce').mean()) if len(cmp_before_314) else 0.0),
        },
        {
            'metric': 'term_cnt_mae',
            'before_old_logic': float(pd.to_numeric(cmp_before_314['term_cnt_abs'], errors='coerce').mean()) if len(cmp_before_314) else 0.0,
            'after_ab314_logic': float(pd.to_numeric(cmp_ab314_retl_term_df['term_cnt_abs_ab314'], errors='coerce').mean()) if len(cmp_ab314_retl_term_df) else 0.0,
            'delta': (float(pd.to_numeric(cmp_ab314_retl_term_df['term_cnt_abs_ab314'], errors='coerce').mean()) if len(cmp_ab314_retl_term_df) else 0.0) - (float(pd.to_numeric(cmp_before_314['term_cnt_abs'], errors='coerce').mean()) if len(cmp_before_314) else 0.0),
        },
    ])

    # percent divergence by totals
    def _num_series(df, col):
        if col not in df.columns:
            return pd.Series(dtype='float64')
        return pd.to_numeric(df[col], errors='coerce')

    def _pct_div(lake_total, excel_total):
        if pd.isna(excel_total) or float(excel_total) == 0.0:
            return None
        return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0

    divergence_rows_314 = []
    for metric_name in ['retl_cnt', 'term_cnt']:
        old_lake_total = _num_series(cmp_before_314, f'{metric_name}_lake').fillna(0).sum()
        old_excel_total = _num_series(cmp_before_314, f'{metric_name}_excel').fillna(0).sum()

        new_lake_total = _num_series(cmp_ab314_retl_term_df, f'{metric_name}_lake_ab314').fillna(0).sum()
        new_excel_total = _num_series(cmp_ab314_retl_term_df, f'{metric_name}_excel').fillna(0).sum()

        old_pct = _pct_div(old_lake_total, old_excel_total)
        new_pct = _pct_div(new_lake_total, new_excel_total)

        divergence_rows_314.append({
            'metric': metric_name,
            'old_lake_total': old_lake_total,
            'old_excel_total': old_excel_total,
            'old_divergence_pct': old_pct,
            'old_abs_divergence_pct': abs(old_pct) if old_pct is not None else None,
            'new_lake_total': new_lake_total,
            'new_excel_total': new_excel_total,
            'new_divergence_pct': new_pct,
            'new_abs_divergence_pct': abs(new_pct) if new_pct is not None else None,
            'abs_divergence_improvement_pct_points': (
                abs(old_pct) - abs(new_pct)
                if old_pct is not None and new_pct is not None
                else None
            ),
        })

    divergence_pct_ab314_df = pd.DataFrame(divergence_rows_314)

    # fallback usage summary
    fallback_summary_314_df = pd.DataFrame([
        {
            'total_n_agr': len(ab314_counts_df),
            'n_agr_with_agr_terms_source': int((pd.to_numeric(ab314_counts_df['has_agr_terms_source'], errors='coerce').fillna(0) > 0).sum()) if len(ab314_counts_df) else 0,
            'n_agr_with_fallback_source': int((pd.to_numeric(ab314_counts_df['has_fallback_source'], errors='coerce').fillna(0) > 0).sum()) if len(ab314_counts_df) else 0,
            'n_agr_only_fallback': int(((pd.to_numeric(ab314_counts_df['has_fallback_source'], errors='coerce').fillna(0) > 0) & (pd.to_numeric(ab314_counts_df['has_agr_terms_source'], errors='coerce').fillna(0) == 0)).sum()) if len(ab314_counts_df) else 0,
        }
    ])

    # focus INN check: 7106527696
    focus_inn = '7106527696'
    focus_cmp_excel = ex_retl_term_df_314[ex_retl_term_df_314['inn_key'] == focus_inn].copy()

    focus_old = (
        cmp_before_314[cmp_before_314['inn_key'] == focus_inn][
            ['inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake']
        ]
        .drop_duplicates(subset=['inn_key', 'agr_id_key'])
        .rename(columns={'retl_cnt_lake': 'retl_cnt_lake_old', 'term_cnt_lake': 'term_cnt_lake_old'})
    )

    focus_new = (
        cmp_ab314_retl_term_df[cmp_ab314_retl_term_df['inn_key'] == focus_inn][
            ['inn_key', 'agr_id_key', 'retl_cnt_lake_ab314', 'term_cnt_lake_ab314']
        ]
        .drop_duplicates(subset=['inn_key', 'agr_id_key'])
        .rename(columns={'retl_cnt_lake_ab314': 'retl_cnt_lake_new', 'term_cnt_lake_ab314': 'term_cnt_lake_new'})
    )

    focus_7106527696_ab314_df = (
        focus_cmp_excel
        .merge(focus_old, on=['inn_key', 'agr_id_key'], how='left')
        .merge(focus_new, on=['inn_key', 'agr_id_key'], how='left')
    )

    if len(focus_7106527696_ab314_df):
        focus_7106527696_ab314_df['retl_delta_old'] = pd.to_numeric(focus_7106527696_ab314_df['retl_cnt_lake_old'], errors='coerce').fillna(0) - pd.to_numeric(focus_7106527696_ab314_df['retl_cnt_excel'], errors='coerce').fillna(0)
        focus_7106527696_ab314_df['retl_delta_new'] = pd.to_numeric(focus_7106527696_ab314_df['retl_cnt_lake_new'], errors='coerce').fillna(0) - pd.to_numeric(focus_7106527696_ab314_df['retl_cnt_excel'], errors='coerce').fillna(0)
        focus_7106527696_ab314_df['term_delta_old'] = pd.to_numeric(focus_7106527696_ab314_df['term_cnt_lake_old'], errors='coerce').fillna(0) - pd.to_numeric(focus_7106527696_ab314_df['term_cnt_excel'], errors='coerce').fillna(0)
        focus_7106527696_ab314_df['term_delta_new'] = pd.to_numeric(focus_7106527696_ab314_df['term_cnt_lake_new'], errors='coerce').fillna(0) - pd.to_numeric(focus_7106527696_ab314_df['term_cnt_excel'], errors='coerce').fillna(0)

print('KPI: old vs AB314 (AGR_TERMS-first + fallback only if missing)')
display(kpi_retl_term_ab314_df)

print('\nПроцент расхождения Lake vs Excel (old vs AB314):')
display(divergence_pct_ab314_df)

print('\nFallback usage summary for AB314 perimeter:')
display(fallback_summary_314_df)

print('\nFocus INN 7106527696: Excel vs old/new')
display(focus_7106527696_ab314_df)

if len(cmp_ab314_retl_term_df):
    print('\nTOP-20 retl abs delta (AB314):')
    display(
        cmp_ab314_retl_term_df.sort_values('retl_cnt_abs_ab314', ascending=False)[
            ['agr_id_key', 'inn_key', 'retl_cnt_lake_ab314', 'retl_cnt_excel', 'retl_cnt_delta_ab314', 'retl_cnt_abs_ab314']
        ].head(20)
    )

    print('TOP-20 term abs delta (AB314):')
    display(
        cmp_ab314_retl_term_df.sort_values('term_cnt_abs_ab314', ascending=False)[
            ['agr_id_key', 'inn_key', 'term_cnt_lake_ab314', 'term_cnt_excel', 'term_cnt_delta_ab314', 'term_cnt_abs_ab314']
        ].head(20)
    )

### 3.15 SQL-шаблон "как в 547" для `retl_cnt`/`term_cnt` + фокус `ИНН=7106527696` + общий KPI к Excel

Что проверяем:
- периметр строго по `AGR_TERMS` (`N_AGR` -> `C_NMRC`) с фильтрами из 547;
- исключение резервных мерчантов;
- `term_cnt` как `COUNT(DISTINCT C_POS_SERIAL)` по активному окну периода;
- сравнение old logic vs `547-like` по KPI (`exact_match`, `MAE`, coverage, divergence %);
- отдельный разбор `ИНН 7106527696` на уровне `agr_id` и `c_nmrc`.

In [ ]:
# 3.15 A/B check: old logic vs 547-like retl/term logic
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')
if 'sa_df' not in globals() or sa_df is None or len(sa_df) == 0:
    raise RuntimeError('Сначала выполните 01_sa_perimeter (нужен sa_df).')
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните Секцию 2 compare (нужен cmp).')


def _clean_keys_315(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in_315(values):
    vals = _clean_keys_315(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def _to_num_315(df, col):
    if col not in df.columns:
        return pd.Series(dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


def _pct_div_315(lake_total, excel_total):
    if pd.isna(excel_total) or float(excel_total) == 0.0:
        return None
    return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0


n_agr_values_315 = _clean_keys_315(sa_df.get('n_agr', pd.Series(dtype=object)).tolist())
if not n_agr_values_315:
    print('В sa_df нет n_agr для 3.15 проверки.')
    counts_547_df = pd.DataFrame()
    cmp_547_retl_term_df = pd.DataFrame()
    kpi_547_vs_old_df = pd.DataFrame()
    divergence_547_vs_old_df = pd.DataFrame()
    focus_7106527696_547_df = pd.DataFrame()
    focus_7106527696_nmrc_547_df = pd.DataFrame()
else:
    n_agr_in_315 = _sql_in_315(n_agr_values_315)

    def _build_sql_547_counts(use_m_acq_filter=True):
        m_acq_filter_sql = "and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'" if use_m_acq_filter else ''

        return f"""
        with scope_n_agr as (
          select distinct cast(a.n_agr as string) as n_agr
          from ods_alpha.scd1_agreements a
          where cast(a.n_agr as string) in ({n_agr_in_315})
        ),
        am_raw as (
          select distinct
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.n_agr as string) as n_agr,
            cast(a.n_cmp_client as string) as n_cmp_client,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
            cast(t.c_nmrc as string) as c_nmrc
          from ods_alpha.scd1_agreements a
          join scope_n_agr s
            on s.n_agr = cast(a.n_agr as string)
          join ods_alpha.scd1_agr_terms t
            on cast(a.n_agr as string) = cast(t.n_agr as string)
          join ods_alpha.scd1_merchants m
            on cast(t.c_nmrc as string) = cast(m.c_nmrc as string)
          join ods_alpha.scd1_companies c
            on c.n_cmp = a.n_cmp_client
          where upper(trim(cast(a.acq_class as string))) = 'SA'
            and t.c_nmrc is not null
            and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
            {m_acq_filter_sql}
            and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
            and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
            and coalesce(a.ods_deleted_flg, '0') <> '1'
            and coalesce(t.ods_deleted_flg, '0') <> '1'
            and coalesce(m.ods_deleted_flg, '0') <> '1'
            and coalesce(c.ods_deleted_flg, '0') <> '1'
        ),
        am as (
          select
            am_raw.agr_id,
            am_raw.n_agr,
            am_raw.n_cmp_client,
            am_raw.inn,
            am_raw.c_nmrc,
            row_number() over (
              partition by am_raw.c_nmrc
              order by am_raw.c_nmrc, am_raw.n_agr
            ) as rn
          from am_raw
        ),
        pos_period as (
          select distinct
            cast(p.c_nmrc as string) as c_nmrc,
            cast(p.c_pos_serial as string) as c_pos_serial
          from ods_alpha.scd1_pos_terminals p
          where p.c_pos_serial is not null
            and coalesce(cast(p.d_ter_install as date), cast('1900-01-01' as date)) < date_add(cast('{month_end}' as date), 1)
            and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
            and coalesce(p.ods_deleted_flg, '0') <> '1'
        )
        select
          am.agr_id,
          am.n_agr,
          am.n_cmp_client,
          am.inn,
          count(distinct am.c_nmrc) as retl_cnt_547,
          count(distinct pp.c_pos_serial) as term_cnt_547,
          count(distinct case when am.rn = 1 then am.c_nmrc else null end) as retl_cnt_rn1_hint
        from am
        left join pos_period pp
          on pp.c_nmrc = am.c_nmrc
        group by am.agr_id, am.n_agr, am.n_cmp_client, am.inn
        """

    used_m_acq_filter_315 = True
    try:
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            counts_547_df = imp.fetch(_build_sql_547_counts(use_m_acq_filter=True))
    except Exception as exc_547:
        used_m_acq_filter_315 = False
        print(f"m.acq_class filter disabled in 3.15 due to: {type(exc_547).__name__}")
        with imp:
            imp.execute('set MEM_LIMIT=16g')
            counts_547_df = imp.fetch(_build_sql_547_counts(use_m_acq_filter=False))

    if counts_547_df is None:
        counts_547_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'inn', 'retl_cnt_547', 'term_cnt_547', 'retl_cnt_rn1_hint'])

    if len(counts_547_df):
        for c in ['agr_id', 'n_agr', 'n_cmp_client', 'inn']:
            if c in counts_547_df.columns:
                counts_547_df[c] = counts_547_df[c].astype(str)
        counts_547_df['inn_key'] = counts_547_df['inn'].apply(normalize_inn_q1)
        counts_547_df['agr_id_key'] = counts_547_df['agr_id'].apply(normalize_agr_q1)
        counts_547_df['retl_cnt_lake_547'] = pd.to_numeric(counts_547_df['retl_cnt_547'], errors='coerce')
        counts_547_df['term_cnt_lake_547'] = pd.to_numeric(counts_547_df['term_cnt_547'], errors='coerce')

    # NEW (547-like) aggregated by compare key
    lk_547_agg_df = (
        counts_547_df.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt_lake_547': 'max',
            'term_cnt_lake_547': 'max',
        })
    ) if len(counts_547_df) else pd.DataFrame(columns=['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547'])

    # Excel reference for retl/term
    if 'ex_agg' in globals() and isinstance(ex_agg, pd.DataFrame) and len(ex_agg):
        ex_retl_term_315_df = ex_agg[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']].copy()
    else:
        ex_retl_term_315_df = (
            cmp[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
            .drop_duplicates(subset=['inn_key', 'agr_id_key'])
            .copy()
        )

    cmp_547_retl_term_df = lk_547_agg_df.merge(
        ex_retl_term_315_df,
        on=['inn_key', 'agr_id_key'],
        how='inner'
    )

    for m in ['retl_cnt', 'term_cnt']:
        cmp_547_retl_term_df[f'{m}_delta_547'] = _to_num_315(cmp_547_retl_term_df, f'{m}_lake_547').fillna(0) - _to_num_315(cmp_547_retl_term_df, f'{m}_excel').fillna(0)
        cmp_547_retl_term_df[f'{m}_abs_547'] = _to_num_315(cmp_547_retl_term_df, f'{m}_delta_547').abs()

    # OLD baseline from existing cmp
    cmp_before_315 = cmp.copy()
    if 'retl_cnt_abs' not in cmp_before_315.columns:
        cmp_before_315['retl_cnt_abs'] = (_to_num_315(cmp_before_315, 'retl_cnt_lake').fillna(0) - _to_num_315(cmp_before_315, 'retl_cnt_excel').fillna(0)).abs()
    if 'term_cnt_abs' not in cmp_before_315.columns:
        cmp_before_315['term_cnt_abs'] = (_to_num_315(cmp_before_315, 'term_cnt_lake').fillna(0) - _to_num_315(cmp_before_315, 'term_cnt_excel').fillna(0)).abs()

    excel_keys_315 = ex_retl_term_315_df[['inn_key', 'agr_id_key']].drop_duplicates()
    old_keys_315 = cmp_before_315[['inn_key', 'agr_id_key']].drop_duplicates()
    new_keys_315 = cmp_547_retl_term_df[['inn_key', 'agr_id_key']].drop_duplicates()

    covered_old_315 = len(excel_keys_315.merge(old_keys_315, on=['inn_key', 'agr_id_key'], how='inner'))
    covered_new_315 = len(excel_keys_315.merge(new_keys_315, on=['inn_key', 'agr_id_key'], how='inner'))

    kpi_547_vs_old_df = pd.DataFrame([
        {
            'metric': 'excel_key_count',
            'before_old_logic': len(excel_keys_315),
            'after_547_logic': len(excel_keys_315),
            'delta': 0,
        },
        {
            'metric': 'covered_excel_keys',
            'before_old_logic': covered_old_315,
            'after_547_logic': covered_new_315,
            'delta': covered_new_315 - covered_old_315,
        },
        {
            'metric': 'rows_compared',
            'before_old_logic': len(cmp_before_315),
            'after_547_logic': len(cmp_547_retl_term_df),
            'delta': len(cmp_547_retl_term_df) - len(cmp_before_315),
        },
        {
            'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_315['retl_cnt_abs'], tol=0.0),
            'after_547_logic': exact_match_rate_from_abs(cmp_547_retl_term_df['retl_cnt_abs_547'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_547_retl_term_df['retl_cnt_abs_547'], tol=0.0) - exact_match_rate_from_abs(cmp_before_315['retl_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'term_cnt_exact_match_rate_pct_tol_0',
            'before_old_logic': exact_match_rate_from_abs(cmp_before_315['term_cnt_abs'], tol=0.0),
            'after_547_logic': exact_match_rate_from_abs(cmp_547_retl_term_df['term_cnt_abs_547'], tol=0.0),
            'delta': exact_match_rate_from_abs(cmp_547_retl_term_df['term_cnt_abs_547'], tol=0.0) - exact_match_rate_from_abs(cmp_before_315['term_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'retl_cnt_mae',
            'before_old_logic': float(_to_num_315(cmp_before_315, 'retl_cnt_abs').mean()) if len(cmp_before_315) else 0.0,
            'after_547_logic': float(_to_num_315(cmp_547_retl_term_df, 'retl_cnt_abs_547').mean()) if len(cmp_547_retl_term_df) else 0.0,
            'delta': (float(_to_num_315(cmp_547_retl_term_df, 'retl_cnt_abs_547').mean()) if len(cmp_547_retl_term_df) else 0.0) - (float(_to_num_315(cmp_before_315, 'retl_cnt_abs').mean()) if len(cmp_before_315) else 0.0),
        },
        {
            'metric': 'term_cnt_mae',
            'before_old_logic': float(_to_num_315(cmp_before_315, 'term_cnt_abs').mean()) if len(cmp_before_315) else 0.0,
            'after_547_logic': float(_to_num_315(cmp_547_retl_term_df, 'term_cnt_abs_547').mean()) if len(cmp_547_retl_term_df) else 0.0,
            'delta': (float(_to_num_315(cmp_547_retl_term_df, 'term_cnt_abs_547').mean()) if len(cmp_547_retl_term_df) else 0.0) - (float(_to_num_315(cmp_before_315, 'term_cnt_abs').mean()) if len(cmp_before_315) else 0.0),
        },
    ])

    divergence_rows_315 = []
    for metric_name in ['retl_cnt', 'term_cnt']:
        old_lake_total = _to_num_315(cmp_before_315, f'{metric_name}_lake').fillna(0).sum()
        old_excel_total = _to_num_315(cmp_before_315, f'{metric_name}_excel').fillna(0).sum()

        new_lake_total = _to_num_315(cmp_547_retl_term_df, f'{metric_name}_lake_547').fillna(0).sum()
        new_excel_total = _to_num_315(cmp_547_retl_term_df, f'{metric_name}_excel').fillna(0).sum()

        old_pct = _pct_div_315(old_lake_total, old_excel_total)
        new_pct = _pct_div_315(new_lake_total, new_excel_total)

        divergence_rows_315.append({
            'metric': metric_name,
            'old_lake_total': old_lake_total,
            'old_excel_total': old_excel_total,
            'old_divergence_pct': old_pct,
            'old_abs_divergence_pct': abs(old_pct) if old_pct is not None else None,
            'new_lake_total': new_lake_total,
            'new_excel_total': new_excel_total,
            'new_divergence_pct': new_pct,
            'new_abs_divergence_pct': abs(new_pct) if new_pct is not None else None,
            'abs_divergence_improvement_pct_points': (
                abs(old_pct) - abs(new_pct)
                if old_pct is not None and new_pct is not None
                else None
            ),
        })

    divergence_547_vs_old_df = pd.DataFrame(divergence_rows_315)

    # Focus INN analysis (7106527696)
    focus_inn = '7106527696'

    focus_excel_315 = ex_retl_term_315_df[ex_retl_term_315_df['inn_key'] == focus_inn].copy()
    focus_old_315 = (
        cmp_before_315[cmp_before_315['inn_key'] == focus_inn][['inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake']]
        .drop_duplicates(subset=['inn_key', 'agr_id_key'])
        .rename(columns={'retl_cnt_lake': 'retl_cnt_lake_old', 'term_cnt_lake': 'term_cnt_lake_old'})
    )
    focus_new_315 = (
        cmp_547_retl_term_df[cmp_547_retl_term_df['inn_key'] == focus_inn][['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547']]
        .drop_duplicates(subset=['inn_key', 'agr_id_key'])
        .rename(columns={'retl_cnt_lake_547': 'retl_cnt_lake_new_547', 'term_cnt_lake_547': 'term_cnt_lake_new_547'})
    )

    focus_7106527696_547_df = (
        focus_excel_315
        .merge(focus_old_315, on=['inn_key', 'agr_id_key'], how='left')
        .merge(focus_new_315, on=['inn_key', 'agr_id_key'], how='left')
    )

    if len(focus_7106527696_547_df):
        focus_7106527696_547_df['retl_delta_old'] = _to_num_315(focus_7106527696_547_df, 'retl_cnt_lake_old').fillna(0) - _to_num_315(focus_7106527696_547_df, 'retl_cnt_excel').fillna(0)
        focus_7106527696_547_df['retl_delta_new_547'] = _to_num_315(focus_7106527696_547_df, 'retl_cnt_lake_new_547').fillna(0) - _to_num_315(focus_7106527696_547_df, 'retl_cnt_excel').fillna(0)
        focus_7106527696_547_df['term_delta_old'] = _to_num_315(focus_7106527696_547_df, 'term_cnt_lake_old').fillna(0) - _to_num_315(focus_7106527696_547_df, 'term_cnt_excel').fillna(0)
        focus_7106527696_547_df['term_delta_new_547'] = _to_num_315(focus_7106527696_547_df, 'term_cnt_lake_new_547').fillna(0) - _to_num_315(focus_7106527696_547_df, 'term_cnt_excel').fillna(0)

    focus_7106527696_nmrc_547_df = counts_547_df[
        counts_547_df['inn_key'] == focus_inn
    ][['agr_id', 'n_agr', 'n_cmp_client', 'inn', 'retl_cnt_547', 'term_cnt_547', 'retl_cnt_rn1_hint']].copy() if len(counts_547_df) else pd.DataFrame()

    print('3.15 note: used m.acq_class <> SV filter =', used_m_acq_filter_315)

print('KPI: old vs 547-like logic')
display(kpi_547_vs_old_df)

print('\nПроцент расхождения Lake vs Excel (old vs 547-like):')
display(divergence_547_vs_old_df)

print('\nFocus INN 7106527696: Excel vs old vs 547-like')
display(focus_7106527696_547_df)

print('\nFocus INN 7106527696: contract-level 547-like counters')
display(focus_7106527696_nmrc_547_df)

if len(cmp_547_retl_term_df):
    print('\nTOP-20 retl abs delta (547-like):')
    display(
        cmp_547_retl_term_df.sort_values('retl_cnt_abs_547', ascending=False)[
            ['agr_id_key', 'inn_key', 'retl_cnt_lake_547', 'retl_cnt_excel', 'retl_cnt_delta_547', 'retl_cnt_abs_547']
        ].head(20)
    )

    print('TOP-20 term abs delta (547-like):')
    display(
        cmp_547_retl_term_df.sort_values('term_cnt_abs_547', ascending=False)[
            ['agr_id_key', 'inn_key', 'term_cnt_lake_547', 'term_cnt_excel', 'term_cnt_delta_547', 'term_cnt_abs_547']
        ].head(20)
    )

### 3.16 Разбор выпавших ключей (`old covered` -> `new missing`) + KPI на общем множестве ключей (`old ∩ new`)

Что делает секция:
- выделяет ключи `(inn_key, agr_id_key)`, которые были покрыты old-логикой, но пропали в `547-like`;
- раскладывает эти ключи по вероятным причинам (нет AGR_TERMS, все `REZERVNYI`, `SV`, нет активного `C_POS_SERIAL` и т.д.);
- считает KPI old vs 547-like **только на общем множестве ключей**, чтобы убрать эффект изменения coverage.

In [ ]:
# 3.16 Coverage diagnostics + common-key KPI (old vs 547-like)
required_316 = [
    'cmp_before_315',
    'cmp_547_retl_term_df',
    'ex_retl_term_315_df',
    'counts_547_df',
]
missing_316 = [x for x in required_316 if x not in globals()]
if missing_316:
    raise RuntimeError(f"Сначала выполните 3.15. Не найдены объекты: {missing_316}")


def _clean_keys_316(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in_316(values):
    vals = _clean_keys_316(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def _num_316(df, col):
    if col not in df.columns:
        return pd.Series(dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


def _safe_pct_316(v):
    if pd.isna(v):
        return None
    return float(v)


def _div_pct_316(lake_total, excel_total):
    if pd.isna(excel_total) or float(excel_total) == 0.0:
        return None
    return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0


excel_keys_316 = ex_retl_term_315_df[['inn_key', 'agr_id_key']].drop_duplicates().copy()
old_keys_316 = cmp_before_315[['inn_key', 'agr_id_key']].drop_duplicates().copy()
new_keys_316 = cmp_547_retl_term_df[['inn_key', 'agr_id_key']].drop_duplicates().copy()

old_covered_316 = excel_keys_316.merge(old_keys_316, on=['inn_key', 'agr_id_key'], how='inner')
new_covered_316 = excel_keys_316.merge(new_keys_316, on=['inn_key', 'agr_id_key'], how='inner')
common_covered_316 = old_covered_316.merge(new_covered_316, on=['inn_key', 'agr_id_key'], how='inner')

dropped_old_to_new_316 = (
    old_covered_316
    .merge(new_covered_316, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

added_new_vs_old_316 = (
    new_covered_316
    .merge(old_covered_316, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

coverage_316_df = pd.DataFrame([
    {
        'excel_key_count': len(excel_keys_316),
        'old_covered_keys': len(old_covered_316),
        'new_covered_keys': len(new_covered_316),
        'common_covered_keys': len(common_covered_316),
        'dropped_old_to_new': len(dropped_old_to_new_316),
        'added_new_vs_old': len(added_new_vs_old_316),
    }
])

# --- Reasons for dropped keys ---
dropped_reasons_316_df = pd.DataFrame()
reason_summary_316_df = pd.DataFrame(columns=['drop_reason', 'keys_cnt'])

if len(dropped_old_to_new_316):
    dropped_agr_ids_316 = _clean_keys_316(dropped_old_to_new_316['agr_id_key'].tolist())
    dropped_agr_in_316 = _sql_in_316(dropped_agr_ids_316)

    sql_drop_diag_316 = f"""
    with focus_agr as (
      select distinct
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
      from ods_alpha.scd1_agreements a
      join ods_alpha.scd1_companies c
        on c.n_cmp = a.n_cmp_client
      where cast(a.abs_agr_id as string) in ({dropped_agr_in_316})
        and upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
        and coalesce(c.ods_deleted_flg, '0') <> '1'
    ),
    terms as (
      select distinct
        fa.agr_id,
        fa.n_agr,
        fa.n_cmp_client,
        fa.inn,
        cast(t.c_nmrc as string) as c_nmrc
      from focus_agr fa
      left join ods_alpha.scd1_agr_terms t
        on cast(t.n_agr as string) = fa.n_agr
       and t.c_nmrc is not null
       and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
       and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
       and coalesce(t.ods_deleted_flg, '0') <> '1'
    ),
    merch as (
      select distinct
        t.agr_id,
        t.n_agr,
        t.inn,
        t.c_nmrc,
        upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) as mrc_name_up,
        upper(coalesce(trim(cast(m.acq_class as string)), 'NA')) as m_acq_class
      from terms t
      left join ods_alpha.scd1_merchants m
        on cast(m.c_nmrc as string) = t.c_nmrc
       and coalesce(m.ods_deleted_flg, '0') <> '1'
    ),
    pos_period as (
      select distinct
        cast(p.c_nmrc as string) as c_nmrc,
        cast(p.c_pos_serial as string) as c_pos_serial
      from ods_alpha.scd1_pos_terminals p
      where p.c_pos_serial is not null
        and coalesce(cast(p.d_ter_install as date), cast('1900-01-01' as date)) < date_add(cast('{month_end}' as date), 1)
        and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
        and coalesce(p.ods_deleted_flg, '0') <> '1'
    )
    select
      fa.agr_id,
      max(fa.n_agr) as n_agr_any,
      max(fa.inn) as inn,
      count(distinct t.c_nmrc) as terms_nmrc_cnt,
      count(distinct merch.c_nmrc) as merch_join_nmrc_cnt,
      count(distinct case when merch.mrc_name_up not like 'REZERVNYI TERMINAL%' then merch.c_nmrc else null end) as non_reserve_nmrc_cnt,
      count(distinct case when merch.mrc_name_up not like 'REZERVNYI TERMINAL%' and merch.m_acq_class <> 'SV' then merch.c_nmrc else null end) as non_reserve_non_sv_nmrc_cnt,
      count(distinct case when merch.mrc_name_up not like 'REZERVNYI TERMINAL%' and merch.m_acq_class <> 'SV' then pos_period.c_pos_serial else null end) as pos_serial_after_filters_cnt
    from focus_agr fa
    left join terms t
      on t.agr_id = fa.agr_id
    left join merch
      on merch.agr_id = fa.agr_id
     and merch.c_nmrc = t.c_nmrc
    left join pos_period
      on pos_period.c_nmrc = merch.c_nmrc
    group by fa.agr_id
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        dropped_diag_sql_316_df = imp.fetch(sql_drop_diag_316)

    if dropped_diag_sql_316_df is None:
        dropped_diag_sql_316_df = pd.DataFrame(columns=[
            'agr_id', 'n_agr_any', 'inn', 'terms_nmrc_cnt', 'merch_join_nmrc_cnt',
            'non_reserve_nmrc_cnt', 'non_reserve_non_sv_nmrc_cnt', 'pos_serial_after_filters_cnt'
        ])

    # Guard: even on empty result we still need merge keys.
    for _req_col in ['agr_id_key', 'inn_key_sql']:
        if _req_col not in dropped_diag_sql_316_df.columns:
            dropped_diag_sql_316_df[_req_col] = None

    if len(dropped_diag_sql_316_df):
        dropped_diag_sql_316_df['agr_id_key'] = dropped_diag_sql_316_df['agr_id'].apply(normalize_agr_q1)
        dropped_diag_sql_316_df['inn_key_sql'] = dropped_diag_sql_316_df['inn'].apply(normalize_inn_q1)

    dropped_reasons_316_df = dropped_old_to_new_316.merge(
        dropped_diag_sql_316_df,
        on='agr_id_key',
        how='left'
    )

    dropped_reasons_316_df = dropped_reasons_316_df.merge(
        cmp_before_315[['inn_key', 'agr_id_key', 'retl_cnt_lake', 'retl_cnt_excel', 'term_cnt_lake', 'term_cnt_excel']].drop_duplicates(),
        on=['inn_key', 'agr_id_key'],
        how='left'
    )

    for c in ['terms_nmrc_cnt', 'merch_join_nmrc_cnt', 'non_reserve_nmrc_cnt', 'non_reserve_non_sv_nmrc_cnt', 'pos_serial_after_filters_cnt']:
        if c in dropped_reasons_316_df.columns:
            dropped_reasons_316_df[c] = pd.to_numeric(dropped_reasons_316_df[c], errors='coerce').fillna(0)

    def _reason_316(row):
        if pd.isna(row.get('n_agr_any')):
            return 'agr_not_found_in_focus_sa'
        if row.get('terms_nmrc_cnt', 0) <= 0:
            return 'no_agr_terms_in_period'
        if row.get('merch_join_nmrc_cnt', 0) <= 0:
            return 'no_merchants_for_agr_terms'
        if row.get('non_reserve_nmrc_cnt', 0) <= 0:
            return 'all_nmrc_filtered_as_reserve'
        if row.get('non_reserve_non_sv_nmrc_cnt', 0) <= 0:
            return 'all_nmrc_filtered_as_sv'
        if row.get('pos_serial_after_filters_cnt', 0) <= 0:
            return 'no_active_pos_serial_after_filters'
        return 'other_or_key_mismatch'

    dropped_reasons_316_df['drop_reason'] = dropped_reasons_316_df.apply(_reason_316, axis=1)

    reason_summary_316_df = (
        dropped_reasons_316_df.groupby('drop_reason', as_index=False)
        .agg(keys_cnt=('agr_id_key', 'size'))
        .sort_values('keys_cnt', ascending=False)
        .reset_index(drop=True)
    )

# --- KPI on common key set only (remove coverage effect) ---
common_pair_316_df = common_covered_316.copy()

old_part_316 = cmp_before_315[['inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake', 'retl_cnt_excel', 'term_cnt_excel']].drop_duplicates()
new_part_316 = cmp_547_retl_term_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547']].drop_duplicates()

common_pair_316_df = (
    common_pair_316_df
    .merge(old_part_316, on=['inn_key', 'agr_id_key'], how='left')
    .merge(new_part_316, on=['inn_key', 'agr_id_key'], how='left')
)

for m in ['retl_cnt', 'term_cnt']:
    common_pair_316_df[f'{m}_delta_old_common'] = _num_316(common_pair_316_df, f'{m}_lake').fillna(0) - _num_316(common_pair_316_df, f'{m}_excel').fillna(0)
    common_pair_316_df[f'{m}_abs_old_common'] = _num_316(common_pair_316_df, f'{m}_delta_old_common').abs()

    common_pair_316_df[f'{m}_delta_new_common'] = _num_316(common_pair_316_df, f'{m}_lake_547').fillna(0) - _num_316(common_pair_316_df, f'{m}_excel').fillna(0)
    common_pair_316_df[f'{m}_abs_new_common'] = _num_316(common_pair_316_df, f'{m}_delta_new_common').abs()

kpi_common_keys_316_df = pd.DataFrame([
    {
        'metric': 'common_keys_count',
        'old_on_common': len(common_pair_316_df),
        'new_on_common': len(common_pair_316_df),
        'delta': 0,
    },
    {
        'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
        'old_on_common': exact_match_rate_from_abs(common_pair_316_df['retl_cnt_abs_old_common'], tol=0.0),
        'new_on_common': exact_match_rate_from_abs(common_pair_316_df['retl_cnt_abs_new_common'], tol=0.0),
        'delta': exact_match_rate_from_abs(common_pair_316_df['retl_cnt_abs_new_common'], tol=0.0) - exact_match_rate_from_abs(common_pair_316_df['retl_cnt_abs_old_common'], tol=0.0),
    },
    {
        'metric': 'term_cnt_exact_match_rate_pct_tol_0',
        'old_on_common': exact_match_rate_from_abs(common_pair_316_df['term_cnt_abs_old_common'], tol=0.0),
        'new_on_common': exact_match_rate_from_abs(common_pair_316_df['term_cnt_abs_new_common'], tol=0.0),
        'delta': exact_match_rate_from_abs(common_pair_316_df['term_cnt_abs_new_common'], tol=0.0) - exact_match_rate_from_abs(common_pair_316_df['term_cnt_abs_old_common'], tol=0.0),
    },
    {
        'metric': 'retl_cnt_mae',
        'old_on_common': float(_num_316(common_pair_316_df, 'retl_cnt_abs_old_common').mean()) if len(common_pair_316_df) else 0.0,
        'new_on_common': float(_num_316(common_pair_316_df, 'retl_cnt_abs_new_common').mean()) if len(common_pair_316_df) else 0.0,
        'delta': (float(_num_316(common_pair_316_df, 'retl_cnt_abs_new_common').mean()) if len(common_pair_316_df) else 0.0) - (float(_num_316(common_pair_316_df, 'retl_cnt_abs_old_common').mean()) if len(common_pair_316_df) else 0.0),
    },
    {
        'metric': 'term_cnt_mae',
        'old_on_common': float(_num_316(common_pair_316_df, 'term_cnt_abs_old_common').mean()) if len(common_pair_316_df) else 0.0,
        'new_on_common': float(_num_316(common_pair_316_df, 'term_cnt_abs_new_common').mean()) if len(common_pair_316_df) else 0.0,
        'delta': (float(_num_316(common_pair_316_df, 'term_cnt_abs_new_common').mean()) if len(common_pair_316_df) else 0.0) - (float(_num_316(common_pair_316_df, 'term_cnt_abs_old_common').mean()) if len(common_pair_316_df) else 0.0),
    },
])

divergence_common_316_rows = []
for metric_name in ['retl_cnt', 'term_cnt']:
    old_lake_total = _num_316(common_pair_316_df, f'{metric_name}_lake').fillna(0).sum()
    new_lake_total = _num_316(common_pair_316_df, f'{metric_name}_lake_547').fillna(0).sum()
    excel_total = _num_316(common_pair_316_df, f'{metric_name}_excel').fillna(0).sum()

    old_pct = _div_pct_316(old_lake_total, excel_total)
    new_pct = _div_pct_316(new_lake_total, excel_total)

    divergence_common_316_rows.append({
        'metric': metric_name,
        'old_lake_total_common': old_lake_total,
        'new_lake_total_common': new_lake_total,
        'excel_total_common': excel_total,
        'old_divergence_pct_common': _safe_pct_316(old_pct),
        'new_divergence_pct_common': _safe_pct_316(new_pct),
        'old_abs_divergence_pct_common': abs(old_pct) if old_pct is not None else None,
        'new_abs_divergence_pct_common': abs(new_pct) if new_pct is not None else None,
        'abs_divergence_improvement_pct_points_common': (
            abs(old_pct) - abs(new_pct)
            if old_pct is not None and new_pct is not None
            else None
        ),
    })

divergence_common_316_df = pd.DataFrame(divergence_common_316_rows)

print('Coverage summary (3.16):')
display(coverage_316_df)

print('\nDropped keys reasons summary (old covered -> new missing):')
display(reason_summary_316_df)

if len(dropped_reasons_316_df):
    print('Dropped keys details (top-200):')
    display_cols_316 = [
        'inn_key', 'agr_id_key', 'drop_reason',
        'n_agr_any', 'terms_nmrc_cnt', 'merch_join_nmrc_cnt',
        'non_reserve_nmrc_cnt', 'non_reserve_non_sv_nmrc_cnt', 'pos_serial_after_filters_cnt',
        'retl_cnt_lake', 'retl_cnt_excel', 'term_cnt_lake', 'term_cnt_excel'
    ]
    display_cols_316 = [c for c in display_cols_316 if c in dropped_reasons_316_df.columns]
    display(dropped_reasons_316_df[display_cols_316].head(200))

print('\nKPI on common keys only (old ∩ new):')
display(kpi_common_keys_316_df)

print('Divergence on common keys only:')
display(divergence_common_316_df)

### 3.17 Hybrid: `547-like` + fallback только для missing-ключей (`old/r2`) и проверка KPI

Задача секции:
- не ломать ядро `547-like` логики;
- вернуть покрытие ключей, выпавших из `547-like`, через точечный patch (только для ключей `excel ∖ 547-like`);
- сравнить `547-like` vs `hybrid` по KPI и отдельно показать, сколько ключей вернулось именно через `r2_fallback` (если доступен `agr_id_source`).

In [ ]:
# 3.17 Hybrid check: 547-like core + fallback patch for missing keys only
required_317 = ['cmp_before_315', 'cmp_547_retl_term_df', 'ex_retl_term_315_df']
missing_317 = [x for x in required_317 if x not in globals()]
if missing_317:
    raise RuntimeError(f"Сначала выполните 3.15. Не найдены объекты: {missing_317}")


def _num_317(df, col):
    if col not in df.columns:
        return pd.Series(dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


def _pct_div_317(lake_total, excel_total):
    if pd.isna(excel_total) or float(excel_total) == 0.0:
        return None
    return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0


# 1) Key-level inputs
excel_ref_317_df = (
    ex_retl_term_315_df[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
    .dropna(subset=['inn_key', 'agr_id_key'])
    .drop_duplicates(subset=['inn_key', 'agr_id_key'])
    .copy()
)

old_key_metrics_317_df = (
    cmp_before_315[['inn_key', 'agr_id_key', 'retl_cnt_lake', 'term_cnt_lake']]
    .dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg({'retl_cnt_lake': 'max', 'term_cnt_lake': 'max'})
    .rename(columns={
        'retl_cnt_lake': 'retl_cnt_lake_fallback',
        'term_cnt_lake': 'term_cnt_lake_fallback',
    })
)

like547_key_metrics_317_df = (
    cmp_547_retl_term_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547']]
    .dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg({'retl_cnt_lake_547': 'max', 'term_cnt_lake_547': 'max'})
)

# 2) Missing keys vs 547-like and patchability by old logic
excel_keys_317_df = excel_ref_317_df[['inn_key', 'agr_id_key']].drop_duplicates()
keys_547_317_df = like547_key_metrics_317_df[['inn_key', 'agr_id_key']].drop_duplicates()

missing_vs_547_317_df = (
    excel_keys_317_df
    .merge(keys_547_317_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

patchable_by_old_317_df = missing_vs_547_317_df.merge(
    old_key_metrics_317_df,
    on=['inn_key', 'agr_id_key'],
    how='inner'
)

unpatchable_317_df = (
    missing_vs_547_317_df
    .merge(old_key_metrics_317_df[['inn_key', 'agr_id_key']], on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

# Optional: detect whether fallback patch keys are from r2_fallback (if base_df is available)
if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df) and 'agr_id_source' in base_df.columns:
    r2_probe_317_df = base_df.copy()
    if 'inn' in r2_probe_317_df.columns and 'agr_id' in r2_probe_317_df.columns:
        r2_probe_317_df['inn_key'] = r2_probe_317_df['inn'].apply(normalize_inn_q1)
        r2_probe_317_df['agr_id_key'] = r2_probe_317_df['agr_id'].apply(normalize_agr_q1)
        r2_probe_317_df['is_r2_fallback_key'] = (r2_probe_317_df['agr_id_source'].astype(str) == 'r2_fallback').astype(int)
        r2_probe_317_df = (
            r2_probe_317_df
            .dropna(subset=['inn_key', 'agr_id_key'])
            .groupby(['inn_key', 'agr_id_key'], as_index=False)
            .agg({'is_r2_fallback_key': 'max'})
        )
        patchable_by_old_317_df = patchable_by_old_317_df.merge(
            r2_probe_317_df,
            on=['inn_key', 'agr_id_key'],
            how='left'
        )
        patchable_by_old_317_df['is_r2_fallback_key'] = patchable_by_old_317_df['is_r2_fallback_key'].fillna(0).astype(int)
    else:
        patchable_by_old_317_df['is_r2_fallback_key'] = 0
else:
    patchable_by_old_317_df['is_r2_fallback_key'] = 0

patchable_by_old_317_df['fallback_source_type'] = patchable_by_old_317_df['is_r2_fallback_key'].map({1: 'r2_fallback', 0: 'old_logic_non_r2'})

# 3) Build hybrid key metrics: keep 547-like core + patch only missing keys
hybrid_core_317_df = like547_key_metrics_317_df.copy()
hybrid_core_317_df = hybrid_core_317_df.rename(columns={
    'retl_cnt_lake_547': 'retl_cnt_lake_hybrid',
    'term_cnt_lake_547': 'term_cnt_lake_hybrid',
})
hybrid_core_317_df['calc_source'] = '547_like'
hybrid_core_317_df['fallback_source_type'] = None

hybrid_patch_317_df = patchable_by_old_317_df.copy()
hybrid_patch_317_df = hybrid_patch_317_df.rename(columns={
    'retl_cnt_lake_fallback': 'retl_cnt_lake_hybrid',
    'term_cnt_lake_fallback': 'term_cnt_lake_hybrid',
})
hybrid_patch_317_df['calc_source'] = 'fallback_patch'

hybrid_key_metrics_317_df = pd.concat([
    hybrid_core_317_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_hybrid', 'term_cnt_lake_hybrid', 'calc_source', 'fallback_source_type']],
    hybrid_patch_317_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_hybrid', 'term_cnt_lake_hybrid', 'calc_source', 'fallback_source_type']],
], ignore_index=True)

# just in case: one row per key
hybrid_key_metrics_317_df = (
    hybrid_key_metrics_317_df
    .sort_values(['calc_source'], ascending=True)
    .drop_duplicates(subset=['inn_key', 'agr_id_key'], keep='first')
)

# 4) Compare 547-like vs hybrid against Excel
cmp_hybrid_retl_term_317_df = excel_ref_317_df.merge(
    hybrid_key_metrics_317_df,
    on=['inn_key', 'agr_id_key'],
    how='inner'
)

for m in ['retl_cnt', 'term_cnt']:
    cmp_hybrid_retl_term_317_df[f'{m}_delta_hybrid'] = _num_317(cmp_hybrid_retl_term_317_df, f'{m}_lake_hybrid').fillna(0) - _num_317(cmp_hybrid_retl_term_317_df, f'{m}_excel').fillna(0)
    cmp_hybrid_retl_term_317_df[f'{m}_abs_hybrid'] = _num_317(cmp_hybrid_retl_term_317_df, f'{m}_delta_hybrid').abs()

# 547-like baseline frame for this section
cmp_547_base_317_df = excel_ref_317_df.merge(
    like547_key_metrics_317_df,
    on=['inn_key', 'agr_id_key'],
    how='inner'
)
for m in ['retl_cnt', 'term_cnt']:
    cmp_547_base_317_df[f'{m}_delta_547'] = _num_317(cmp_547_base_317_df, f'{m}_lake_547').fillna(0) - _num_317(cmp_547_base_317_df, f'{m}_excel').fillna(0)
    cmp_547_base_317_df[f'{m}_abs_547'] = _num_317(cmp_547_base_317_df, f'{m}_delta_547').abs()

coverage_hybrid_317_df = pd.DataFrame([
    {
        'excel_key_count': len(excel_keys_317_df),
        'covered_547_like': len(cmp_547_base_317_df),
        'missing_vs_547_like': len(missing_vs_547_317_df),
        'patchable_by_old': len(patchable_by_old_317_df),
        'unpatchable_after_old': len(unpatchable_317_df),
        'covered_hybrid': len(cmp_hybrid_retl_term_317_df),
        'added_coverage_vs_547_like': len(cmp_hybrid_retl_term_317_df) - len(cmp_547_base_317_df),
        'patched_from_r2_fallback': int((patchable_by_old_317_df['is_r2_fallback_key'] == 1).sum()) if len(patchable_by_old_317_df) else 0,
        'patched_from_old_non_r2': int((patchable_by_old_317_df['is_r2_fallback_key'] == 0).sum()) if len(patchable_by_old_317_df) else 0,
    }
])

kpi_547_vs_hybrid_317_df = pd.DataFrame([
    {
        'metric': 'rows_compared',
        '547_like': len(cmp_547_base_317_df),
        'hybrid': len(cmp_hybrid_retl_term_317_df),
        'delta_hybrid_minus_547': len(cmp_hybrid_retl_term_317_df) - len(cmp_547_base_317_df),
    },
    {
        'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
        '547_like': exact_match_rate_from_abs(cmp_547_base_317_df['retl_cnt_abs_547'], tol=0.0),
        'hybrid': exact_match_rate_from_abs(cmp_hybrid_retl_term_317_df['retl_cnt_abs_hybrid'], tol=0.0),
        'delta_hybrid_minus_547': exact_match_rate_from_abs(cmp_hybrid_retl_term_317_df['retl_cnt_abs_hybrid'], tol=0.0) - exact_match_rate_from_abs(cmp_547_base_317_df['retl_cnt_abs_547'], tol=0.0),
    },
    {
        'metric': 'term_cnt_exact_match_rate_pct_tol_0',
        '547_like': exact_match_rate_from_abs(cmp_547_base_317_df['term_cnt_abs_547'], tol=0.0),
        'hybrid': exact_match_rate_from_abs(cmp_hybrid_retl_term_317_df['term_cnt_abs_hybrid'], tol=0.0),
        'delta_hybrid_minus_547': exact_match_rate_from_abs(cmp_hybrid_retl_term_317_df['term_cnt_abs_hybrid'], tol=0.0) - exact_match_rate_from_abs(cmp_547_base_317_df['term_cnt_abs_547'], tol=0.0),
    },
    {
        'metric': 'retl_cnt_mae',
        '547_like': float(_num_317(cmp_547_base_317_df, 'retl_cnt_abs_547').mean()) if len(cmp_547_base_317_df) else 0.0,
        'hybrid': float(_num_317(cmp_hybrid_retl_term_317_df, 'retl_cnt_abs_hybrid').mean()) if len(cmp_hybrid_retl_term_317_df) else 0.0,
        'delta_hybrid_minus_547': (float(_num_317(cmp_hybrid_retl_term_317_df, 'retl_cnt_abs_hybrid').mean()) if len(cmp_hybrid_retl_term_317_df) else 0.0) - (float(_num_317(cmp_547_base_317_df, 'retl_cnt_abs_547').mean()) if len(cmp_547_base_317_df) else 0.0),
    },
    {
        'metric': 'term_cnt_mae',
        '547_like': float(_num_317(cmp_547_base_317_df, 'term_cnt_abs_547').mean()) if len(cmp_547_base_317_df) else 0.0,
        'hybrid': float(_num_317(cmp_hybrid_retl_term_317_df, 'term_cnt_abs_hybrid').mean()) if len(cmp_hybrid_retl_term_317_df) else 0.0,
        'delta_hybrid_minus_547': (float(_num_317(cmp_hybrid_retl_term_317_df, 'term_cnt_abs_hybrid').mean()) if len(cmp_hybrid_retl_term_317_df) else 0.0) - (float(_num_317(cmp_547_base_317_df, 'term_cnt_abs_547').mean()) if len(cmp_547_base_317_df) else 0.0),
    },
])

divergence_547_vs_hybrid_317_rows = []
for metric_name in ['retl_cnt', 'term_cnt']:
    lake_547_total = _num_317(cmp_547_base_317_df, f'{metric_name}_lake_547').fillna(0).sum()
    excel_547_total = _num_317(cmp_547_base_317_df, f'{metric_name}_excel').fillna(0).sum()

    lake_hybrid_total = _num_317(cmp_hybrid_retl_term_317_df, f'{metric_name}_lake_hybrid').fillna(0).sum()
    excel_hybrid_total = _num_317(cmp_hybrid_retl_term_317_df, f'{metric_name}_excel').fillna(0).sum()

    pct_547 = _pct_div_317(lake_547_total, excel_547_total)
    pct_hybrid = _pct_div_317(lake_hybrid_total, excel_hybrid_total)

    divergence_547_vs_hybrid_317_rows.append({
        'metric': metric_name,
        'lake_547_total': lake_547_total,
        'excel_547_total': excel_547_total,
        'divergence_pct_547': pct_547,
        'lake_hybrid_total': lake_hybrid_total,
        'excel_hybrid_total': excel_hybrid_total,
        'divergence_pct_hybrid': pct_hybrid,
        'abs_divergence_improvement_pct_points': (
            abs(pct_547) - abs(pct_hybrid)
            if pct_547 is not None and pct_hybrid is not None
            else None
        ),
    })


divergence_547_vs_hybrid_317_df = pd.DataFrame(divergence_547_vs_hybrid_317_rows)

# 5) Focus INN 7106527696
focus_inn_317 = '7106527696'
focus_excel_317_df = excel_ref_317_df[excel_ref_317_df['inn_key'] == focus_inn_317].copy()
focus_547_317_df = cmp_547_base_317_df[cmp_547_base_317_df['inn_key'] == focus_inn_317][
    ['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547']
].drop_duplicates()
focus_hybrid_317_df = cmp_hybrid_retl_term_317_df[cmp_hybrid_retl_term_317_df['inn_key'] == focus_inn_317][
    ['inn_key', 'agr_id_key', 'retl_cnt_lake_hybrid', 'term_cnt_lake_hybrid', 'calc_source', 'fallback_source_type']
].drop_duplicates()

focus_7106527696_hybrid_317_df = (
    focus_excel_317_df
    .merge(focus_547_317_df, on=['inn_key', 'agr_id_key'], how='left')
    .merge(focus_hybrid_317_df, on=['inn_key', 'agr_id_key'], how='left')
)

if len(focus_7106527696_hybrid_317_df):
    focus_7106527696_hybrid_317_df['retl_delta_547'] = _num_317(focus_7106527696_hybrid_317_df, 'retl_cnt_lake_547').fillna(0) - _num_317(focus_7106527696_hybrid_317_df, 'retl_cnt_excel').fillna(0)
    focus_7106527696_hybrid_317_df['retl_delta_hybrid'] = _num_317(focus_7106527696_hybrid_317_df, 'retl_cnt_lake_hybrid').fillna(0) - _num_317(focus_7106527696_hybrid_317_df, 'retl_cnt_excel').fillna(0)
    focus_7106527696_hybrid_317_df['term_delta_547'] = _num_317(focus_7106527696_hybrid_317_df, 'term_cnt_lake_547').fillna(0) - _num_317(focus_7106527696_hybrid_317_df, 'term_cnt_excel').fillna(0)
    focus_7106527696_hybrid_317_df['term_delta_hybrid'] = _num_317(focus_7106527696_hybrid_317_df, 'term_cnt_lake_hybrid').fillna(0) - _num_317(focus_7106527696_hybrid_317_df, 'term_cnt_excel').fillna(0)

print('Coverage summary (3.17):')
display(coverage_hybrid_317_df)

print('\nKPI: 547-like vs hybrid (missing-key patch):')
display(kpi_547_vs_hybrid_317_df)

print('Divergence: 547-like vs hybrid:')
display(divergence_547_vs_hybrid_317_df)

print('\nPatch keys by source type (top-200):')
if len(hybrid_patch_317_df):
    display(hybrid_patch_317_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_hybrid', 'term_cnt_lake_hybrid', 'fallback_source_type']].head(200))
else:
    print('No patch keys (hybrid == 547-like)')

print(f'\nFocus INN {focus_inn_317}: Excel vs 547-like vs hybrid')
display(focus_7106527696_hybrid_317_df)

### 3.18 Трехрежимное сравнение: `547_like` vs `hybrid_all` vs `hybrid_non_r2_only`

Цель:
- оценить, сколько покрытия и качества дает гибрид;
- отдельно проверить вариант без `r2_fallback`-патча (`hybrid_non_r2_only`), чтобы понять вклад именно `r2` ключей в деградацию KPI.

In [ ]:
# 3.18 Three-mode compare: 547_like vs hybrid_all vs hybrid_non_r2_only
required_318 = [
    'excel_ref_317_df',
    'like547_key_metrics_317_df',
    'hybrid_key_metrics_317_df',
    'patchable_by_old_317_df',
]
missing_318 = [x for x in required_318 if x not in globals()]
if missing_318:
    raise RuntimeError(f"Сначала выполните 3.17. Не найдены объекты: {missing_318}")


def _num_318(df, col):
    if col not in df.columns:
        return pd.Series(dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


def _pct_div_318(lake_total, excel_total):
    if pd.isna(excel_total) or float(excel_total) == 0.0:
        return None
    return (float(lake_total) - float(excel_total)) / float(excel_total) * 100.0


def _build_cmp_mode_318(excel_df, key_df):
    cmp_df = excel_df.merge(
        key_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode']],
        on=['inn_key', 'agr_id_key'],
        how='inner'
    )
    for m in ['retl_cnt', 'term_cnt']:
        cmp_df[f'{m}_delta_mode'] = _num_318(cmp_df, f'{m}_lake_mode').fillna(0) - _num_318(cmp_df, f'{m}_excel').fillna(0)
        cmp_df[f'{m}_abs_mode'] = _num_318(cmp_df, f'{m}_delta_mode').abs()
    return cmp_df


def _kpi_mode_318(cmp_df):
    return {
        'rows_compared': len(cmp_df),
        'retl_cnt_exact_match_rate_pct_tol_0': exact_match_rate_from_abs(cmp_df['retl_cnt_abs_mode'], tol=0.0),
        'term_cnt_exact_match_rate_pct_tol_0': exact_match_rate_from_abs(cmp_df['term_cnt_abs_mode'], tol=0.0),
        'retl_cnt_mae': float(_num_318(cmp_df, 'retl_cnt_abs_mode').mean()) if len(cmp_df) else 0.0,
        'term_cnt_mae': float(_num_318(cmp_df, 'term_cnt_abs_mode').mean()) if len(cmp_df) else 0.0,
    }


excel_ref_318_df = (
    excel_ref_317_df[['inn_key', 'agr_id_key', 'retl_cnt_excel', 'term_cnt_excel']]
    .dropna(subset=['inn_key', 'agr_id_key'])
    .drop_duplicates(subset=['inn_key', 'agr_id_key'])
    .copy()
)

# 547-like keys (standardized column names)
keys_547_318_df = (
    like547_key_metrics_317_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_547', 'term_cnt_lake_547']]
    .rename(columns={
        'retl_cnt_lake_547': 'retl_cnt_lake_mode',
        'term_cnt_lake_547': 'term_cnt_lake_mode',
    })
    .drop_duplicates(subset=['inn_key', 'agr_id_key'])
)
keys_547_318_df['mode_source'] = '547_like'

# Hybrid all keys (already includes patch)
keys_hybrid_all_318_df = (
    hybrid_key_metrics_317_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_hybrid', 'term_cnt_lake_hybrid', 'calc_source', 'fallback_source_type']]
    .rename(columns={
        'retl_cnt_lake_hybrid': 'retl_cnt_lake_mode',
        'term_cnt_lake_hybrid': 'term_cnt_lake_mode',
    })
    .drop_duplicates(subset=['inn_key', 'agr_id_key'])
)
keys_hybrid_all_318_df['mode_source'] = 'hybrid_all'

# Hybrid non-r2-only keys: core 547 + patch only non-r2 keys
patch_probe_318_df = patchable_by_old_317_df.copy()
if 'is_r2_fallback_key' not in patch_probe_318_df.columns:
    patch_probe_318_df['is_r2_fallback_key'] = 0

patch_non_r2_318_df = (
    patch_probe_318_df[patch_probe_318_df['is_r2_fallback_key'] == 0][['inn_key', 'agr_id_key', 'retl_cnt_lake_fallback', 'term_cnt_lake_fallback']]
    .rename(columns={
        'retl_cnt_lake_fallback': 'retl_cnt_lake_mode',
        'term_cnt_lake_fallback': 'term_cnt_lake_mode',
    })
    .drop_duplicates(subset=['inn_key', 'agr_id_key'])
)
patch_non_r2_318_df['mode_source'] = 'fallback_non_r2_patch'

keys_hybrid_non_r2_318_df = pd.concat([
    keys_547_318_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode', 'mode_source']],
    patch_non_r2_318_df[['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode', 'mode_source']],
], ignore_index=True)

# keep core 547 if duplicates appear
keys_hybrid_non_r2_318_df['_priority'] = keys_hybrid_non_r2_318_df['mode_source'].map({'547_like': 0, 'fallback_non_r2_patch': 1}).fillna(9)
keys_hybrid_non_r2_318_df = (
    keys_hybrid_non_r2_318_df
    .sort_values(['_priority', 'inn_key', 'agr_id_key'])
    .drop_duplicates(subset=['inn_key', 'agr_id_key'], keep='first')
    .drop(columns=['_priority'])
)
keys_hybrid_non_r2_318_df['mode_source'] = 'hybrid_non_r2_only'

# Build compare frames for 3 modes
cmp_547_318_df = _build_cmp_mode_318(excel_ref_318_df, keys_547_318_df)
cmp_hybrid_all_318_df = _build_cmp_mode_318(excel_ref_318_df, keys_hybrid_all_318_df)
cmp_hybrid_non_r2_318_df = _build_cmp_mode_318(excel_ref_318_df, keys_hybrid_non_r2_318_df)

kpi_547_318 = _kpi_mode_318(cmp_547_318_df)
kpi_hybrid_all_318 = _kpi_mode_318(cmp_hybrid_all_318_df)
kpi_hybrid_non_r2_318 = _kpi_mode_318(cmp_hybrid_non_r2_318_df)

kpi_three_modes_318_df = pd.DataFrame([
    {
        'metric': 'rows_compared',
        '547_like': kpi_547_318['rows_compared'],
        'hybrid_all': kpi_hybrid_all_318['rows_compared'],
        'hybrid_non_r2_only': kpi_hybrid_non_r2_318['rows_compared'],
    },
    {
        'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
        '547_like': kpi_547_318['retl_cnt_exact_match_rate_pct_tol_0'],
        'hybrid_all': kpi_hybrid_all_318['retl_cnt_exact_match_rate_pct_tol_0'],
        'hybrid_non_r2_only': kpi_hybrid_non_r2_318['retl_cnt_exact_match_rate_pct_tol_0'],
    },
    {
        'metric': 'term_cnt_exact_match_rate_pct_tol_0',
        '547_like': kpi_547_318['term_cnt_exact_match_rate_pct_tol_0'],
        'hybrid_all': kpi_hybrid_all_318['term_cnt_exact_match_rate_pct_tol_0'],
        'hybrid_non_r2_only': kpi_hybrid_non_r2_318['term_cnt_exact_match_rate_pct_tol_0'],
    },
    {
        'metric': 'retl_cnt_mae',
        '547_like': kpi_547_318['retl_cnt_mae'],
        'hybrid_all': kpi_hybrid_all_318['retl_cnt_mae'],
        'hybrid_non_r2_only': kpi_hybrid_non_r2_318['retl_cnt_mae'],
    },
    {
        'metric': 'term_cnt_mae',
        '547_like': kpi_547_318['term_cnt_mae'],
        'hybrid_all': kpi_hybrid_all_318['term_cnt_mae'],
        'hybrid_non_r2_only': kpi_hybrid_non_r2_318['term_cnt_mae'],
    },
])

kpi_three_modes_318_df['delta_hybrid_all_minus_547'] = kpi_three_modes_318_df['hybrid_all'] - kpi_three_modes_318_df['547_like']
kpi_three_modes_318_df['delta_non_r2_minus_547'] = kpi_three_modes_318_df['hybrid_non_r2_only'] - kpi_three_modes_318_df['547_like']

# Coverage summary
coverage_three_modes_318_df = pd.DataFrame([
    {
        'excel_key_count': len(excel_ref_318_df[['inn_key', 'agr_id_key']].drop_duplicates()),
        'covered_547_like': len(cmp_547_318_df),
        'covered_hybrid_all': len(cmp_hybrid_all_318_df),
        'covered_hybrid_non_r2_only': len(cmp_hybrid_non_r2_318_df),
        'added_vs_547_hybrid_all': len(cmp_hybrid_all_318_df) - len(cmp_547_318_df),
        'added_vs_547_non_r2_only': len(cmp_hybrid_non_r2_318_df) - len(cmp_547_318_df),
        'patched_keys_all': len(patch_probe_318_df),
        'patched_keys_r2': int((patch_probe_318_df['is_r2_fallback_key'] == 1).sum()) if len(patch_probe_318_df) else 0,
        'patched_keys_non_r2': int((patch_probe_318_df['is_r2_fallback_key'] == 0).sum()) if len(patch_probe_318_df) else 0,
    }
])

# Divergence totals by mode
divergence_three_modes_rows_318 = []
for metric_name in ['retl_cnt', 'term_cnt']:
    lake_547_total = _num_318(cmp_547_318_df, f'{metric_name}_lake_mode').fillna(0).sum()
    excel_547_total = _num_318(cmp_547_318_df, f'{metric_name}_excel').fillna(0).sum()
    pct_547 = _pct_div_318(lake_547_total, excel_547_total)

    lake_h_all_total = _num_318(cmp_hybrid_all_318_df, f'{metric_name}_lake_mode').fillna(0).sum()
    excel_h_all_total = _num_318(cmp_hybrid_all_318_df, f'{metric_name}_excel').fillna(0).sum()
    pct_h_all = _pct_div_318(lake_h_all_total, excel_h_all_total)

    lake_h_n2_total = _num_318(cmp_hybrid_non_r2_318_df, f'{metric_name}_lake_mode').fillna(0).sum()
    excel_h_n2_total = _num_318(cmp_hybrid_non_r2_318_df, f'{metric_name}_excel').fillna(0).sum()
    pct_h_n2 = _pct_div_318(lake_h_n2_total, excel_h_n2_total)

    divergence_three_modes_rows_318.append({
        'metric': metric_name,
        'divergence_pct_547_like': pct_547,
        'divergence_pct_hybrid_all': pct_h_all,
        'divergence_pct_hybrid_non_r2_only': pct_h_n2,
        'abs_improvement_hybrid_all_vs_547': (abs(pct_547) - abs(pct_h_all)) if pct_547 is not None and pct_h_all is not None else None,
        'abs_improvement_non_r2_vs_547': (abs(pct_547) - abs(pct_h_n2)) if pct_547 is not None and pct_h_n2 is not None else None,
    })

divergence_three_modes_318_df = pd.DataFrame(divergence_three_modes_rows_318)

# Focus INN
focus_inn_318 = '7106527696'
focus_excel_318_df = excel_ref_318_df[excel_ref_318_df['inn_key'] == focus_inn_318].copy()

focus_547_vals_318_df = cmp_547_318_df[cmp_547_318_df['inn_key'] == focus_inn_318][['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode']].drop_duplicates().rename(
    columns={'retl_cnt_lake_mode': 'retl_lake_547_like', 'term_cnt_lake_mode': 'term_lake_547_like'}
)
focus_hall_vals_318_df = cmp_hybrid_all_318_df[cmp_hybrid_all_318_df['inn_key'] == focus_inn_318][['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode']].drop_duplicates().rename(
    columns={'retl_cnt_lake_mode': 'retl_lake_hybrid_all', 'term_cnt_lake_mode': 'term_lake_hybrid_all'}
)
focus_hn2_vals_318_df = cmp_hybrid_non_r2_318_df[cmp_hybrid_non_r2_318_df['inn_key'] == focus_inn_318][['inn_key', 'agr_id_key', 'retl_cnt_lake_mode', 'term_cnt_lake_mode']].drop_duplicates().rename(
    columns={'retl_cnt_lake_mode': 'retl_lake_hybrid_non_r2', 'term_cnt_lake_mode': 'term_lake_hybrid_non_r2'}
)

focus_7106527696_modes_318_df = (
    focus_excel_318_df
    .merge(focus_547_vals_318_df, on=['inn_key', 'agr_id_key'], how='left')
    .merge(focus_hall_vals_318_df, on=['inn_key', 'agr_id_key'], how='left')
    .merge(focus_hn2_vals_318_df, on=['inn_key', 'agr_id_key'], how='left')
)

if len(focus_7106527696_modes_318_df):
    focus_7106527696_modes_318_df['retl_delta_547_like'] = _num_318(focus_7106527696_modes_318_df, 'retl_lake_547_like').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'retl_cnt_excel').fillna(0)
    focus_7106527696_modes_318_df['retl_delta_hybrid_all'] = _num_318(focus_7106527696_modes_318_df, 'retl_lake_hybrid_all').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'retl_cnt_excel').fillna(0)
    focus_7106527696_modes_318_df['retl_delta_hybrid_non_r2'] = _num_318(focus_7106527696_modes_318_df, 'retl_lake_hybrid_non_r2').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'retl_cnt_excel').fillna(0)

    focus_7106527696_modes_318_df['term_delta_547_like'] = _num_318(focus_7106527696_modes_318_df, 'term_lake_547_like').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'term_cnt_excel').fillna(0)
    focus_7106527696_modes_318_df['term_delta_hybrid_all'] = _num_318(focus_7106527696_modes_318_df, 'term_lake_hybrid_all').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'term_cnt_excel').fillna(0)
    focus_7106527696_modes_318_df['term_delta_hybrid_non_r2'] = _num_318(focus_7106527696_modes_318_df, 'term_lake_hybrid_non_r2').fillna(0) - _num_318(focus_7106527696_modes_318_df, 'term_cnt_excel').fillna(0)

print('Coverage summary (3.18):')
display(coverage_three_modes_318_df)

print('\nKPI compare: 547_like vs hybrid_all vs hybrid_non_r2_only')
display(kpi_three_modes_318_df)

print('Divergence compare (totals):')
display(divergence_three_modes_318_df)

print('\nFocus INN 7106527696 across 3 modes:')
display(focus_7106527696_modes_318_df)

if len(patch_probe_318_df):
    patch_source_summary_318_df = (
        patch_probe_318_df.assign(
            fallback_source_type=patch_probe_318_df['is_r2_fallback_key'].map({1: 'r2_fallback', 0: 'old_non_r2'})
        )
        .groupby('fallback_source_type', as_index=False)
        .agg(keys_cnt=('agr_id_key', 'size'))
        .sort_values('keys_cnt', ascending=False)
    )
    print('Patch source mix:')
    display(patch_source_summary_318_df)

### 3.19 Deep-dive по `agr_id=125890176264`: какие терминалы считаются лишними

Цель:
- посмотреть «сырые» записи из таблиц связки договора и терминалов;
- сравнить периметры `547-like` и текущей fallback-логики;
- глазами увидеть, какие `c_nmrc/c_pos_serial/c_nter` дают завышение `term_cnt`.

In [ ]:
# 3.19 Deep-dive case: agr_id=125890176264 (without POS_DEVICES/POS_DEV_STAT)
if 'imp' not in globals():
    raise RuntimeError('Сначала выполните секцию подключения к Impala.')
if 'month_start' not in globals() or 'month_end' not in globals():
    raise RuntimeError('Сначала выполните секцию параметров месяца.')


def _clean_keys_319(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in ['None', 'nan', 'NaN']:
            out.append(s)
    return sorted(set(out))


def _sql_in_319(values):
    vals = _clean_keys_319(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


case_agr_id_319 = '125890176264'

# A) Agreement card
sql_case_agr_319 = f"""
select distinct
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.n_cmp_client as string) as n_cmp_client,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  cast(a.acq_class as string) as acq_class
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where cast(a.abs_agr_id as string) = '{case_agr_id_319}'
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    case_agr_319_df = imp.fetch(sql_case_agr_319)

if case_agr_319_df is None:
    case_agr_319_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'n_cmp_client', 'inn', 'company_name', 'd_valid_from', 'd_valid_to', 'acq_class'])

case_context_source_319 = 'agreements'

# Some problematic keys exist only as fallback/r2 ids and are absent in agreements.abs_agr_id.
# In that case, keep the diagnostic running using local context from already built dataframes.
if case_agr_319_df.empty:
    case_context_source_319 = 'not_found_in_agreements'

    if 'base_df' in globals() and isinstance(base_df, pd.DataFrame) and len(base_df) and 'agr_id' in base_df.columns:
        base_probe_319 = base_df.copy()
        base_probe_319['agr_id_key'] = base_probe_319['agr_id'].apply(normalize_agr_q1)
        base_probe_319 = base_probe_319[base_probe_319['agr_id_key'] == case_agr_id_319]

        if len(base_probe_319):
            cols_map = {
                'agr_id': base_probe_319['agr_id'].astype(str),
                'n_agr': base_probe_319['n_agr'].astype(str) if 'n_agr' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'n_cmp_client': base_probe_319['n_cmp_client'].astype(str) if 'n_cmp_client' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'inn': base_probe_319['inn'].astype(str) if 'inn' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'company_name': base_probe_319['company_name'].astype(str) if 'company_name' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'd_valid_from': base_probe_319['d_valid_from'] if 'd_valid_from' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'd_valid_to': base_probe_319['d_valid_to'] if 'd_valid_to' in base_probe_319.columns else pd.Series([None] * len(base_probe_319)),
                'acq_class': pd.Series(['SA'] * len(base_probe_319)),
            }
            case_agr_319_df = pd.DataFrame(cols_map).drop_duplicates()
            case_context_source_319 = 'base_df_fallback'

    if case_agr_319_df.empty and 'cmp_before_315' in globals() and isinstance(cmp_before_315, pd.DataFrame) and len(cmp_before_315):
        cmp_probe_319 = cmp_before_315[cmp_before_315['agr_id_key'] == case_agr_id_319].copy() if 'agr_id_key' in cmp_before_315.columns else pd.DataFrame()
        if len(cmp_probe_319):
            inn_key_val = cmp_probe_319['inn_key'].dropna().astype(str).iloc[0] if 'inn_key' in cmp_probe_319.columns and len(cmp_probe_319['inn_key'].dropna()) else None
            case_agr_319_df = pd.DataFrame([
                {
                    'agr_id': case_agr_id_319,
                    'n_agr': None,
                    'n_cmp_client': None,
                    'inn': inn_key_val,
                    'company_name': None,
                    'd_valid_from': None,
                    'd_valid_to': None,
                    'acq_class': 'SA',
                }
            ])
            case_context_source_319 = 'cmp_before_315_fallback'

if case_agr_319_df.empty:
    case_agr_319_df = pd.DataFrame([
        {
            'agr_id': case_agr_id_319,
            'n_agr': None,
            'n_cmp_client': None,
            'inn': None,
            'company_name': None,
            'd_valid_from': None,
            'd_valid_to': None,
            'acq_class': 'SA',
        }
    ])
    case_context_source_319 = 'no_context_found'

for c in ['agr_id', 'n_agr', 'n_cmp_client', 'inn', 'company_name', 'acq_class']:
    if c in case_agr_319_df.columns:
        case_agr_319_df[c] = case_agr_319_df[c].astype(str)

case_n_agr_319 = _clean_keys_319(case_agr_319_df['n_agr'].tolist())
case_n_cmp_319 = _clean_keys_319(case_agr_319_df['n_cmp_client'].tolist())
case_n_agr_in_319 = _sql_in_319(case_n_agr_319)
case_n_cmp_in_319 = _sql_in_319(case_n_cmp_319)

print('3.19 context source =', case_context_source_319)
if not case_n_agr_319:
    print('3.19 note: n_agr not available for this agr_id, AGR_TERMS branch may be empty.')
if not case_n_cmp_319:
    print('3.19 note: n_cmp_client not available for this agr_id, n_cmp fallback branch may be empty.')

# B) 547-like nmrc perimeter via AGR_TERMS
sql_case_terms_319 = f"""
select distinct
  cast(t.n_agr as string) as n_agr,
  cast(t.c_nmrc as string) as c_nmrc,
  cast(t.d_valid_from as date) as d_valid_from,
  cast(t.d_valid_to as date) as d_valid_to
from ods_alpha.scd1_agr_terms t
where cast(t.n_agr as string) in ({case_n_agr_in_319})
  and t.c_nmrc is not null
  and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
  and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(t.ods_deleted_flg, '0') <> '1'
order by cast(t.n_agr as string), cast(t.c_nmrc as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    case_terms_319_df = imp.fetch(sql_case_terms_319)

if case_terms_319_df is None:
    case_terms_319_df = pd.DataFrame(columns=['n_agr', 'c_nmrc', 'd_valid_from', 'd_valid_to'])

for c in ['n_agr', 'c_nmrc']:
    if c in case_terms_319_df.columns:
        case_terms_319_df[c] = case_terms_319_df[c].astype(str).str.strip()

# C) Merchants by AGR_TERMS c_nmrc + merchants by n_cmp fallback perimeter
terms_nmrc_319 = _clean_keys_319(case_terms_319_df['c_nmrc'].tolist())
terms_nmrc_in_319 = _sql_in_319(terms_nmrc_319)

if terms_nmrc_319:
    sql_merch_terms_319 = f"""
    select distinct
      cast(m.c_nmrc as string) as c_nmrc,
      cast(m.n_cmp as string) as n_cmp,
      cast(m.c_mrc_name as string) as c_mrc_name,
      cast(m.acq_class as string) as acq_class,
      cast(m.n_mcc as string) as n_mcc
    from ods_alpha.scd1_merchants m
    where cast(m.c_nmrc as string) in ({terms_nmrc_in_319})
      and coalesce(m.ods_deleted_flg, '0') <> '1'
    """
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        case_merch_terms_319_df = imp.fetch(sql_merch_terms_319)
else:
    case_merch_terms_319_df = pd.DataFrame(columns=['c_nmrc', 'n_cmp', 'c_mrc_name', 'acq_class', 'n_mcc'])

if case_merch_terms_319_df is None:
    case_merch_terms_319_df = pd.DataFrame(columns=['c_nmrc', 'n_cmp', 'c_mrc_name', 'acq_class', 'n_mcc'])

sql_merch_ncmp_319 = f"""
select distinct
  cast(m.c_nmrc as string) as c_nmrc,
  cast(m.n_cmp as string) as n_cmp,
  cast(m.c_mrc_name as string) as c_mrc_name,
  cast(m.acq_class as string) as acq_class,
  cast(m.n_mcc as string) as n_mcc
from ods_alpha.scd1_merchants m
where cast(m.n_cmp as string) in ({case_n_cmp_in_319})
  and m.c_nmrc is not null
  and coalesce(m.ods_deleted_flg, '0') <> '1'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    case_merch_ncmp_319_df = imp.fetch(sql_merch_ncmp_319)

if case_merch_ncmp_319_df is None:
    case_merch_ncmp_319_df = pd.DataFrame(columns=['c_nmrc', 'n_cmp', 'c_mrc_name', 'acq_class', 'n_mcc'])

for df in [case_merch_terms_319_df, case_merch_ncmp_319_df]:
    for c in ['c_nmrc', 'n_cmp', 'c_mrc_name', 'acq_class', 'n_mcc']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

# D) POS_TERMINALS by both perimeters
sql_pos_by_terms_319 = f"""
with terms as (
  select distinct cast(t.c_nmrc as string) as c_nmrc
  from ods_alpha.scd1_agr_terms t
  where cast(t.n_agr as string) in ({case_n_agr_in_319})
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(t.ods_deleted_flg, '0') <> '1'
)
select distinct
  cast(t.c_nmrc as string) as c_nmrc,
  cast(t.c_nter as string) as c_nter,
  cast(t.c_pos_serial as string) as c_pos_serial,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  cast(t.last_trx_date as date) as last_trx_date,
  case
    when cast(t.d_ter_install as date) is not null
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
    then 1 else 0
  end as is_active_in_month
from terms x
join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = x.c_nmrc
where t.c_nter is not null
  and coalesce(t.ods_deleted_flg, '0') <> '1'
order by cast(t.c_nmrc as string), cast(t.c_pos_serial as string), cast(t.c_nter as string)
"""

sql_pos_by_ncmp_319 = f"""
with m as (
  select distinct cast(mm.c_nmrc as string) as c_nmrc
  from ods_alpha.scd1_merchants mm
  where cast(mm.n_cmp as string) in ({case_n_cmp_in_319})
    and mm.c_nmrc is not null
    and coalesce(mm.ods_deleted_flg, '0') <> '1'
)
select distinct
  cast(t.c_nmrc as string) as c_nmrc,
  cast(t.c_nter as string) as c_nter,
  cast(t.c_pos_serial as string) as c_pos_serial,
  cast(t.d_ter_install as date) as d_ter_install,
  cast(t.d_ter_close as date) as d_ter_close,
  cast(t.last_trx_date as date) as last_trx_date,
  case
    when cast(t.d_ter_install as date) is not null
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
    then 1 else 0
  end as is_active_in_month
from m x
join ods_alpha.scd1_pos_terminals t
  on cast(t.c_nmrc as string) = x.c_nmrc
where t.c_nter is not null
  and coalesce(t.ods_deleted_flg, '0') <> '1'
order by cast(t.c_nmrc as string), cast(t.c_pos_serial as string), cast(t.c_nter as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=12g')
    case_pos_terms_319_df = imp.fetch(sql_pos_by_terms_319)
    case_pos_ncmp_319_df = imp.fetch(sql_pos_by_ncmp_319)

if case_pos_terms_319_df is None:
    case_pos_terms_319_df = pd.DataFrame(columns=['c_nmrc', 'c_nter', 'c_pos_serial', 'd_ter_install', 'd_ter_close', 'last_trx_date', 'is_active_in_month'])
if case_pos_ncmp_319_df is None:
    case_pos_ncmp_319_df = pd.DataFrame(columns=['c_nmrc', 'c_nter', 'c_pos_serial', 'd_ter_install', 'd_ter_close', 'last_trx_date', 'is_active_in_month'])

for df in [case_pos_terms_319_df, case_pos_ncmp_319_df]:
    for c in ['c_nmrc', 'c_nter', 'c_pos_serial']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
    if 'is_active_in_month' in df.columns:
        df['is_active_in_month'] = pd.to_numeric(df['is_active_in_month'], errors='coerce').fillna(0).astype(int)

# E) Sets and differences (active month)
terms_active_serials_319 = set(
    case_pos_terms_319_df.loc[case_pos_terms_319_df['is_active_in_month'] == 1, 'c_pos_serial']
    .dropna().astype(str).str.strip().tolist()
)
ncmp_active_serials_319 = set(
    case_pos_ncmp_319_df.loc[case_pos_ncmp_319_df['is_active_in_month'] == 1, 'c_pos_serial']
    .dropna().astype(str).str.strip().tolist()
)

terms_active_nters_319 = set(
    case_pos_terms_319_df.loc[case_pos_terms_319_df['is_active_in_month'] == 1, 'c_nter']
    .dropna().astype(str).str.strip().tolist()
)
ncmp_active_nters_319 = set(
    case_pos_ncmp_319_df.loc[case_pos_ncmp_319_df['is_active_in_month'] == 1, 'c_nter']
    .dropna().astype(str).str.strip().tolist()
)

serial_only_in_ncmp_319 = sorted(ncmp_active_serials_319 - terms_active_serials_319)
serial_only_in_terms_319 = sorted(terms_active_serials_319 - ncmp_active_serials_319)

case_diff_rows_319 = []
for s in serial_only_in_ncmp_319:
    case_diff_rows_319.append({'c_pos_serial': s, 'where_seen': 'only_in_ncmp_fallback_active'})
for s in serial_only_in_terms_319:
    case_diff_rows_319.append({'c_pos_serial': s, 'where_seen': 'only_in_agr_terms_active'})

case_serial_diff_319_df = pd.DataFrame(case_diff_rows_319)

# F) Transactions on extra terminals (to inspect if "extra" terminals really transact)
extra_nter_319 = _clean_keys_319(
    case_pos_ncmp_319_df[
        (case_pos_ncmp_319_df['is_active_in_month'] == 1)
        & (case_pos_ncmp_319_df['c_pos_serial'].isin(serial_only_in_ncmp_319))
    ]['c_nter'].tolist()
)

extra_trx_319_parts = []
if extra_nter_319:
    for i in range(0, len(extra_nter_319), 900):
        nter_chunk = extra_nter_319[i:i+900]
        nter_in = _sql_in_319(nter_chunk)
        sql_extra_trx_319 = f"""
        select
          cast(t.c_nter as string) as c_nter,
          count(distinct cast(t.n_trx as string)) as trx_cnt,
          sum(cast(t.n_amt_src as double)) as trx_sum
        from ods_alpha.scd1_trx t
        where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
          and cast(t.c_nter as string) in ({nter_in})
          and coalesce(t.ods_deleted_flg, '0') <> '1'
          and t.c_trx_class = 'SA'
          and t.c_trx_type = 'S01'
          and coalesce(t.cf_trx_stat, '') <> 'R'
        group by cast(t.c_nter as string)
        """
        with imp:
            imp.execute('set MEM_LIMIT=8g')
            part_df = imp.fetch(sql_extra_trx_319)
        if part_df is not None and len(part_df):
            extra_trx_319_parts.append(part_df)

if extra_trx_319_parts:
    extra_trx_319_df = pd.concat(extra_trx_319_parts, ignore_index=True)
else:
    extra_trx_319_df = pd.DataFrame(columns=['c_nter', 'trx_cnt', 'trx_sum'])

extra_terminals_detail_319_df = (
    case_pos_ncmp_319_df[
        (case_pos_ncmp_319_df['is_active_in_month'] == 1)
        & (case_pos_ncmp_319_df['c_pos_serial'].isin(serial_only_in_ncmp_319))
    ][['c_nmrc', 'c_nter', 'c_pos_serial', 'd_ter_install', 'd_ter_close', 'last_trx_date']]
    .drop_duplicates()
)
if len(extra_trx_319_df):
    extra_terminals_detail_319_df = extra_terminals_detail_319_df.merge(extra_trx_319_df, on='c_nter', how='left')

# G) Summary with old/new counts from existing compare frames
case_inn_key_319 = normalize_inn_q1(case_agr_319_df['inn'].iloc[0]) if len(case_agr_319_df) else None
case_agr_key_319 = normalize_agr_q1(case_agr_id_319)

case_old_row_319_df = pd.DataFrame()
if 'cmp_before_315' in globals() and isinstance(cmp_before_315, pd.DataFrame) and len(cmp_before_315):
    case_old_row_319_df = cmp_before_315[
        (cmp_before_315['inn_key'] == case_inn_key_319)
        & (cmp_before_315['agr_id_key'] == case_agr_key_319)
    ][['retl_cnt_lake', 'term_cnt_lake', 'retl_cnt_excel', 'term_cnt_excel']].head(1)

case_547_row_319_df = pd.DataFrame()
if 'cmp_547_retl_term_df' in globals() and isinstance(cmp_547_retl_term_df, pd.DataFrame) and len(cmp_547_retl_term_df):
    case_547_row_319_df = cmp_547_retl_term_df[
        (cmp_547_retl_term_df['inn_key'] == case_inn_key_319)
        & (cmp_547_retl_term_df['agr_id_key'] == case_agr_key_319)
    ][['retl_cnt_lake_547', 'term_cnt_lake_547', 'retl_cnt_excel', 'term_cnt_excel']].head(1)

case_summary_319 = {
    'agr_id': case_agr_id_319,
    'n_agr_list': ','.join(case_n_agr_319),
    'n_cmp_client_list': ','.join(case_n_cmp_319),
    'inn': case_inn_key_319,
    'retl_terms_nmrc_cnt': int(case_terms_319_df['c_nmrc'].nunique()) if len(case_terms_319_df) else 0,
    'retl_ncmp_nmrc_cnt': int(case_merch_ncmp_319_df['c_nmrc'].nunique()) if len(case_merch_ncmp_319_df) else 0,
    'term_serial_terms_active_cnt': len(terms_active_serials_319),
    'term_serial_ncmp_active_cnt': len(ncmp_active_serials_319),
    'term_nter_terms_active_cnt': len(terms_active_nters_319),
    'term_nter_ncmp_active_cnt': len(ncmp_active_nters_319),
    'serial_only_in_ncmp_active_cnt': len(serial_only_in_ncmp_319),
    'serial_only_in_terms_active_cnt': len(serial_only_in_terms_319),
}

if len(case_old_row_319_df):
    case_summary_319['retl_old_lake'] = float(pd.to_numeric(case_old_row_319_df['retl_cnt_lake'], errors='coerce').iloc[0])
    case_summary_319['term_old_lake'] = float(pd.to_numeric(case_old_row_319_df['term_cnt_lake'], errors='coerce').iloc[0])
    case_summary_319['retl_excel'] = float(pd.to_numeric(case_old_row_319_df['retl_cnt_excel'], errors='coerce').iloc[0])
    case_summary_319['term_excel'] = float(pd.to_numeric(case_old_row_319_df['term_cnt_excel'], errors='coerce').iloc[0])

if len(case_547_row_319_df):
    case_summary_319['retl_547_like'] = float(pd.to_numeric(case_547_row_319_df['retl_cnt_lake_547'], errors='coerce').iloc[0])
    case_summary_319['term_547_like'] = float(pd.to_numeric(case_547_row_319_df['term_cnt_lake_547'], errors='coerce').iloc[0])

case_summary_319_df = pd.DataFrame([case_summary_319])

print('Case card:')
display(case_agr_319_df)

print('Summary (counts and deltas):')
display(case_summary_319_df)

print('AGR_TERMS perimeter (period):')
display(case_terms_319_df.head(200))

print('Merchants by AGR_TERMS c_nmrc:')
display(case_merch_terms_319_df.head(200))

print('Merchants by n_cmp_client fallback perimeter:')
display(case_merch_ncmp_319_df.head(200))

print('POS_TERMINALS by AGR_TERMS perimeter:')
display(case_pos_terms_319_df.head(300))

print('POS_TERMINALS by n_cmp_client fallback perimeter:')
display(case_pos_ncmp_319_df.head(300))

print('Serial differences between perimeters (active month):')
display(case_serial_diff_319_df.head(300))

print('Details for serials only in n_cmp fallback + trx in month:')
display(extra_terminals_detail_319_df.head(300))

### 3.20 Excel key lineage: потери ключей в compare + оценка эффекта фикса

Цели секции:
- понять, сколько ключей теряется на этапе парсинга `agr_id_key` из Excel;
- отдельно посчитать потери из-за `inner`-сопоставления Lake vs Excel;
- смоделировать эффект фикса (`agr_id` fallback-парсинг из поля договора) на coverage и KPI;
- проверить фокус-кейс `ИНН=0254010612` построчно.

In [ ]:
# 3.20 Excel key lineage: key-loss diagnostics + fix impact simulation
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните 10_apply_tariff_fix_and_formulas (нужен final_df).')
if 'excel_path' not in globals() or not excel_path:
    raise RuntimeError('Не найден excel_path. Сначала выполните секцию параметров.')

excel_header_320 = excel_header if 'excel_header' in globals() else 0
ex_raw_320_df = pd.read_excel(excel_path, header=excel_header_320)

# Resolve columns (same style as section 2)
col_map_320 = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'contract_col': ['Номер договора', 'c_agr_number', 'C_AGR_NUMBER', 'Договор'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
}

resolved_320 = {k: pick_col_robust(ex_raw_320_df.columns, v) for k, v in col_map_320.items()}
if resolved_320['inn_col'] is None or resolved_320['agr_col'] is None:
    raise RuntimeError(f"Не найдены ключевые колонки inn/agr в Excel. resolved={resolved_320}")


def _extract_agr_candidates_320(v):
    if pd.isna(v):
        return []
    s = str(v).strip()
    if not s:
        return []

    candidates = [s]

    digits_chunks = re.findall(r'\d{6,}', s)
    candidates.extend(digits_chunks)

    all_digits = re.sub(r'\D+', '', s)
    if len(all_digits) >= 6:
        candidates.append(all_digits)

    # unique while preserving order
    uniq = []
    seen = set()
    for c in candidates:
        cc = str(c).strip()
        if cc and cc not in seen:
            uniq.append(cc)
            seen.add(cc)
    return uniq


def _parse_agr_key_extended_320(agr_raw, contract_raw):
    k0 = normalize_agr_q1(agr_raw)
    if k0:
        return k0, 'agr_col'

    for cand in _extract_agr_candidates_320(agr_raw):
        k = normalize_agr_q1(cand)
        if k:
            return k, 'agr_col_digits'

    for cand in _extract_agr_candidates_320(contract_raw):
        k = normalize_agr_q1(cand)
        if k:
            return k, 'contract_col_fallback'

    return None, 'unparsed'


def _num_320(df, col):
    if col not in df.columns:
        return pd.Series(dtype='float64')
    return pd.to_numeric(df[col], errors='coerce')


ex_diag_320_df = ex_raw_320_df.copy()
ex_diag_320_df['_row_id'] = range(1, len(ex_diag_320_df) + 1)

inn_col_320 = resolved_320['inn_col']
agr_col_320 = resolved_320['agr_col']
contract_col_320 = resolved_320['contract_col']

ex_diag_320_df['inn_key'] = ex_diag_320_df[inn_col_320].apply(normalize_inn_q1)
ex_diag_320_df['agr_raw'] = ex_diag_320_df[agr_col_320]
ex_diag_320_df['contract_raw'] = ex_diag_320_df[contract_col_320] if contract_col_320 else None

ex_diag_320_df['agr_id_key_current'] = ex_diag_320_df['agr_raw'].apply(normalize_agr_q1)

parsed_pairs_320 = ex_diag_320_df.apply(
    lambda r: _parse_agr_key_extended_320(r.get('agr_raw'), r.get('contract_raw')),
    axis=1
)
ex_diag_320_df['agr_id_key_fixed'] = parsed_pairs_320.apply(lambda x: x[0])
ex_diag_320_df['agr_parse_source_fixed'] = parsed_pairs_320.apply(lambda x: x[1])


def _issue_current_320(r):
    if pd.isna(r.get('inn_key')):
        return 'missing_inn_key'
    if pd.isna(r.get('agr_raw')) or str(r.get('agr_raw')).strip() in ['', 'None', 'nan', 'NaN']:
        return 'missing_agr_raw'
    if pd.isna(r.get('agr_id_key_current')):
        return 'unparsed_agr_raw'
    return 'parsed_ok'


def _issue_fixed_320(r):
    if pd.isna(r.get('inn_key')):
        return 'missing_inn_key'
    if pd.isna(r.get('agr_id_key_fixed')):
        return 'unparsed_after_fix'
    return 'parsed_ok'


ex_diag_320_df['parse_issue_current'] = ex_diag_320_df.apply(_issue_current_320, axis=1)
ex_diag_320_df['parse_issue_fixed'] = ex_diag_320_df.apply(_issue_fixed_320, axis=1)

# Build key sets
excel_keys_current_320_df = (
    ex_diag_320_df.dropna(subset=['inn_key', 'agr_id_key_current'])
    [['inn_key', 'agr_id_key_current']]
    .rename(columns={'agr_id_key_current': 'agr_id_key'})
    .drop_duplicates()
)
excel_keys_fixed_320_df = (
    ex_diag_320_df.dropna(subset=['inn_key', 'agr_id_key_fixed'])
    [['inn_key', 'agr_id_key_fixed']]
    .rename(columns={'agr_id_key_fixed': 'agr_id_key'})
    .drop_duplicates()
)

recovered_keys_320_df = (
    excel_keys_fixed_320_df
    .merge(excel_keys_current_320_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns=['_merge'])
)

# Lake keys
lk_320_df = final_df.copy()
lk_320_df['inn_key'] = lk_320_df['inn'].apply(normalize_inn_q1)
lk_320_df['agr_id_key'] = lk_320_df['agr_id'].apply(normalize_agr_q1)
lake_keys_320_df = lk_320_df.dropna(subset=['inn_key', 'agr_id_key'])[['inn_key', 'agr_id_key']].drop_duplicates()

# Coverage current vs fixed
matched_current_320 = len(excel_keys_current_320_df.merge(lake_keys_320_df, on=['inn_key', 'agr_id_key'], how='inner'))
matched_fixed_320 = len(excel_keys_fixed_320_df.merge(lake_keys_320_df, on=['inn_key', 'agr_id_key'], how='inner'))

only_excel_current_320 = len(
    excel_keys_current_320_df
    .merge(lake_keys_320_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
)
only_excel_fixed_320 = len(
    excel_keys_fixed_320_df
    .merge(lake_keys_320_df, on=['inn_key', 'agr_id_key'], how='left', indicator=True)
    .query("_merge == 'left_only'")
)

key_loss_summary_320_df = pd.DataFrame([
    {
        'excel_rows_total': len(ex_diag_320_df),
        'excel_keys_current': len(excel_keys_current_320_df),
        'excel_keys_fixed': len(excel_keys_fixed_320_df),
        'recovered_keys_by_fix': len(recovered_keys_320_df),
        'lake_keys': len(lake_keys_320_df),
        'matched_current_inner': matched_current_320,
        'matched_fixed_inner': matched_fixed_320,
        'match_gain_after_fix': matched_fixed_320 - matched_current_320,
        'only_excel_current': only_excel_current_320,
        'only_excel_fixed': only_excel_fixed_320,
    }
])

parse_issue_summary_320_df = (
    ex_diag_320_df.groupby(['parse_issue_current', 'parse_issue_fixed'], as_index=False)
    .agg(rows=('inn_key', 'size'))
    .sort_values('rows', ascending=False)
    .reset_index(drop=True)
)

parse_source_fixed_320_df = (
    ex_diag_320_df.groupby('agr_parse_source_fixed', as_index=False)
    .agg(rows=('inn_key', 'size'))
    .sort_values('rows', ascending=False)
    .reset_index(drop=True)
)

# KPI simulation for retl/term
retl_col_320 = resolved_320['retl_col']
term_col_320 = resolved_320['term_col']

kpi_compare_current_vs_fixed_320_df = pd.DataFrame()
cmp_current_320_df = pd.DataFrame()
cmp_fixed_320_df = pd.DataFrame()

if retl_col_320 is not None and term_col_320 is not None:
    ex_diag_320_df['retl_cnt_excel'] = pd.to_numeric(ex_diag_320_df[retl_col_320], errors='coerce')
    ex_diag_320_df['term_cnt_excel'] = pd.to_numeric(ex_diag_320_df[term_col_320], errors='coerce')

    ex_agg_current_320_df = (
        ex_diag_320_df.dropna(subset=['inn_key', 'agr_id_key_current'])
        .groupby(['inn_key', 'agr_id_key_current'], as_index=False)
        .agg({'retl_cnt_excel': 'max', 'term_cnt_excel': 'max'})
        .rename(columns={'agr_id_key_current': 'agr_id_key'})
    )

    ex_agg_fixed_320_df = (
        ex_diag_320_df.dropna(subset=['inn_key', 'agr_id_key_fixed'])
        .groupby(['inn_key', 'agr_id_key_fixed'], as_index=False)
        .agg({'retl_cnt_excel': 'max', 'term_cnt_excel': 'max'})
        .rename(columns={'agr_id_key_fixed': 'agr_id_key'})
    )

    lk_agg_320_df = (
        lk_320_df.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt': 'max',
            'term_cnt': 'max',
        })
        .rename(columns={'retl_cnt': 'retl_cnt_lake', 'term_cnt': 'term_cnt_lake'})
    )

    cmp_current_320_df = lk_agg_320_df.merge(ex_agg_current_320_df, on=['inn_key', 'agr_id_key'], how='inner')
    cmp_fixed_320_df = lk_agg_320_df.merge(ex_agg_fixed_320_df, on=['inn_key', 'agr_id_key'], how='inner')

    for m in ['retl_cnt', 'term_cnt']:
        cmp_current_320_df[f'{m}_abs'] = (_num_320(cmp_current_320_df, f'{m}_lake').fillna(0) - _num_320(cmp_current_320_df, f'{m}_excel').fillna(0)).abs()
        cmp_fixed_320_df[f'{m}_abs'] = (_num_320(cmp_fixed_320_df, f'{m}_lake').fillna(0) - _num_320(cmp_fixed_320_df, f'{m}_excel').fillna(0)).abs()

    kpi_compare_current_vs_fixed_320_df = pd.DataFrame([
        {
            'metric': 'rows_compared',
            'current_parse': len(cmp_current_320_df),
            'fixed_parse': len(cmp_fixed_320_df),
            'delta_fixed_minus_current': len(cmp_fixed_320_df) - len(cmp_current_320_df),
        },
        {
            'metric': 'retl_cnt_exact_match_rate_pct_tol_0',
            'current_parse': exact_match_rate_from_abs(cmp_current_320_df['retl_cnt_abs'], tol=0.0),
            'fixed_parse': exact_match_rate_from_abs(cmp_fixed_320_df['retl_cnt_abs'], tol=0.0),
            'delta_fixed_minus_current': exact_match_rate_from_abs(cmp_fixed_320_df['retl_cnt_abs'], tol=0.0) - exact_match_rate_from_abs(cmp_current_320_df['retl_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'term_cnt_exact_match_rate_pct_tol_0',
            'current_parse': exact_match_rate_from_abs(cmp_current_320_df['term_cnt_abs'], tol=0.0),
            'fixed_parse': exact_match_rate_from_abs(cmp_fixed_320_df['term_cnt_abs'], tol=0.0),
            'delta_fixed_minus_current': exact_match_rate_from_abs(cmp_fixed_320_df['term_cnt_abs'], tol=0.0) - exact_match_rate_from_abs(cmp_current_320_df['term_cnt_abs'], tol=0.0),
        },
        {
            'metric': 'retl_cnt_mae',
            'current_parse': float(_num_320(cmp_current_320_df, 'retl_cnt_abs').mean()) if len(cmp_current_320_df) else 0.0,
            'fixed_parse': float(_num_320(cmp_fixed_320_df, 'retl_cnt_abs').mean()) if len(cmp_fixed_320_df) else 0.0,
            'delta_fixed_minus_current': (float(_num_320(cmp_fixed_320_df, 'retl_cnt_abs').mean()) if len(cmp_fixed_320_df) else 0.0) - (float(_num_320(cmp_current_320_df, 'retl_cnt_abs').mean()) if len(cmp_current_320_df) else 0.0),
        },
        {
            'metric': 'term_cnt_mae',
            'current_parse': float(_num_320(cmp_current_320_df, 'term_cnt_abs').mean()) if len(cmp_current_320_df) else 0.0,
            'fixed_parse': float(_num_320(cmp_fixed_320_df, 'term_cnt_abs').mean()) if len(cmp_fixed_320_df) else 0.0,
            'delta_fixed_minus_current': (float(_num_320(cmp_fixed_320_df, 'term_cnt_abs').mean()) if len(cmp_fixed_320_df) else 0.0) - (float(_num_320(cmp_current_320_df, 'term_cnt_abs').mean()) if len(cmp_current_320_df) else 0.0),
        },
    ])

    # divergence by totals
    div_rows_320 = []
    for metric_name in ['retl_cnt', 'term_cnt']:
        cur_lake_total = _num_320(cmp_current_320_df, f'{metric_name}_lake').fillna(0).sum()
        cur_excel_total = _num_320(cmp_current_320_df, f'{metric_name}_excel').fillna(0).sum()
        fix_lake_total = _num_320(cmp_fixed_320_df, f'{metric_name}_lake').fillna(0).sum()
        fix_excel_total = _num_320(cmp_fixed_320_df, f'{metric_name}_excel').fillna(0).sum()

        cur_pct = _pct_div_320(cur_lake_total, cur_excel_total)
        fix_pct = _pct_div_320(fix_lake_total, fix_excel_total)

        div_rows_320.append({
            'metric': metric_name,
            'divergence_pct_current_parse': cur_pct,
            'divergence_pct_fixed_parse': fix_pct,
            'abs_divergence_improvement_pct_points': (
                abs(cur_pct) - abs(fix_pct)
                if cur_pct is not None and fix_pct is not None
                else None
            ),
        })

    divergence_current_vs_fixed_320_df = pd.DataFrame(div_rows_320)
else:
    divergence_current_vs_fixed_320_df = pd.DataFrame()

# Focus INN deep-dive in compare pipeline
focus_inn_320 = '0254010612'
focus_lineage_320_df = ex_diag_320_df[ex_diag_320_df['inn_key'] == focus_inn_320].copy()

focus_cols_320 = [
    '_row_id',
    inn_col_320,
    agr_col_320,
    contract_col_320 if contract_col_320 else None,
    'inn_key',
    'agr_id_key_current',
    'agr_id_key_fixed',
    'agr_parse_source_fixed',
    'parse_issue_current',
    'parse_issue_fixed',
]
if retl_col_320 is not None:
    focus_cols_320.append(retl_col_320)
if term_col_320 is not None:
    focus_cols_320.append(term_col_320)
focus_cols_320 = [c for c in focus_cols_320 if c is not None and c in focus_lineage_320_df.columns]

focus_lineage_320_df = focus_lineage_320_df[focus_cols_320].copy()

focus_excel_keys_current_320_df = excel_keys_current_320_df[excel_keys_current_320_df['inn_key'] == focus_inn_320].copy()
focus_excel_keys_fixed_320_df = excel_keys_fixed_320_df[excel_keys_fixed_320_df['inn_key'] == focus_inn_320].copy()
focus_lake_keys_320_df = lake_keys_320_df[lake_keys_320_df['inn_key'] == focus_inn_320].copy()

focus_key_compare_320_df = (
    focus_excel_keys_fixed_320_df
    .merge(focus_excel_keys_current_320_df.assign(in_current=1), on=['inn_key', 'agr_id_key'], how='left')
    .merge(focus_lake_keys_320_df.assign(in_lake=1), on=['inn_key', 'agr_id_key'], how='left')
)
focus_key_compare_320_df['in_current'] = focus_key_compare_320_df['in_current'].fillna(0).astype(int)
focus_key_compare_320_df['in_lake'] = focus_key_compare_320_df['in_lake'].fillna(0).astype(int)

print('Resolved columns (3.20):')
display(pd.DataFrame([resolved_320]))

print('Key-loss summary (global):')
display(key_loss_summary_320_df)

print('Parse issue summary (current vs fixed):')
display(parse_issue_summary_320_df)

print('Fixed parse source breakdown:')
display(parse_source_fixed_320_df)

print('Recovered keys by fix (top-200):')
display(recovered_keys_320_df.head(200))

if len(kpi_compare_current_vs_fixed_320_df):
    print('\nKPI impact simulation: current parse vs fixed parse')
    display(kpi_compare_current_vs_fixed_320_df)
    print('Divergence impact simulation:')
    display(divergence_current_vs_fixed_320_df)
else:
    print('KPI simulation skipped: retl/term columns not resolved in Excel')

print(f'\nFocus INN {focus_inn_320}: row lineage in Excel compare pipeline')
display(focus_lineage_320_df)

print(f'Focus INN {focus_inn_320}: key compare (fixed/current/lake)')
display(focus_key_compare_320_df)